In [1]:
from pathlib import Path

import re
import pandas as pd
import numpy as np
import openpyxl

In [2]:
FINAL_METRIC_FILE = Path(
    "data/idx_financial_current_metrics_final.csv"
)

print(
    "Input exists:",
    FINAL_METRIC_FILE.exists()
)

Input exists: True


In [3]:
final_metrics_df = pd.read_csv(
    FINAL_METRIC_FILE,
    low_memory=False
)

print(
    "Rows:",
    len(final_metrics_df)
)

print(
    "Tickers:",
    final_metrics_df["ticker"].nunique()
)

print(
    "Unique source files:",
    final_metrics_df["source_file"].nunique()
)

display(
    final_metrics_df.head()
)

Rows: 88400
Tickers: 948
Unique source files: 15602


,ticker,year,quarter,metric,selected_value,selection_status,current_candidate_count,unique_current_values,source_file_count,source_sheet_count,...,earliest_date,expected_period_date,suspicious_duplicate,clean_candidate_value,is_preferred_sheet,is_exact_period_match,exact_match_value,filename_trailing_ticker,ticker_matches_filename,identity_valid_candidate
0,AADI,2024,Q4,cash,1518688.0,SELECTED_MULTIPLE_SAME_VALUE,2.0,1.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AADI,2024,Q4,gross_profit,1465951.0,SELECTED_MULTIPLE_SAME_VALUE,2.0,1.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AADI,2024,Q4,operating_cash_flow,1198515.0,SELECTED_MULTIPLE_SAME_VALUE,2.0,1.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AADI,2024,Q4,revenue,5319582.0,SELECTED_MULTIPLE_SAME_VALUE,2.0,1.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AADI,2024,Q4,total_assets,5992658.0,SELECTED_MULTIPLE_SAME_VALUE,2.0,1.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
source_files_df = (
    final_metrics_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "source_path"
        ]
    ]
    .drop_duplicates(
        subset=[
            "source_file",
            "source_path"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Unique source files to consider:",
    len(source_files_df)
)

display(
    source_files_df.head()
)

Unique source files to consider: 15602


,ticker,year,quarter,source_file,source_path
0,AADI,2024,Q4,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,data\idx_financial_statements\Financial_Statem...
1,AADI,2025,Q1,AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
2,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
3,AALI,2020,Q2,AALI_2020_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...
4,AALI,2020,Q3,AALI_2020_Q3_FS.xlsx,data\idx_financial_statements\Financial_Statem...


In [5]:
source_files_df = (
    final_metrics_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "source_path"
        ]
    ]
    .drop_duplicates(
        subset=[
            "source_file",
            "source_path"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Unique source files to consider:",
    len(source_files_df)
)

display(
    source_files_df.head()
)

Unique source files to consider: 15602


,ticker,year,quarter,source_file,source_path
0,AADI,2024,Q4,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,data\idx_financial_statements\Financial_Statem...
1,AADI,2025,Q1,AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
2,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
3,AALI,2020,Q2,AALI_2020_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...
4,AALI,2020,Q3,AALI_2020_Q3_FS.xlsx,data\idx_financial_statements\Financial_Statem...


In [11]:
sample_source_files_df = (
    source_files_df
    .sort_values(
        [
            "year",
            "ticker",
            "quarter"
        ]
    )
    .groupby(
        "year",
        group_keys=False
    )
    .head(8)
    .reset_index(
        drop=True
    )
)

print(
    "Sample files:",
    len(sample_source_files_df)
)

display(
    sample_source_files_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file"
        ]
    ]
)

Sample files: 48


,ticker,year,quarter,source_file
0,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx
1,AALI,2020,Q2,AALI_2020_Q2_FS.xlsx
2,AALI,2020,Q3,AALI_2020_Q3_FS.xlsx
3,AALI,2020,Q4,AALI_2020_Q4_FS.xlsx
4,ABBA,2020,Q1,ABBA_2020_Q1_FS.xlsx
5,ABBA,2020,Q2,ABBA_2020_Q2_FS.xlsx
6,ABBA,2020,Q3,ABBA_2020_Q3_FS.xlsx
7,ABBA,2020,Q4,ABBA_2020_Q4_FS.xlsx
8,AALI,2021,Q1,AALI_2021_Q1_FS.xlsx
9,AALI,2021,Q2,AALI_2021_Q2_FS.xlsx


In [12]:
PROJECT_ROOT = Path.cwd()


def resolve_source_path(
    source_path
):
    if pd.isna(source_path):
        return None

    path = Path(
        str(source_path)
    )

    if path.exists():
        return path

    relative_path = (
        PROJECT_ROOT
        /
        path
    )

    if relative_path.exists():
        return relative_path

    return None

In [13]:
CURRENCY_UNIT_KEYWORDS = [
    "currency",
    "currencies",
    "mata uang",
    "rupiah",
    "dollar",
    "dollars",
    "idr",
    "usd",

    "unit",
    "units",
    "satuan",

    "scale",
    "scaling",

    "thousand",
    "thousands",
    "ribuan",

    "million",
    "millions",
    "jutaan",

    "billion",
    "billions",
    "miliar",
    "milyar",
]

In [14]:
def normalize_text(
    value
):
    if value is None:
        return ""

    return (
        str(value)
        .strip()
        .lower()
    )

In [15]:
def discover_currency_unit_metadata(
    file_row,
    max_rows_per_sheet=120,
    max_columns=20
):
    results = []

    resolved_path = resolve_source_path(
        file_row["source_path"]
    )

    if resolved_path is None:
        return [
            {
                "ticker": file_row["ticker"],
                "year": file_row["year"],
                "quarter": file_row["quarter"],
                "source_file": file_row["source_file"],
                "source_sheet": None,
                "row_number": None,
                "column_number": None,
                "cell_value": None,
                "matched_keyword": None,
                "status": "FILE_NOT_FOUND"
            }
        ]

    try:
        workbook = openpyxl.load_workbook(
            resolved_path,
            read_only=True,
            data_only=True
        )

    except Exception as error:
        return [
            {
                "ticker": file_row["ticker"],
                "year": file_row["year"],
                "quarter": file_row["quarter"],
                "source_file": file_row["source_file"],
                "source_sheet": None,
                "row_number": None,
                "column_number": None,
                "cell_value": None,
                "matched_keyword": None,
                "status": f"READ_ERROR: {type(error).__name__}"
            }
        ]

    try:
        for sheet_name in workbook.sheetnames:

            worksheet = workbook[
                sheet_name
            ]

            max_row = min(
                worksheet.max_row,
                max_rows_per_sheet
            )

            max_col = min(
                worksheet.max_column,
                max_columns
            )

            for row_number, row in enumerate(
                worksheet.iter_rows(
                    min_row=1,
                    max_row=max_row,
                    max_col=max_col,
                    values_only=True
                ),
                start=1
            ):

                for column_number, value in enumerate(
                    row,
                    start=1
                ):
                    if value is None:
                        continue

                    text = normalize_text(
                        value
                    )

                    if not text:
                        continue

                    matched_keywords = [
                        keyword
                        for keyword
                        in CURRENCY_UNIT_KEYWORDS
                        if keyword in text
                    ]

                    if not matched_keywords:
                        continue

                    results.append(
                        {
                            "ticker": file_row["ticker"],
                            "year": file_row["year"],
                            "quarter": file_row["quarter"],
                            "source_file": file_row["source_file"],
                            "source_sheet": sheet_name,
                            "row_number": row_number,
                            "column_number": column_number,
                            "cell_value": value,
                            "matched_keyword": "|".join(
                                matched_keywords
                            ),
                            "status": "FOUND"
                        }
                    )

    finally:
        workbook.close()

    return results

In [16]:
sample_discovery_results = []

for index, file_row in (
    sample_source_files_df
    .iterrows()
):
    file_results = (
        discover_currency_unit_metadata(
            file_row
        )
    )

    sample_discovery_results.extend(
        file_results
    )

    print(
        f"[{index + 1}/{len(sample_source_files_df)}]",
        file_row["source_file"],
        "->",
        len(file_results),
        "matches"
    )

[1/48] AALI_2020_Q1_FS.xlsx -> 8 matches
[2/48] AALI_2020_Q2_FS.xlsx -> 8 matches
[3/48] AALI_2020_Q3_FS.xlsx -> 8 matches
[4/48] AALI_2020_Q4_FS.xlsx -> 8 matches
[5/48] ABBA_2020_Q1_FS.xlsx -> 8 matches
[6/48] ABBA_2020_Q2_FS.xlsx -> 8 matches
[7/48] ABBA_2020_Q3_FS.xlsx -> 8 matches
[8/48] ABBA_2020_Q4_FS.xlsx -> 8 matches
[9/48] AALI_2021_Q1_FS.xlsx -> 8 matches
[10/48] AALI_2021_Q2_FS.xlsx -> 8 matches
[11/48] AALI_2021_Q3_FS.xlsx -> 8 matches
[12/48] AALI_2021_Q4_FS.xlsx -> 8 matches
[13/48] ABBA_2021_Q1_FS.xlsx -> 8 matches
[14/48] ABBA_2021_Q2_FS.xlsx -> 8 matches
[15/48] ABBA_2021_Q3_FS.xlsx -> 8 matches
[16/48] ABBA_2021_Q4_FS.xlsx -> 8 matches
[17/48] AALI_2022_Q1_FS.xlsx -> 8 matches
[18/48] AALI_2022_Q2_FS.xlsx -> 8 matches
[19/48] AALI_2022_Q3_FS.xlsx -> 8 matches
[20/48] AALI_2022_Q4_FS.xlsx -> 627 matches
[21/48] ABBA_2022_Q1_FS.xlsx -> 8 matches
[22/48] ABBA_2022_Q2_FS.xlsx -> 8 matches
[23/48] ABBA_2022_Q3_FS.xlsx -> 8 matches
[24/48] ABBA_2022_Q4_FS.xlsx -> 617 match

e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


[28/48] AALI_2023_Q4_FS.xlsx -> 513 matches
[29/48] ABBA_2023_Q1_FS.xlsx -> 617 matches
[30/48] ABBA_2023_Q2_FS.xlsx -> 617 matches
[31/48] ABBA_2023_Q4_FS.xlsx -> 513 matches
[32/48] ABDA_2023_Q1_FS.xlsx -> 38 matches
[33/48] AADI_2024_Q4_FinancialStatement-2024-Tahunan-AADI.xlsx -> 616 matches
[34/48] AALI_2024_Q2_FS.xlsx -> 619 matches
[35/48] AALI_2024_Q3_FS.xlsx -> 838 matches
[36/48] AALI_2024_Q4_FS.xlsx -> 522 matches
[37/48] ABBA_2024_Q1_FS.xlsx -> 513 matches
[38/48] ABBA_2024_Q2_FS.xlsx -> 513 matches
[39/48] ABBA_2024_Q3_FS.xlsx -> 513 matches
[40/48] ABBA_2024_Q4_FS.xlsx -> 513 matches
[41/48] AADI_2025_Q1_FS.xlsx -> 608 matches
[42/48] AALI_2025_Q1_FS.xlsx -> 533 matches
[43/48] ABBA_2025_Q1_FS.xlsx -> 513 matches
[44/48] ABDA_2025_Q1_FS.xlsx -> 28 matches
[45/48] ABMM_2025_Q1_FS.xlsx -> 606 matches
[46/48] ACES_2025_Q1_FS.xlsx -> 606 matches
[47/48] ACRO_2025_Q1_FS.xlsx -> 513 matches
[48/48] ACST_2025_Q1_FS.xlsx -> 482 matches


In [17]:
sample_currency_unit_df = (
    pd.DataFrame(
        sample_discovery_results
    )
)

print(
    "Discovery rows:",
    len(sample_currency_unit_df)
)

display(
    sample_currency_unit_df.head(200)
)

Discovery rows: 14133


,ticker,year,quarter,source_file,source_sheet,row_number,column_number,cell_value,matched_keyword,status
0,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,Context,85,1,CurrentYearDuration_NetAssetsAttributableToPar...,unit,FOUND
1,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,1000000,23,1,Mata uang pelaporan,mata uang,FOUND
2,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,1000000,23,2,Rupiah / IDR,rupiah|idr,FOUND
3,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,1000000,23,3,Description of presentation currency,currency,FOUND
4,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,1000000,24,1,Kurs konversi pada tanggal pelaporan jika mata...,mata uang|rupiah,FOUND
...,...,...,...,...,...,...,...,...,...,...
195,AALI,2022,Q4,AALI_2022_Q4_FS.xlsx,1621000a,18,8,USD,usd,FOUND
196,AALI,2022,Q4,AALI_2022_Q4_FS.xlsx,1621000a,19,2,Mata uang lainnya,mata uang,FOUND
197,AALI,2022,Q4,AALI_2022_Q4_FS.xlsx,1621000a,19,8,Other currency,currency,FOUND
198,AALI,2022,Q4,AALI_2022_Q4_FS.xlsx,1621000a,20,2,Mata uang,mata uang,FOUND


In [18]:
sheet_summary = (
    sample_currency_unit_df[
        sample_currency_unit_df[
            "status"
        ].eq(
            "FOUND"
        )
    ]
    .groupby(
        "source_sheet",
        dropna=False
    )
    .size()
    .reset_index(
        name="matches"
    )
    .sort_values(
        "matches",
        ascending=False
    )
)

display(
    sheet_summary.head(50)
)

,source_sheet,matches
17,1693000,2634
14,1691000a,2634
19,1696000,2414
16,1692000,2300
7,1621000a,624
18,1694000a,624
8,1621100,610
20,1697000,590
6,1620100,451
11,1640100,392


In [19]:
metadata_value_summary = (
    sample_currency_unit_df[
        sample_currency_unit_df[
            "status"
        ].eq(
            "FOUND"
        )
    ]
    .groupby(
        [
            "source_sheet",
            "cell_value",
            "matched_keyword"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="occurrences"
    )
    .sort_values(
        "occurrences",
        ascending=False
    )
)

display(
    metadata_value_summary.head(200)
)

,source_sheet,cell_value,matched_keyword,occurrences
156,1696000,IDR,idr,806
133,1692000,IDR,idr,766
161,1696000,USD,usd,724
115,1691000a,IDR,idr,702
140,1693000,IDR,idr,702
...,...,...,...,...
123,1691100,a.\tPerjanjian Fasilitas AS$250.000 dan Rp3.80...,currency|mata uang|rupiah,1
191,6610000,a. Imbalan kerja jangka pendek \nImbalan kerja...,mata uang|rupiah|unit,1
188,6610000,Pembukuan Grup diselenggarakan dalam mata uang...,mata uang|rupiah,1
186,6610000,Pembukuan Grup diselenggarakan dalam mata uang...,mata uang|rupiah,1


In [20]:
def get_metadata_context(
    source_path,
    sheet_name,
    row_number,
    column_number,
    row_radius=1,
    column_radius=3
):
    resolved_path = resolve_source_path(
        source_path
    )

    if resolved_path is None:
        return None

    workbook = openpyxl.load_workbook(
        resolved_path,
        read_only=True,
        data_only=True
    )

    try:
        if sheet_name not in workbook.sheetnames:
            return None

        worksheet = workbook[
            sheet_name
        ]

        values = []

        start_row = max(
            1,
            int(row_number) - row_radius
        )

        end_row = min(
            worksheet.max_row,
            int(row_number) + row_radius
        )

        start_col = max(
            1,
            int(column_number) - column_radius
        )

        end_col = min(
            worksheet.max_column,
            int(column_number) + column_radius
        )

        for row in worksheet.iter_rows(
            min_row=start_row,
            max_row=end_row,
            min_col=start_col,
            max_col=end_col,
            values_only=True
        ):
            values.append(
                list(row)
            )

        return values

    finally:
        workbook.close()

In [21]:
context_samples_df = (
    sample_currency_unit_df[
        sample_currency_unit_df[
            "status"
        ].eq(
            "FOUND"
        )
    ]
    .drop_duplicates(
        subset=[
            "source_file",
            "source_sheet",
            "row_number",
            "column_number"
        ]
    )
    .head(20)
    .copy()
)

In [22]:
for _, row in context_samples_df.iterrows():

    source_path = (
        sample_source_files_df.loc[
            sample_source_files_df[
                "source_file"
            ].eq(
                row["source_file"]
            ),
            "source_path"
        ]
        .iloc[0]
    )

    context = get_metadata_context(
        source_path=source_path,
        sheet_name=row["source_sheet"],
        row_number=row["row_number"],
        column_number=row["column_number"]
    )

    print(
        "\nFILE:",
        row["source_file"]
    )

    print(
        "SHEET:",
        row["source_sheet"]
    )

    print(
        "MATCH:",
        row["cell_value"]
    )

    print(
        "CONTEXT:"
    )

    for context_row in (
        context or []
    ):
        print(
            context_row
        )

    print(
        "-" * 80
    )


FILE: AALI_2020_Q1_FS.xlsx
SHEET: Context
MATCH: CurrentYearDuration_NetAssetsAttributableToParticipationUnitHoldersMember
CONTEXT:
[None, None]
['CurrentYearDuration_NetAssetsAttributableToParticipationUnitHoldersMember', None]
['entity', None]
--------------------------------------------------------------------------------

FILE: AALI_2020_Q1_FS.xlsx
SHEET: 1000000
MATCH: Mata uang pelaporan
CONTEXT:
['Tanggal akhir periode sebelumnya', '2019-03-31', 'Prior period end date', None]
['Mata uang pelaporan', 'Rupiah / IDR', 'Description of presentation currency', None]
['Kurs konversi pada tanggal pelaporan jika mata uang penyajian selain rupiah', None, 'Conversion rate at reporting date if presentation currency is other than rupiah', None]
--------------------------------------------------------------------------------

FILE: AALI_2020_Q1_FS.xlsx
SHEET: 1000000
MATCH: Rupiah / IDR
CONTEXT:
['Tanggal akhir periode sebelumnya', '2019-03-31', 'Prior period end date', None]
['Mata uang pel

In [23]:
# CELL 17 - INSPECT SHEET 1000000 ONLY

def inspect_metadata_sheet(
    file_row,
    sheet_name="1000000",
    max_rows=80,
    max_columns=8
):
    resolved_path = resolve_source_path(
        file_row["source_path"]
    )

    if resolved_path is None:
        return []

    try:
        workbook = openpyxl.load_workbook(
            resolved_path,
            read_only=True,
            data_only=True
        )

        if sheet_name not in workbook.sheetnames:
            workbook.close()
            return []

        worksheet = workbook[sheet_name]

        rows = []

        for row_number, row_values in enumerate(
            worksheet.iter_rows(
                min_row=1,
                max_row=min(
                    worksheet.max_row,
                    max_rows
                ),
                max_col=min(
                    worksheet.max_column,
                    max_columns
                ),
                values_only=True
            ),
            start=1
        ):
            rows.append(
                {
                    "row_number": row_number,
                    "values": list(row_values)
                }
            )

        workbook.close()

        return rows

    except Exception:
        return []

In [24]:
# CELL 18 - TARGETED METADATA DISCOVERY

TARGET_METADATA_TERMS = [
    "mata uang pelaporan",
    "presentation currency",
    "description of presentation currency",
    "satuan",
    "unit",
    "ribuan",
    "thousand",
    "jutaan",
    "million",
]


targeted_metadata_results = []

for _, file_row in sample_source_files_df.iterrows():

    metadata_rows = inspect_metadata_sheet(
        file_row
    )

    for row in metadata_rows:

        row_values = row["values"]

        row_text = " | ".join(
            str(value)
            for value in row_values
            if value is not None
        ).lower()

        matched_terms = [
            term
            for term in TARGET_METADATA_TERMS
            if term in row_text
        ]

        if matched_terms:
            targeted_metadata_results.append(
                {
                    "ticker": file_row["ticker"],
                    "year": file_row["year"],
                    "quarter": file_row["quarter"],
                    "source_file": file_row["source_file"],
                    "row_number": row["row_number"],
                    "matched_terms": "|".join(
                        matched_terms
                    ),
                    "row_values": row_values
                }
            )


targeted_metadata_df = pd.DataFrame(
    targeted_metadata_results
)

print(
    "Targeted metadata rows:",
    len(targeted_metadata_df)
)

display(
    targeted_metadata_df.head(200)
)

e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Targeted metadata rows: 144


,ticker,year,quarter,source_file,row_number,matched_terms,row_values
0,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,23,mata uang pelaporan|presentation currency|desc...,"[Mata uang pelaporan, Rupiah / IDR, Descriptio..."
1,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,24,presentation currency,[Kurs konversi pada tanggal pelaporan jika mat...
2,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,25,jutaan|million,[Pembulatan yang digunakan dalam penyajian jum...
3,AALI,2020,Q2,AALI_2020_Q2_FS.xlsx,23,mata uang pelaporan|presentation currency|desc...,"[Mata uang pelaporan, Rupiah / IDR, Descriptio..."
4,AALI,2020,Q2,AALI_2020_Q2_FS.xlsx,24,presentation currency,[Kurs konversi pada tanggal pelaporan jika mat...
...,...,...,...,...,...,...,...
139,ACRO,2025,Q1,ACRO_2025_Q1_FS.xlsx,30,presentation currency,[Kurs konversi pada tanggal pelaporan jika mat...
140,ACRO,2025,Q1,ACRO_2025_Q1_FS.xlsx,31,satuan,[Pembulatan yang digunakan dalam penyajian jum...
141,ACST,2025,Q1,ACST_2025_Q1_FS.xlsx,29,mata uang pelaporan|presentation currency|desc...,"[Mata uang pelaporan, Rupiah / IDR, Descriptio..."
142,ACST,2025,Q1,ACST_2025_Q1_FS.xlsx,30,presentation currency,[Kurs konversi pada tanggal pelaporan jika mat...


In [25]:
# CELL 19 - METADATA LABEL SUMMARY

if not targeted_metadata_df.empty:

    targeted_metadata_df[
        "row_text"
    ] = (
        targeted_metadata_df[
            "row_values"
        ]
        .apply(
            lambda values:
                " | ".join(
                    str(value)
                    for value in values
                    if value is not None
                )
        )
    )

    metadata_row_summary = (
        targeted_metadata_df[
            "row_text"
        ]
        .value_counts()
        .reset_index()
    )

    metadata_row_summary.columns = [
        "row_text",
        "occurrences"
    ]

    display(
        metadata_row_summary.head(100)
    )

,row_text,occurrences
0,Mata uang pelaporan | Rupiah / IDR | Descripti...,45
1,Kurs konversi pada tanggal pelaporan jika mata...,41
2,Pembulatan yang digunakan dalam penyajian juml...,23
3,Pembulatan yang digunakan dalam penyajian juml...,21
4,Kurs konversi pada tanggal pelaporan jika mata...,5
5,Pembulatan yang digunakan dalam penyajian juml...,4
6,Mata uang pelaporan | Dollar Amerika / USD | D...,3
7,Kurs konversi pada tanggal pelaporan jika mata...,1
8,Kurs konversi pada tanggal pelaporan jika mata...,1


In [26]:
# CELL 20 - INSPECT SCALE VALUES

scale_rows_df = (
    targeted_metadata_df[
        targeted_metadata_df[
            "row_text"
        ]
        .str.contains(
            "pembulatan",
            case=False,
            na=False
        )
    ]
    .copy()
)

print(
    "Scale metadata rows:",
    len(scale_rows_df)
)

display(
    scale_rows_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "row_number",
            "row_values",
            "row_text"
        ]
    ].head(100)
)

Scale metadata rows: 48


,ticker,year,quarter,source_file,row_number,row_values,row_text
2,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,25,[Pembulatan yang digunakan dalam penyajian jum...,Pembulatan yang digunakan dalam penyajian juml...
5,AALI,2020,Q2,AALI_2020_Q2_FS.xlsx,25,[Pembulatan yang digunakan dalam penyajian jum...,Pembulatan yang digunakan dalam penyajian juml...
8,AALI,2020,Q3,AALI_2020_Q3_FS.xlsx,25,[Pembulatan yang digunakan dalam penyajian jum...,Pembulatan yang digunakan dalam penyajian juml...
11,AALI,2020,Q4,AALI_2020_Q4_FS.xlsx,25,[Pembulatan yang digunakan dalam penyajian jum...,Pembulatan yang digunakan dalam penyajian juml...
14,ABBA,2020,Q1,ABBA_2020_Q1_FS.xlsx,25,[Pembulatan yang digunakan dalam penyajian jum...,Pembulatan yang digunakan dalam penyajian juml...
17,ABBA,2020,Q2,ABBA_2020_Q2_FS.xlsx,25,[Pembulatan yang digunakan dalam penyajian jum...,Pembulatan yang digunakan dalam penyajian juml...
20,ABBA,2020,Q3,ABBA_2020_Q3_FS.xlsx,25,[Pembulatan yang digunakan dalam penyajian jum...,Pembulatan yang digunakan dalam penyajian juml...
23,ABBA,2020,Q4,ABBA_2020_Q4_FS.xlsx,25,[Pembulatan yang digunakan dalam penyajian jum...,Pembulatan yang digunakan dalam penyajian juml...
26,AALI,2021,Q1,AALI_2021_Q1_FS.xlsx,25,[Pembulatan yang digunakan dalam penyajian jum...,Pembulatan yang digunakan dalam penyajian juml...
29,AALI,2021,Q2,AALI_2021_Q2_FS.xlsx,25,[Pembulatan yang digunakan dalam penyajian jum...,Pembulatan yang digunakan dalam penyajian juml...


In [27]:
# CELL 21 - SAMPLE METADATA PARSER

def parse_currency_from_row(values):
    text = " | ".join(
        str(v)
        for v in values
        if v is not None
    ).lower()

    if "rupiah / idr" in text or "idr" in text:
        return "IDR"

    if (
        "dollar amerika / usd" in text
        or "usd" in text
    ):
        return "USD"

    return None


def parse_scale_from_row(values):
    text = " | ".join(
        str(v)
        for v in values
        if v is not None
    ).lower()

    if "jutaan" in text or "million" in text:
        return {
            "unit_scale": "MILLION",
            "multiplier": 1_000_000
        }

    if "ribuan" in text or "thousand" in text:
        return {
            "unit_scale": "THOUSAND",
            "multiplier": 1_000
        }

    if (
        "satuan" in text
        or "unit" in text
    ):
        return {
            "unit_scale": "UNIT",
            "multiplier": 1
        }

    return None

In [28]:
# CELL 22 - BUILD SAMPLE CURRENCY / SCALE TABLE

sample_metadata_records = []

for _, file_row in sample_source_files_df.iterrows():

    metadata_rows = inspect_metadata_sheet(
        file_row
    )

    currency = None
    unit_scale = None
    multiplier = None

    currency_row = None
    scale_row = None

    for row in metadata_rows:

        values = row["values"]

        row_text = " | ".join(
            str(v)
            for v in values
            if v is not None
        ).lower()

        if (
            currency is None
            and "mata uang pelaporan" in row_text
        ):
            currency = parse_currency_from_row(
                values
            )

            currency_row = row[
                "row_number"
            ]

        if (
            unit_scale is None
            and "pembulatan" in row_text
        ):
            scale_info = parse_scale_from_row(
                values
            )

            if scale_info is not None:
                unit_scale = scale_info[
                    "unit_scale"
                ]

                multiplier = scale_info[
                    "multiplier"
                ]

                scale_row = row[
                    "row_number"
                ]

    sample_metadata_records.append(
        {
            "ticker": file_row["ticker"],
            "year": file_row["year"],
            "quarter": file_row["quarter"],
            "source_file": file_row["source_file"],
            "currency": currency,
            "unit_scale": unit_scale,
            "multiplier": multiplier,
            "currency_row": currency_row,
            "scale_row": scale_row
        }
    )


sample_metadata_parsed_df = pd.DataFrame(
    sample_metadata_records
)

display(
    sample_metadata_parsed_df
)

e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,ticker,year,quarter,source_file,currency,unit_scale,multiplier,currency_row,scale_row
0,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,IDR,MILLION,1000000,23,25
1,AALI,2020,Q2,AALI_2020_Q2_FS.xlsx,IDR,MILLION,1000000,23,25
2,AALI,2020,Q3,AALI_2020_Q3_FS.xlsx,IDR,MILLION,1000000,23,25
3,AALI,2020,Q4,AALI_2020_Q4_FS.xlsx,IDR,MILLION,1000000,23,25
4,ABBA,2020,Q1,ABBA_2020_Q1_FS.xlsx,IDR,UNIT,1,23,25
5,ABBA,2020,Q2,ABBA_2020_Q2_FS.xlsx,IDR,UNIT,1,23,25
6,ABBA,2020,Q3,ABBA_2020_Q3_FS.xlsx,IDR,UNIT,1,23,25
7,ABBA,2020,Q4,ABBA_2020_Q4_FS.xlsx,IDR,UNIT,1,23,25
8,AALI,2021,Q1,AALI_2021_Q1_FS.xlsx,IDR,MILLION,1000000,23,25
9,AALI,2021,Q2,AALI_2021_Q2_FS.xlsx,IDR,MILLION,1000000,23,25


In [29]:
# CELL 23 - SAMPLE COVERAGE SUMMARY

print(
    "Sample files:",
    len(sample_metadata_parsed_df)
)

print(
    "Currency found:",
    sample_metadata_parsed_df[
        "currency"
    ].notna().sum()
)

print(
    "Scale found:",
    sample_metadata_parsed_df[
        "unit_scale"
    ].notna().sum()
)

print()

display(
    sample_metadata_parsed_df[
        "currency"
    ]
    .value_counts(
        dropna=False
    )
)

display(
    sample_metadata_parsed_df[
        "unit_scale"
    ]
    .value_counts(
        dropna=False
    )
)

Sample files: 48
Currency found: 48
Scale found: 48



currency
IDR    45
USD     3
Name: count, dtype: int64

unit_scale
UNIT        23
MILLION     21
THOUSAND     4
Name: count, dtype: int64

In [30]:
# CELL 24 - PARSE CURRENCY / SCALE FOR ONE SOURCE FILE

def parse_file_currency_scale(
    file_row,
    sheet_name="1000000",
    max_rows=80,
    max_columns=8
):
    resolved_path = resolve_source_path(
        file_row["source_path"]
    )

    base_result = {
        "ticker": file_row["ticker"],
        "year": file_row["year"],
        "quarter": file_row["quarter"],
        "source_file": file_row["source_file"],
        "source_path": file_row["source_path"],

        "currency": None,
        "unit_scale": None,
        "multiplier": None,

        "currency_row": None,
        "scale_row": None,

        "mapping_status": None
    }

    if resolved_path is None:
        base_result["mapping_status"] = "FILE_NOT_FOUND"
        return base_result

    try:
        workbook = openpyxl.load_workbook(
            resolved_path,
            read_only=True,
            data_only=True
        )

    except Exception as error:
        base_result["mapping_status"] = (
            f"READ_ERROR:{type(error).__name__}"
        )
        return base_result

    try:
        if sheet_name not in workbook.sheetnames:
            base_result["mapping_status"] = (
                "METADATA_SHEET_NOT_FOUND"
            )
            return base_result

        worksheet = workbook[
            sheet_name
        ]

        currency = None
        unit_scale = None
        multiplier = None

        currency_row = None
        scale_row = None

        for row_number, row_values in enumerate(
            worksheet.iter_rows(
                min_row=1,
                max_row=min(
                    worksheet.max_row,
                    max_rows
                ),
                max_col=min(
                    worksheet.max_column,
                    max_columns
                ),
                values_only=True
            ),
            start=1
        ):
            values = list(
                row_values
            )

            row_text = " | ".join(
                str(value)
                for value in values
                if value is not None
            ).lower()

            if (
                currency is None
                and "mata uang pelaporan" in row_text
            ):
                currency = parse_currency_from_row(
                    values
                )

                currency_row = row_number

            if (
                unit_scale is None
                and "pembulatan" in row_text
            ):
                scale_info = parse_scale_from_row(
                    values
                )

                if scale_info is not None:
                    unit_scale = scale_info[
                        "unit_scale"
                    ]

                    multiplier = scale_info[
                        "multiplier"
                    ]

                    scale_row = row_number

            if (
                currency is not None
                and unit_scale is not None
            ):
                break

        base_result["currency"] = currency
        base_result["unit_scale"] = unit_scale
        base_result["multiplier"] = multiplier
        base_result["currency_row"] = currency_row
        base_result["scale_row"] = scale_row

        if (
            currency is not None
            and unit_scale is not None
        ):
            base_result["mapping_status"] = "FOUND"

        elif currency is None and unit_scale is None:
            base_result["mapping_status"] = (
                "CURRENCY_AND_SCALE_NOT_FOUND"
            )

        elif currency is None:
            base_result["mapping_status"] = (
                "CURRENCY_NOT_FOUND"
            )

        else:
            base_result["mapping_status"] = (
                "SCALE_NOT_FOUND"
            )

        return base_result

    finally:
        workbook.close()

In [31]:
# CELL 25 - QUICK TEST

quick_test_files_df = (
    source_files_df
    .head(20)
    .copy()
)

quick_test_results = []

for _, file_row in (
    quick_test_files_df
    .iterrows()
):
    result = parse_file_currency_scale(
        file_row
    )

    quick_test_results.append(
        result
    )

quick_test_metadata_df = (
    pd.DataFrame(
        quick_test_results
    )
)

display(
    quick_test_metadata_df
)

,ticker,year,quarter,source_file,source_path,currency,unit_scale,multiplier,currency_row,scale_row,mapping_status
0,AADI,2024,Q4,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,USD,THOUSAND,1000,29,31,FOUND
1,AADI,2025,Q1,AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,USD,THOUSAND,1000,29,31,FOUND
2,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000,23,25,FOUND
3,AALI,2020,Q2,AALI_2020_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000,23,25,FOUND
4,AALI,2020,Q3,AALI_2020_Q3_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000,23,25,FOUND
5,AALI,2020,Q4,AALI_2020_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000,23,25,FOUND
6,AALI,2021,Q1,AALI_2021_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000,23,25,FOUND
7,AALI,2021,Q2,AALI_2021_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000,23,25,FOUND
8,AALI,2021,Q3,AALI_2021_Q3_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000,23,25,FOUND
9,AALI,2021,Q4,AALI_2021_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000,23,25,FOUND


In [32]:
# CELL 26 - QUICK TEST SUMMARY

display(
    quick_test_metadata_df[
        "mapping_status"
    ]
    .value_counts(
        dropna=False
    )
)

display(
    quick_test_metadata_df[
        "currency"
    ]
    .value_counts(
        dropna=False
    )
)

display(
    quick_test_metadata_df[
        "unit_scale"
    ]
    .value_counts(
        dropna=False
    )
)

mapping_status
FOUND    20
Name: count, dtype: int64

currency
IDR    18
USD     2
Name: count, dtype: int64

unit_scale
MILLION     18
THOUSAND     2
Name: count, dtype: int64

In [34]:
# CELL 27 - FULL CURRENCY / SCALE MAPPING

from tqdm.auto import tqdm

CURRENCY_SCALE_OUTPUT_FILE = Path(
    "data/idx_financial_currency_scale_mapping.csv"
)

CURRENCY_SCALE_CHECKPOINT_FILE = Path(
    "data/idx_financial_currency_scale_checkpoint.csv"
)

CHECKPOINT_EVERY = 250


full_currency_scale_results = []


for file_index, (_, file_row) in enumerate(
    tqdm(
        source_files_df.iterrows(),
        total=len(source_files_df),
        desc="Mapping currency / scale"
    ),
    start=1
):
    result = parse_file_currency_scale(
        file_row
    )

    full_currency_scale_results.append(
        result
    )

    if (
        file_index % CHECKPOINT_EVERY == 0
    ):
        checkpoint_df = pd.DataFrame(
            full_currency_scale_results
        )

        checkpoint_df.to_csv(
            CURRENCY_SCALE_CHECKPOINT_FILE,
            index=False
        )


currency_scale_mapping_df = (
    pd.DataFrame(
        full_currency_scale_results
    )
)

print(
    "Mapped source files:",
    len(currency_scale_mapping_df)
)

display(
    currency_scale_mapping_df.head()
)

Mapping currency / scale:   7%|▋         | 1136/15602 [11:19<4:40:39,  1.16s/it]e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Mapping currency / scale:  91%|█████████▏| 14259/15602 [2:20:38<21:27,  1.04it/s]  e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Mapping currency / scale: 100%|██████████| 15602/15602 [2:34:38<00:00,  1.68it/s]

Mapped source files: 15602


,ticker,year,quarter,source_file,source_path,currency,unit_scale,multiplier,currency_row,scale_row,mapping_status
0,AADI,2024,Q4,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,USD,THOUSAND,1000.0,29,31.0,FOUND
1,AADI,2025,Q1,AADI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,USD,THOUSAND,1000.0,29,31.0,FOUND
2,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000.0,23,25.0,FOUND
3,AALI,2020,Q2,AALI_2020_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000.0,23,25.0,FOUND
4,AALI,2020,Q3,AALI_2020_Q3_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,MILLION,1000000.0,23,25.0,FOUND


In [35]:
# CELL 28 - STATUS SUMMARY

currency_scale_status_summary = (
    currency_scale_mapping_df[
        "mapping_status"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

currency_scale_status_summary.columns = [
    "mapping_status",
    "files"
]

display(
    currency_scale_status_summary
)

,mapping_status,files
0,FOUND,15563
1,SCALE_NOT_FOUND,37
2,CURRENCY_AND_SCALE_NOT_FOUND,2


In [36]:
# CELL 29 - CURRENCY SUMMARY

currency_summary = (
    currency_scale_mapping_df[
        "currency"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

currency_summary.columns = [
    "currency",
    "files"
]

display(
    currency_summary
)

,currency,files
0,IDR,13708
1,USD,1892
2,NaN,2


In [37]:
# CELL 30 - SCALE SUMMARY

scale_summary = (
    currency_scale_mapping_df[
        "unit_scale"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

scale_summary.columns = [
    "unit_scale",
    "files"
]

display(
    scale_summary
)

,unit_scale,files
0,UNIT,10127
1,MILLION,3453
2,THOUSAND,1983
3,NaN,39


In [38]:
# CELL 31 - SAVE FULL MAPPING + INSPECT UNRESOLVED FILES

# Save the complete 15,602-file mapping immediately
currency_scale_mapping_df.to_csv(
    CURRENCY_SCALE_OUTPUT_FILE,
    index=False
)

print(
    "Saved full currency / scale mapping to:",
    CURRENCY_SCALE_OUTPUT_FILE
)

print(
    "Total saved rows:",
    len(currency_scale_mapping_df)
)


# Inspect files where currency or scale mapping is incomplete
currency_scale_failures_df = (
    currency_scale_mapping_df[
        currency_scale_mapping_df[
            "mapping_status"
        ] != "FOUND"
    ]
    .copy()
)

print()
print(
    "Unresolved source files:",
    len(currency_scale_failures_df)
)

display(
    currency_scale_failures_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "currency",
            "unit_scale",
            "multiplier",
            "currency_row",
            "scale_row",
            "mapping_status"
        ]
    ]
)

Saved full currency / scale mapping to: data\idx_financial_currency_scale_mapping.csv
Total saved rows: 15602

Unresolved source files: 39


,ticker,year,quarter,source_file,currency,unit_scale,multiplier,currency_row,scale_row,mapping_status
1121,ASII,2020,Q1,ASII_2020_Q1_FS.xlsx,IDR,NaN,NaN,23,NaN,SCALE_NOT_FOUND
1122,ASII,2020,Q2,ASII_2020_Q2_FS.xlsx,IDR,NaN,NaN,23,NaN,SCALE_NOT_FOUND
1123,ASII,2020,Q3,ASII_2020_Q3_FS.xlsx,IDR,NaN,NaN,23,NaN,SCALE_NOT_FOUND
1124,ASII,2020,Q4,ASII_2020_Q4_FS.xlsx,IDR,NaN,NaN,23,NaN,SCALE_NOT_FOUND
1125,ASII,2021,Q1,ASII_2021_Q1_FS.xlsx,IDR,NaN,NaN,23,NaN,SCALE_NOT_FOUND
1126,ASII,2021,Q2,ASII_2021_Q2_FS.xlsx,IDR,NaN,NaN,23,NaN,SCALE_NOT_FOUND
1127,ASII,2021,Q3,ASII_2021_Q3_FS.xlsx,IDR,NaN,NaN,23,NaN,SCALE_NOT_FOUND
1128,ASII,2021,Q4,ASII_2021_Q4_FS.xlsx,IDR,NaN,NaN,23,NaN,SCALE_NOT_FOUND
1129,ASII,2022,Q1,ASII_2022_Q1_FS.xlsx,IDR,NaN,NaN,23,NaN,SCALE_NOT_FOUND
1130,ASII,2022,Q2,ASII_2022_Q2_FS.xlsx,IDR,NaN,NaN,23,NaN,SCALE_NOT_FOUND


In [39]:
# CELL 32 - INSPECT REPRESENTATIVE UNRESOLVED METADATA

representative_cases = [
    ("ASII", 2020, "Q1"),
    ("TLKM", 2020, "Q1"),
    ("CTRA", 2023, "Q2"),
    ("TEBE", 2023, "Q3"),
]


for ticker, year, quarter in representative_cases:

    print("\n" + "=" * 100)
    print(
        f"INSPECTING: {ticker} {year} {quarter}"
    )
    print("=" * 100)

    matched_file = source_files_df[
        (source_files_df["ticker"] == ticker)
        & (source_files_df["year"] == year)
        & (source_files_df["quarter"] == quarter)
    ]

    if matched_file.empty:
        print("SOURCE FILE NOT FOUND")
        continue

    file_row = matched_file.iloc[0]

    print(
        "Source file:",
        file_row["source_file"]
    )

    metadata_preview = inspect_metadata_sheet(
        file_row,
        sheet_name="1000000",
        max_rows=40,
        max_columns=8
    )

    display(
        metadata_preview
    )


INSPECTING: ASII 2020 Q1
Source file: ASII_2020_Q1_FS.xlsx


[{'row_number': 1,
  'values': ['[1000000] General information', None, None, None]},
 {'row_number': 2, 'values': [None, None, None, None]},
 {'row_number': 3,
  'values': ['Informasi umum', None, 'General information', None]},
 {'row_number': 4, 'values': [None, '31 March 2020', None, None]},
 {'row_number': 5,
  'values': ['Nama entitas', 'Astra International Tbk', 'Entity name', None]},
 {'row_number': 6,
  'values': ['Penjelasan perubahan nama dari akhir periode laporan sebelumnya',
   None,
   'Explanation of change in name from the end of the preceding reporting period',
   None]},
 {'row_number': 7, 'values': ['Kode entitas', 'ASII', 'Entity code', None]},
 {'row_number': 8,
  'values': ['Nomor identifikasi entitas',
   'AA075',
   'Entity identification number',
   None]},
 {'row_number': 9,
  'values': ['Industri utama entitas',
   'Umum / General',
   'Entity main industry',
   None]},
 {'row_number': 10,
  'values': ['Sektor', '4. Miscellaneous Industry', 'Sector', None]},
 


INSPECTING: TLKM 2020 Q1
Source file: TLKM_2020_Q1_FS.xlsx


[{'row_number': 1,
  'values': ['[1000000] General information', None, None, None]},
 {'row_number': 2, 'values': [None, None, None, None]},
 {'row_number': 3,
  'values': ['Informasi umum', None, 'General information', None]},
 {'row_number': 4, 'values': [None, '31 March 2020', None, None]},
 {'row_number': 5,
  'values': ['Nama entitas',
   'PT Telekomunikasi Indonesia (Persero) Tbk.',
   'Entity name',
   None]},
 {'row_number': 6,
  'values': ['Penjelasan perubahan nama dari akhir periode laporan sebelumnya',
   '-',
   'Explanation of change in name from the end of the preceding reporting period',
   None]},
 {'row_number': 7, 'values': ['Kode entitas', 'TLKM', 'Entity code', None]},
 {'row_number': 8,
  'values': ['Nomor identifikasi entitas',
   'AA252',
   'Entity identification number',
   None]},
 {'row_number': 9,
  'values': ['Industri utama entitas',
   'Infrastruktur / Infrastructure',
   'Entity main industry',
   None]},
 {'row_number': 10,
  'values': ['Sektor',
   '7


INSPECTING: CTRA 2023 Q2
Source file: CTRA_2023_Q2_FS.xlsx


[{'row_number': 1, 'values': ['[1000000] General information', None, None]},
 {'row_number': 2, 'values': [None, None, None]},
 {'row_number': 3, 'values': ['Informasi umum', None, 'General information']},
 {'row_number': 4, 'values': [None, 'CurrentYearInstant', None]},
 {'row_number': 5, 'values': ['Nama entitas', None, 'Entity name']},
 {'row_number': 6,
  'values': ['Penjelasan perubahan nama dari akhir periode laporan sebelumnya',
   '',
   'Explanation of change in name from the end of the preceding reporting period']},
 {'row_number': 7, 'values': ['Kode entitas', None, 'Entity code']},
 {'row_number': 8,
  'values': ['Nomor identifikasi entitas',
   None,
   'Entity identification number']},
 {'row_number': 9,
  'values': ['Industri utama entitas', None, 'Entity main industry']},
 {'row_number': 10,
  'values': ['Standar akutansi yang dipilih',
   'PSAK',
   'Selected accounting standards']},
 {'row_number': 11, 'values': ['Sektor', None, 'Sector']},
 {'row_number': 12, 'values


INSPECTING: TEBE 2023 Q3
Source file: TEBE_2023_Q3_FS.xlsx


[{'row_number': 1, 'values': ['[1000000] General information', None, None]},
 {'row_number': 2, 'values': [None, None, None]},
 {'row_number': 3, 'values': ['Informasi umum', None, 'General information']},
 {'row_number': 4, 'values': [None, 'CurrentYearInstant', None]},
 {'row_number': 5, 'values': ['Nama entitas', None, 'Entity name']},
 {'row_number': 6,
  'values': ['Penjelasan perubahan nama dari akhir periode laporan sebelumnya',
   None,
   'Explanation of change in name from the end of the preceding reporting period']},
 {'row_number': 7, 'values': ['Kode entitas', None, 'Entity code']},
 {'row_number': 8,
  'values': ['Nomor identifikasi entitas',
   None,
   'Entity identification number']},
 {'row_number': 9,
  'values': ['Industri utama entitas', None, 'Entity main industry']},
 {'row_number': 10,
  'values': ['Standar akutansi yang dipilih',
   None,
   'Selected accounting standards']},
 {'row_number': 11, 'values': ['Sektor', None, 'Sector']},
 {'row_number': 12, 'values

In [40]:
def parse_scale_from_row(values):
    text = " | ".join(
        str(v)
        for v in values
        if v is not None
    ).lower()

    if (
        "miliaran" in text
        or "billion" in text
    ):
        return {
            "unit_scale": "BILLION",
            "multiplier": 1_000_000_000
        }

    if (
        "jutaan" in text
        or "million" in text
    ):
        return {
            "unit_scale": "MILLION",
            "multiplier": 1_000_000
        }

    if (
        "ribuan" in text
        or "thousand" in text
    ):
        return {
            "unit_scale": "THOUSAND",
            "multiplier": 1_000
        }

    if (
        "satuan" in text
        or "unit" in text
    ):
        return {
            "unit_scale": "UNIT",
            "multiplier": 1
        }

    return None

In [41]:
# CELL 33 - REPROCESS ONLY UNRESOLVED FILES

resolved_retry_results = []


for _, failure_row in currency_scale_failures_df.iterrows():

    matched_file = source_files_df[
        source_files_df["source_file"]
        == failure_row["source_file"]
    ]

    if matched_file.empty:
        print(
            "SOURCE FILE NOT FOUND:",
            failure_row["source_file"]
        )
        continue

    file_row = matched_file.iloc[0]

    result = parse_file_currency_scale(
        file_row
    )

    resolved_retry_results.append(
        result
    )


currency_scale_retry_df = pd.DataFrame(
    resolved_retry_results
)


print(
    "Retried unresolved files:",
    len(currency_scale_retry_df)
)

print()

print(
    currency_scale_retry_df[
        "mapping_status"
    ].value_counts(
        dropna=False
    )
)

display(
    currency_scale_retry_df
)

Retried unresolved files: 39

mapping_status
FOUND                           37
CURRENCY_AND_SCALE_NOT_FOUND     2
Name: count, dtype: int64


,ticker,year,quarter,source_file,source_path,currency,unit_scale,multiplier,currency_row,scale_row,mapping_status
0,ASII,2020,Q1,ASII_2020_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,BILLION,1.000000e+09,23,25.0,FOUND
1,ASII,2020,Q2,ASII_2020_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,BILLION,1.000000e+09,23,25.0,FOUND
2,ASII,2020,Q3,ASII_2020_Q3_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,BILLION,1.000000e+09,23,25.0,FOUND
3,ASII,2020,Q4,ASII_2020_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,BILLION,1.000000e+09,23,25.0,FOUND
4,ASII,2021,Q1,ASII_2021_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,BILLION,1.000000e+09,23,25.0,FOUND
5,ASII,2021,Q2,ASII_2021_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,BILLION,1.000000e+09,23,25.0,FOUND
6,ASII,2021,Q3,ASII_2021_Q3_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,BILLION,1.000000e+09,23,25.0,FOUND
7,ASII,2021,Q4,ASII_2021_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,BILLION,1.000000e+09,23,25.0,FOUND
8,ASII,2022,Q1,ASII_2022_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,BILLION,1.000000e+09,23,25.0,FOUND
9,ASII,2022,Q2,ASII_2022_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,IDR,BILLION,1.000000e+09,23,25.0,FOUND


In [42]:
# CELL 34 - DEEP INSPECT THE 2 REMAINING UNRESOLVED FILES

remaining_problem_files = [
    ("CTRA", 2023, "Q2"),
    ("TEBE", 2023, "Q3"),
]


for ticker, year, quarter in remaining_problem_files:

    print("\n" + "=" * 120)
    print(f"{ticker} {year} {quarter}")
    print("=" * 120)

    matched_file = source_files_df[
        (source_files_df["ticker"] == ticker)
        & (source_files_df["year"] == year)
        & (source_files_df["quarter"] == quarter)
    ]

    if matched_file.empty:
        print("SOURCE FILE NOT FOUND")
        continue

    file_row = matched_file.iloc[0]

    print("Source file:", file_row["source_file"])

    metadata_preview = inspect_metadata_sheet(
        file_row,
        sheet_name="1000000",
        max_rows=80,
        max_columns=12
    )

    display(metadata_preview)


CTRA 2023 Q2
Source file: CTRA_2023_Q2_FS.xlsx


[{'row_number': 1, 'values': ['[1000000] General information', None, None]},
 {'row_number': 2, 'values': [None, None, None]},
 {'row_number': 3, 'values': ['Informasi umum', None, 'General information']},
 {'row_number': 4, 'values': [None, 'CurrentYearInstant', None]},
 {'row_number': 5, 'values': ['Nama entitas', None, 'Entity name']},
 {'row_number': 6,
  'values': ['Penjelasan perubahan nama dari akhir periode laporan sebelumnya',
   '',
   'Explanation of change in name from the end of the preceding reporting period']},
 {'row_number': 7, 'values': ['Kode entitas', None, 'Entity code']},
 {'row_number': 8,
  'values': ['Nomor identifikasi entitas',
   None,
   'Entity identification number']},
 {'row_number': 9,
  'values': ['Industri utama entitas', None, 'Entity main industry']},
 {'row_number': 10,
  'values': ['Standar akutansi yang dipilih',
   'PSAK',
   'Selected accounting standards']},
 {'row_number': 11, 'values': ['Sektor', None, 'Sector']},
 {'row_number': 12, 'values


TEBE 2023 Q3
Source file: TEBE_2023_Q3_FS.xlsx


[{'row_number': 1, 'values': ['[1000000] General information', None, None]},
 {'row_number': 2, 'values': [None, None, None]},
 {'row_number': 3, 'values': ['Informasi umum', None, 'General information']},
 {'row_number': 4, 'values': [None, 'CurrentYearInstant', None]},
 {'row_number': 5, 'values': ['Nama entitas', None, 'Entity name']},
 {'row_number': 6,
  'values': ['Penjelasan perubahan nama dari akhir periode laporan sebelumnya',
   None,
   'Explanation of change in name from the end of the preceding reporting period']},
 {'row_number': 7, 'values': ['Kode entitas', None, 'Entity code']},
 {'row_number': 8,
  'values': ['Nomor identifikasi entitas',
   None,
   'Entity identification number']},
 {'row_number': 9,
  'values': ['Industri utama entitas', None, 'Entity main industry']},
 {'row_number': 10,
  'values': ['Standar akutansi yang dipilih',
   None,
   'Selected accounting standards']},
 {'row_number': 11, 'values': ['Sektor', None, 'Sector']},
 {'row_number': 12, 'values

In [43]:
# CELL 35 - APPLY RETRY RESULTS TO FINAL CURRENCY / SCALE MAPPING

# Keep only successfully resolved retry rows
resolved_retry_found_df = (
    currency_scale_retry_df[
        currency_scale_retry_df["mapping_status"] == "FOUND"
    ]
    .copy()
)

print(
    "Resolved retry rows to apply:",
    len(resolved_retry_found_df)
)


# Use source_file as the update key
resolved_retry_found_df = (
    resolved_retry_found_df
    .set_index("source_file")
)

currency_scale_mapping_final_df = (
    currency_scale_mapping_df
    .copy()
    .set_index("source_file")
)


# Columns that should be replaced using the successful retry result
columns_to_update = [
    "currency",
    "unit_scale",
    "multiplier",
    "currency_row",
    "scale_row",
    "mapping_status"
]


for column in columns_to_update:
    currency_scale_mapping_final_df.loc[
        resolved_retry_found_df.index,
        column
    ] = resolved_retry_found_df[column]


# Return source_file to a normal column
currency_scale_mapping_final_df = (
    currency_scale_mapping_final_df
    .reset_index()
)


print(
    "Final mapping rows:",
    len(currency_scale_mapping_final_df)
)

print()

print(
    currency_scale_mapping_final_df[
        "mapping_status"
    ].value_counts(
        dropna=False
    )
)

Resolved retry rows to apply: 37
Final mapping rows: 15602

mapping_status
FOUND                           15600
CURRENCY_AND_SCALE_NOT_FOUND        2
Name: count, dtype: int64


In [44]:
# CELL 36 - FINAL CURRENCY / SCALE QUALITY CHECK

print("TOTAL ROWS")
print(
    len(currency_scale_mapping_final_df)
)

print("\nDUPLICATE SOURCE FILES")
print(
    currency_scale_mapping_final_df[
        "source_file"
    ].duplicated().sum()
)

print("\nMAPPING STATUS")
print(
    currency_scale_mapping_final_df[
        "mapping_status"
    ].value_counts(
        dropna=False
    )
)

print("\nCURRENCY")
print(
    currency_scale_mapping_final_df[
        "currency"
    ].value_counts(
        dropna=False
    )
)

print("\nUNIT SCALE")
print(
    currency_scale_mapping_final_df[
        "unit_scale"
    ].value_counts(
        dropna=False
    )
)

print("\nMULTIPLIER")
print(
    currency_scale_mapping_final_df[
        "multiplier"
    ].value_counts(
        dropna=False
    )
)

print("\nREMAINING UNRESOLVED")
display(
    currency_scale_mapping_final_df[
        currency_scale_mapping_final_df[
            "mapping_status"
        ] != "FOUND"
    ][
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "currency",
            "unit_scale",
            "multiplier",
            "mapping_status"
        ]
    ]
)

TOTAL ROWS
15602

DUPLICATE SOURCE FILES
0

MAPPING STATUS
mapping_status
FOUND                           15600
CURRENCY_AND_SCALE_NOT_FOUND        2
Name: count, dtype: int64

CURRENCY
currency
IDR    13708
USD     1892
NaN        2
Name: count, dtype: int64

UNIT SCALE
unit_scale
UNIT        10127
MILLION      3453
THOUSAND     1983
BILLION        37
NaN             2
Name: count, dtype: int64

MULTIPLIER
multiplier
1.000000e+00    10127
1.000000e+06     3453
1.000000e+03     1983
1.000000e+09       37
NaN                 2
Name: count, dtype: int64

REMAINING UNRESOLVED


,ticker,year,quarter,source_file,currency,unit_scale,multiplier,mapping_status
4056,CTRA,2023,Q2,CTRA_2023_Q2_FS.xlsx,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND
14014,TEBE,2023,Q3,TEBE_2023_Q3_FS.xlsx,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND


In [45]:
# CELL 37 - SAVE FINAL VALIDATED CURRENCY / SCALE MAPPING

FINAL_CURRENCY_SCALE_FILE = Path(
    "data/idx_financial_currency_scale_mapping_final.csv"
)


currency_scale_mapping_final_df.to_csv(
    FINAL_CURRENCY_SCALE_FILE,
    index=False
)


print(
    "Saved final validated mapping to:",
    FINAL_CURRENCY_SCALE_FILE
)

print(
    "Total rows:",
    len(currency_scale_mapping_final_df)
)

print(
    "Resolved rows:",
    (
        currency_scale_mapping_final_df[
            "mapping_status"
        ] == "FOUND"
    ).sum()
)

print(
    "Unresolved rows:",
    (
        currency_scale_mapping_final_df[
            "mapping_status"
        ] != "FOUND"
    ).sum()
)

Saved final validated mapping to: data\idx_financial_currency_scale_mapping_final.csv
Total rows: 15602
Resolved rows: 15600
Unresolved rows: 2


In [48]:
# CELL 38 - LOAD FINAL METRICS AND MERGE CURRENCY / SCALE MAPPING

FINAL_METRICS_FILE = Path(
    "data/idx_financial_current_metrics_final.csv"
)

final_metrics_df = pd.read_csv(
    FINAL_METRICS_FILE
)

print(
    "Final metric rows:",
    len(final_metrics_df)
)


# Prepare currency / scale mapping with a unique status column name
currency_scale_for_merge_df = (
    currency_scale_mapping_final_df[
        [
            "source_file",
            "currency",
            "unit_scale",
            "multiplier",
            "mapping_status"
        ]
    ]
    .copy()
    .rename(
        columns={
            "mapping_status": "currency_scale_status"
        }
    )
)


metrics_with_currency_scale_df = (
    final_metrics_df
    .merge(
        currency_scale_for_merge_df,
        on="source_file",
        how="left",
        validate="many_to_one"
    )
)


print(
    "Rows after merge:",
    len(metrics_with_currency_scale_df)
)

print(
    "Missing currency / scale mapping rows:",
    metrics_with_currency_scale_df[
        "currency_scale_status"
    ].isna().sum()
)

Final metric rows: 88400
Rows after merge: 88400
Missing currency / scale mapping rows: 0


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22608\2079827300.py:7: DtypeWarning: Columns (0: source_sheet, 1: period_type, 2: parsed_period_date, 3: latest_date, 4: earliest_date, 5: expected_period_date, 6: suspicious_duplicate, 7: is_preferred_sheet, 8: is_exact_period_match, 9: ticker_matches_filename, 10: identity_valid_candidate) have mixed types. Specify dtype option on import or set low_memory=False.
  final_metrics_df = pd.read_csv(


In [49]:
# CELL 39 - AUDIT MERGED CURRENCY / SCALE COVERAGE

print("CURRENCY / SCALE STATUS")
print(
    metrics_with_currency_scale_df[
        "currency_scale_status"
    ].value_counts(
        dropna=False
    )
)

print("\nCURRENCY")
print(
    metrics_with_currency_scale_df[
        "currency"
    ].value_counts(
        dropna=False
    )
)

print("\nUNIT SCALE")
print(
    metrics_with_currency_scale_df[
        "unit_scale"
    ].value_counts(
        dropna=False
    )
)

print("\nUNRESOLVED METRIC ROWS")

unresolved_metric_rows_df = (
    metrics_with_currency_scale_df[
        metrics_with_currency_scale_df[
            "currency_scale_status"
        ] != "FOUND"
    ]
    .copy()
)

print(
    "Total unresolved metric rows:",
    len(unresolved_metric_rows_df)
)

display(
    unresolved_metric_rows_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "source_file",
            "currency",
            "unit_scale",
            "multiplier",
            "currency_scale_status"
        ]
    ]
)

CURRENCY / SCALE STATUS
currency_scale_status
FOUND                           88396
CURRENCY_AND_SCALE_NOT_FOUND        4
Name: count, dtype: int64

CURRENCY
currency
IDR    77120
USD    11276
NaN        4
Name: count, dtype: int64

UNIT SCALE
unit_scale
UNIT        58894
MILLION     18218
THOUSAND    11082
BILLION       202
NaN             4
Name: count, dtype: int64

UNRESOLVED METRIC ROWS
Total unresolved metric rows: 4


,ticker,year,quarter,metric,selected_value,source_file,currency,unit_scale,multiplier,currency_scale_status
21417,CTRA,2023,Q2,operating_cash_flow,1.453784e+06,CTRA_2023_Q2_FS.xlsx,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND
79296,TEBE,2023,Q3,cash,3.876731e+08,TEBE_2023_Q3_FS.xlsx,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND
79297,TEBE,2023,Q3,total_assets,1.184340e+09,TEBE_2023_Q3_FS.xlsx,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND
79298,TEBE,2023,Q3,total_liabilities,1.137427e+08,TEBE_2023_Q3_FS.xlsx,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND


In [51]:
# CELL 40 - SPOT CHECK RAW VALUES BY UNIT SCALE

spot_check_df = (
    metrics_with_currency_scale_df[
        metrics_with_currency_scale_df[
            "currency_scale_status"
        ] == "FOUND"
    ]
    .groupby(
        "unit_scale",
        group_keys=False
    )
    .head(5)
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "source_file"
        ]
    ]
)

display(
    spot_check_df.sort_values(
        [
            "unit_scale",
            "ticker",
            "year",
            "quarter"
        ]
    )
)

,ticker,year,quarter,metric,selected_value,currency,unit_scale,multiplier,source_file
6180,ASII,2020,Q1,cash,2.925100e+04,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
6181,ASII,2020,Q1,gross_profit,1.208700e+04,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
6182,ASII,2020,Q1,operating_cash_flow,7.829000e+03,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
6183,ASII,2020,Q1,revenue,5.400200e+04,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
6184,ASII,2020,Q1,total_assets,3.667400e+05,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
12,AALI,2020,Q1,cash,1.470866e+06,IDR,MILLION,1.000000e+06,AALI_2020_Q1_FS.xlsx
13,AALI,2020,Q1,gross_profit,9.267910e+05,IDR,MILLION,1.000000e+06,AALI_2020_Q1_FS.xlsx
14,AALI,2020,Q1,operating_cash_flow,6.606100e+05,IDR,MILLION,1.000000e+06,AALI_2020_Q1_FS.xlsx
15,AALI,2020,Q1,revenue,4.796084e+06,IDR,MILLION,1.000000e+06,AALI_2020_Q1_FS.xlsx
16,AALI,2020,Q1,total_assets,2.921860e+07,IDR,MILLION,1.000000e+06,AALI_2020_Q1_FS.xlsx


In [52]:
# CELL 41 - INSPECT BILLION-SCALE METRICS

billion_metrics_df = (
    metrics_with_currency_scale_df[
        (
            metrics_with_currency_scale_df[
                "currency_scale_status"
            ] == "FOUND"
        )
        &
        (
            metrics_with_currency_scale_df[
                "unit_scale"
            ] == "BILLION"
        )
    ]
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "source_file"
        ]
    ]
    .copy()
)

print(
    "Total BILLION-scale metric rows:",
    len(billion_metrics_df)
)

display(
    billion_metrics_df.head(30)
)

Total BILLION-scale metric rows: 202


,ticker,year,quarter,metric,selected_value,currency,unit_scale,multiplier,source_file
6180,ASII,2020,Q1,cash,29251.0,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
6181,ASII,2020,Q1,gross_profit,12087.0,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
6182,ASII,2020,Q1,operating_cash_flow,7829.0,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
6183,ASII,2020,Q1,revenue,54002.0,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
6184,ASII,2020,Q1,total_assets,366740.0,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
6185,ASII,2020,Q1,total_liabilities,170836.0,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx
6186,ASII,2020,Q2,cash,42124.0,IDR,BILLION,1.000000e+09,ASII_2020_Q2_FS.xlsx
6187,ASII,2020,Q2,gross_profit,20174.0,IDR,BILLION,1.000000e+09,ASII_2020_Q2_FS.xlsx
6188,ASII,2020,Q2,operating_cash_flow,13966.0,IDR,BILLION,1.000000e+09,ASII_2020_Q2_FS.xlsx
6189,ASII,2020,Q2,revenue,89795.0,IDR,BILLION,1.000000e+09,ASII_2020_Q2_FS.xlsx


In [58]:
# CELL 42 - DIAGNOSTIC CHECK EXCEL NUMBER FORMAT
# Goal:
# Compare cases that SHOULD need scaling vs cases that may already
# contain full nominal values in the underlying Excel cell.

from openpyxl import load_workbook


diagnostic_cases = [
    # Expected to need scaling
    ("ASII", 2020, "Q1", "revenue"),
    ("AALI", 2020, "Q1", "revenue"),

    # Suspected already-full nominal values
    ("ASII", 2023, "Q4", "total_assets"),
    ("BMRI", 2023, "Q4", "total_assets"),
]


diagnostic_results = []


for ticker, year, quarter, metric in diagnostic_cases:

    print("\n" + "=" * 100)
    print(
        f"{ticker} {year} {quarter} | {metric}"
    )
    print("=" * 100)

    metric_match = metrics_with_currency_scale_df[
        (metrics_with_currency_scale_df["ticker"] == ticker)
        & (metrics_with_currency_scale_df["year"] == year)
        & (metrics_with_currency_scale_df["quarter"] == quarter)
        & (metrics_with_currency_scale_df["metric"] == metric)
    ]

    if metric_match.empty:
        print("METRIC ROW NOT FOUND")
        continue

    metric_row = metric_match.iloc[0]

    source_path = Path(
        metric_row["source_path"]
    )

    source_sheet = str(
        metric_row["source_sheet"]
    )

    row_number = int(
        metric_row["row_number"]
    )

    # The final selected value came from a specific numeric column.
    # We inspect all non-empty cells on that row so we can identify
    # the numeric cell and its Excel number format.
    wb = None

    try:
        wb = load_workbook(
            source_path,
            read_only=False,
            data_only=True
        )

        if source_sheet not in wb.sheetnames:
            print(
                "SOURCE SHEET NOT FOUND:",
                source_sheet
            )
            continue

        ws = wb[source_sheet]

        row_cells = list(
            ws[row_number]
        )

        candidate_cells = []

        for cell in row_cells:

            value = cell.value

            if value is None:
                continue

            if isinstance(
                value,
                (int, float)
            ):
                candidate_cells.append(
                    {
                        "coordinate": cell.coordinate,
                        "value": value,
                        "number_format": cell.number_format
                    }
                )

        print(
            "Selected value:",
            metric_row["selected_value"]
        )

        print(
            "Currency:",
            metric_row["currency"]
        )

        print(
            "Unit scale:",
            metric_row["unit_scale"]
        )

        print(
            "Multiplier:",
            metric_row["multiplier"]
        )

        print(
            "Source file:",
            metric_row["source_file"]
        )

        print(
            "Source sheet:",
            source_sheet
        )

        print(
            "Source row:",
            row_number
        )

        print("\nNumeric cells on source row:")

        for candidate in candidate_cells:
            print(
                candidate
            )

        diagnostic_results.append(
            {
                "ticker": ticker,
                "year": year,
                "quarter": quarter,
                "metric": metric,
                "selected_value": metric_row[
                    "selected_value"
                ],
                "currency": metric_row[
                    "currency"
                ],
                "unit_scale": metric_row[
                    "unit_scale"
                ],
                "multiplier": metric_row[
                    "multiplier"
                ],
                "source_file": metric_row[
                    "source_file"
                ],
                "source_sheet": source_sheet,
                "row_number": row_number,
                "numeric_cells": candidate_cells
            }
        )

    except Exception as e:
        print(
            "ERROR:",
            type(e).__name__,
            str(e)
        )

    finally:
        if wb is not None:
            wb.close()


diagnostic_number_format_df = pd.DataFrame(
    diagnostic_results
)

print("\n" + "=" * 100)
print("DIAGNOSTIC COMPLETE")
print("=" * 100)

display(
    diagnostic_number_format_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "source_file",
            "source_sheet",
            "row_number"
        ]
    ]
)


ASII 2020 Q1 | revenue
Selected value: 54002.0
Currency: IDR
Unit scale: BILLION
Multiplier: 1000000000.0
Source file: ASII_2020_Q1_FS.xlsx
Source sheet: 1321000
Source row: 5

Numeric cells on source row:
{'coordinate': 'B5', 'value': 54002.0, 'number_format': '#,##0;\\(#,##0\\)'}
{'coordinate': 'C5', 'value': 59607.0, 'number_format': '#,##0;\\(#,##0\\)'}

AALI 2020 Q1 | revenue
Selected value: 4796084.0
Currency: IDR
Unit scale: MILLION
Multiplier: 1000000.0
Source file: AALI_2020_Q1_FS.xlsx
Source sheet: 1321000
Source row: 5

Numeric cells on source row:
{'coordinate': 'B5', 'value': 4796084.0, 'number_format': '#,##0;\\(#,##0\\)'}
{'coordinate': 'C5', 'value': 4232857.0, 'number_format': '#,##0;\\(#,##0\\)'}

ASII 2023 Q4 | total_assets


e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Selected value: 445679000000000.0
Currency: IDR
Unit scale: BILLION
Multiplier: 1000000000.0
Source file: ASII_2023_Q4_FS.xlsx
Source sheet: 1210000
Source row: 128

Numeric cells on source row:
{'coordinate': 'B128', 'value': 445679000000000, 'number_format': '#,##0;\\(#,##0\\)'}
{'coordinate': 'C128', 'value': 413297000000000, 'number_format': '#,##0;\\(#,##0\\)'}

BMRI 2023 Q4 | total_assets


e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Selected value: 2174219449000000.0
Currency: IDR
Unit scale: MILLION
Multiplier: 1000000.0
Source file: BMRI_2023_Q4_FS.xlsx
Source sheet: 4220000
Source row: 119

Numeric cells on source row:
{'coordinate': 'B119', 'value': 2174219449000000, 'number_format': '#,##0;\\(#,##0\\)'}
{'coordinate': 'C119', 'value': 1992544687000000, 'number_format': '#,##0;\\(#,##0\\)'}

DIAGNOSTIC COMPLETE


,ticker,year,quarter,metric,selected_value,currency,unit_scale,multiplier,source_file,source_sheet,row_number
0,ASII,2020,Q1,revenue,5.400200e+04,IDR,BILLION,1.000000e+09,ASII_2020_Q1_FS.xlsx,1321000,5
1,AALI,2020,Q1,revenue,4.796084e+06,IDR,MILLION,1.000000e+06,AALI_2020_Q1_FS.xlsx,1321000,5
2,ASII,2023,Q4,total_assets,4.456790e+14,IDR,BILLION,1.000000e+09,ASII_2023_Q4_FS.xlsx,1210000,128
3,BMRI,2023,Q4,total_assets,2.174219e+15,IDR,MILLION,1.000000e+06,BMRI_2023_Q4_FS.xlsx,4220000,119


In [59]:
# CELL 43 - DETECT RAW VALUE SCALE SHIFTS ACROSS PERIODS

scale_shift_audit_df = (
    metrics_with_currency_scale_df[
        (
            metrics_with_currency_scale_df[
                "currency_scale_status"
            ] == "FOUND"
        )
        &
        (
            metrics_with_currency_scale_df[
                "selected_value"
            ].notna()
        )
    ]
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "source_file"
        ]
    ]
    .copy()
)


quarter_order = {
    "Q1": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4": 4
}

scale_shift_audit_df[
    "quarter_order"
] = scale_shift_audit_df[
    "quarter"
].map(
    quarter_order
)


scale_shift_audit_df = (
    scale_shift_audit_df
    .sort_values(
        [
            "ticker",
            "metric",
            "year",
            "quarter_order"
        ]
    )
    .reset_index(drop=True)
)


scale_shift_audit_df[
    "previous_value"
] = (
    scale_shift_audit_df
    .groupby(
        [
            "ticker",
            "metric"
        ]
    )[
        "selected_value"
    ]
    .shift(1)
)


scale_shift_audit_df[
    "raw_change_ratio"
] = (
    scale_shift_audit_df[
        "selected_value"
    ].abs()
    /
    scale_shift_audit_df[
        "previous_value"
    ].abs()
)


# Detect jumps roughly matching common presentation multipliers.
# These are diagnostic bands, NOT normalization rules yet.
suspicious_scale_shift_df = (
    scale_shift_audit_df[
        (
            scale_shift_audit_df[
                "raw_change_ratio"
            ] >= 100
        )
        |
        (
            scale_shift_audit_df[
                "raw_change_ratio"
            ] <= 0.01
        )
    ]
    .copy()
)


print(
    "Suspicious raw scale-shift rows:",
    len(suspicious_scale_shift_df)
)

print("\nBY YEAR / QUARTER")

print(
    suspicious_scale_shift_df
    .groupby(
        [
            "year",
            "quarter"
        ]
    )
    .size()
    .sort_values(
        ascending=False
    )
    .head(30)
)


print("\nBY UNIT SCALE")

print(
    suspicious_scale_shift_df[
        "unit_scale"
    ].value_counts(
        dropna=False
    )
)


display(
    suspicious_scale_shift_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "previous_value",
            "selected_value",
            "raw_change_ratio",
            "currency",
            "unit_scale",
            "multiplier",
            "source_file"
        ]
    ]
    .head(100)
)

Suspicious raw scale-shift rows: 1837

BY YEAR / QUARTER
year  quarter
2023  Q4         472
2024  Q2         365
      Q1         355
2021  Q1         114
2025  Q1          65
2021  Q4          64
2024  Q4          61
2022  Q1          61
2023  Q1          50
2024  Q3          44
2020  Q4          32
2023  Q2          31
      Q3          24
2021  Q3          20
2022  Q2          17
2020  Q2          16
2022  Q4          15
      Q3          14
2021  Q2          11
2020  Q3           6
dtype: int64

BY UNIT SCALE
unit_scale
MILLION     1038
UNIT         603
THOUSAND     186
BILLION       10
Name: count, dtype: int64


,ticker,year,quarter,metric,previous_value,selected_value,raw_change_ratio,currency,unit_scale,multiplier,source_file
27,AALI,2023,Q4,cash,2.522256e+06,2.089508e+12,8.284282e+05,IDR,MILLION,1000000.0,AALI_2023_Q4_FS.xlsx
28,AALI,2024,Q2,cash,2.089508e+12,3.987501e+06,1.908344e-06,IDR,MILLION,1000000.0,AALI_2024_Q2_FS.xlsx
47,AALI,2023,Q4,gross_profit,1.949932e+06,2.770980e+12,1.421065e+06,IDR,MILLION,1000000.0,AALI_2023_Q4_FS.xlsx
48,AALI,2024,Q2,gross_profit,2.770980e+12,1.283726e+06,4.632751e-07,IDR,MILLION,1000000.0,AALI_2024_Q2_FS.xlsx
67,AALI,2023,Q4,operating_cash_flow,2.270148e+06,2.538738e+12,1.118314e+06,IDR,MILLION,1000000.0,AALI_2023_Q4_FS.xlsx
...,...,...,...,...,...,...,...,...,...,...,...
5036,ARGO,2022,Q1,total_assets,7.870470e+07,1.098039e+12,1.395137e+04,IDR,UNIT,1.0,ARGO_2022_Q1_FS.xlsx
5057,ARGO,2022,Q1,total_liabilities,1.718327e+08,2.468423e+12,1.436527e+04,IDR,UNIT,1.0,ARGO_2022_Q1_FS.xlsx
5721,ARTI,2023,Q4,operating_cash_flow,-8.670402e+07,-3.288674e+10,3.792989e+02,IDR,UNIT,1.0,ARTI_2023_Q4_FS.xlsx
5785,ARTO,2020,Q3,operating_cash_flow,-2.616399e+11,-3.646410e+05,1.393675e-06,IDR,MILLION,1000000.0,ARTO_2020_Q3_FS.xlsx


In [60]:
# CELL 44 - BUILD SOURCE FILE SCALE MODE EVIDENCE

scale_mode_df = (
    metrics_with_currency_scale_df[
        (
            metrics_with_currency_scale_df["currency_scale_status"] == "FOUND"
        )
        &
        (
            metrics_with_currency_scale_df["selected_value"].notna()
        )
    ]
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "unit_scale",
            "multiplier",
            "source_file"
        ]
    ]
    .copy()
)

quarter_order = {
    "Q1": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4": 4
}

scale_mode_df["quarter_order"] = (
    scale_mode_df["quarter"].map(quarter_order)
)

scale_mode_df = (
    scale_mode_df
    .sort_values(
        [
            "ticker",
            "metric",
            "year",
            "quarter_order"
        ]
    )
    .reset_index(drop=True)
)


# Previous value from same ticker + metric
scale_mode_df["previous_value"] = (
    scale_mode_df
    .groupby(
        [
            "ticker",
            "metric"
        ]
    )["selected_value"]
    .shift(1)
)

scale_mode_df["previous_multiplier"] = (
    scale_mode_df
    .groupby(
        [
            "ticker",
            "metric"
        ]
    )["multiplier"]
    .shift(1)
)


# Absolute ratio between consecutive raw values
scale_mode_df["raw_ratio"] = (
    scale_mode_df["selected_value"].abs()
    /
    scale_mode_df["previous_value"].abs()
)


# Compare ratio against the current presentation multiplier
scale_mode_df["ratio_vs_multiplier"] = (
    scale_mode_df["raw_ratio"]
    /
    scale_mode_df["multiplier"]
)

scale_mode_df["ratio_vs_inverse_multiplier"] = (
    scale_mode_df["raw_ratio"]
    *
    scale_mode_df["multiplier"]
)


def classify_scale_transition(row):

    ratio = row["raw_ratio"]
    multiplier = row["multiplier"]

    if pd.isna(ratio) or pd.isna(multiplier):
        return "NO_EVIDENCE"

    if multiplier == 1:
        return "NO_SCALE"

    # Current raw value is roughly multiplier times previous value:
    # likely current file stores FULL NOMINAL.
    if (
        ratio >= multiplier * 0.1
        and ratio <= multiplier * 10
    ):
        return "CURRENT_LIKELY_FULL_NOMINAL"

    # Current raw value is roughly previous / multiplier:
    # likely current file stores PRESENTATION-SCALED value.
    inverse = 1 / multiplier

    if (
        ratio >= inverse * 0.1
        and ratio <= inverse * 10
    ):
        return "CURRENT_LIKELY_PRESENTATION_SCALED"

    return "NO_CLEAR_SCALE_SHIFT"


scale_mode_df["transition_evidence"] = (
    scale_mode_df.apply(
        classify_scale_transition,
        axis=1
    )
)


print("TRANSITION EVIDENCE")
print(
    scale_mode_df[
        "transition_evidence"
    ].value_counts()
)


print("\nEVIDENCE BY SOURCE FILE")

source_file_scale_evidence_df = (
    scale_mode_df
    .groupby(
        [
            "source_file",
            "ticker",
            "year",
            "quarter",
            "unit_scale",
            "multiplier"
        ],
        dropna=False
    )["transition_evidence"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

display(
    source_file_scale_evidence_df.head(100)
)

TRANSITION EVIDENCE
transition_evidence
NO_SCALE                              54957
NO_CLEAR_SCALE_SHIFT                  27150
NO_EVIDENCE                            5463
CURRENT_LIKELY_FULL_NOMINAL             444
CURRENT_LIKELY_PRESENTATION_SCALED      382
Name: count, dtype: int64

EVIDENCE BY SOURCE FILE


transition_evidence,source_file,ticker,year,quarter,unit_scale,multiplier,CURRENT_LIKELY_FULL_NOMINAL,CURRENT_LIKELY_PRESENTATION_SCALED,NO_CLEAR_SCALE_SHIFT,NO_EVIDENCE,NO_SCALE
0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI,2024,Q4,THOUSAND,1000.0,0,0,0,6,0
1,AADI_2025_Q1_FS.xlsx,AADI,2025,Q1,THOUSAND,1000.0,0,0,6,0,0
2,AALI_2020_Q1_FS.xlsx,AALI,2020,Q1,MILLION,1000000.0,0,0,0,6,0
3,AALI_2020_Q2_FS.xlsx,AALI,2020,Q2,MILLION,1000000.0,0,0,6,0,0
4,AALI_2020_Q3_FS.xlsx,AALI,2020,Q3,MILLION,1000000.0,0,0,6,0,0
...,...,...,...,...,...,...,...,...,...,...,...
95,ACES_2023_Q1_FS.xlsx,ACES,2023,Q1,UNIT,1.0,0,0,0,0,6
96,ACES_2023_Q2_FS.xlsx,ACES,2023,Q2,UNIT,1.0,0,0,0,0,6
97,ACES_2023_Q3_FS.xlsx,ACES,2023,Q3,UNIT,1.0,0,0,0,0,6
98,ACES_2023_Q4_FS.xlsx,ACES,2023,Q4,UNIT,1.0,0,0,0,0,6


In [61]:
# CELL 45 - PAIRWISE SCALE MODE CONTINUITY AUDIT

continuity_df = (
    metrics_with_currency_scale_df[
        (
            metrics_with_currency_scale_df[
                "currency_scale_status"
            ] == "FOUND"
        )
        &
        (
            metrics_with_currency_scale_df[
                "selected_value"
            ].notna()
        )
    ]
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "unit_scale",
            "multiplier",
            "source_file"
        ]
    ]
    .copy()
)


quarter_order = {
    "Q1": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4": 4
}


continuity_df["quarter_order"] = (
    continuity_df["quarter"].map(
        quarter_order
    )
)


continuity_df = (
    continuity_df
    .sort_values(
        [
            "ticker",
            "metric",
            "year",
            "quarter_order"
        ]
    )
    .reset_index(drop=True)
)


grouped = continuity_df.groupby(
    [
        "ticker",
        "metric"
    ]
)


continuity_df["previous_value"] = (
    grouped[
        "selected_value"
    ]
    .shift(1)
)


continuity_df["previous_multiplier"] = (
    grouped[
        "multiplier"
    ]
    .shift(1)
)


continuity_df["previous_source_file"] = (
    grouped[
        "source_file"
    ]
    .shift(1)
)


def safe_log_distance(
    previous_value,
    current_value
):
    if (
        pd.isna(previous_value)
        or pd.isna(current_value)
    ):
        return np.nan

    previous_value = abs(
        float(previous_value)
    )

    current_value = abs(
        float(current_value)
    )

    if (
        previous_value == 0
        or current_value == 0
    ):
        return np.nan

    return abs(
        np.log10(
            current_value
            /
            previous_value
        )
    )


pairwise_results = []


for _, row in continuity_df.iterrows():

    previous_value = row[
        "previous_value"
    ]

    current_value = row[
        "selected_value"
    ]

    previous_multiplier = row[
        "previous_multiplier"
    ]

    current_multiplier = row[
        "multiplier"
    ]

    if (
        pd.isna(previous_value)
        or pd.isna(previous_multiplier)
        or pd.isna(current_multiplier)
    ):
        continue


    candidates = {
        "PREV_PRESENT_CURRENT_PRESENT": (
            previous_value
            * previous_multiplier,
            current_value
            * current_multiplier
        ),

        "PREV_PRESENT_CURRENT_FULL": (
            previous_value
            * previous_multiplier,
            current_value
        ),

        "PREV_FULL_CURRENT_PRESENT": (
            previous_value,
            current_value
            * current_multiplier
        ),

        "PREV_FULL_CURRENT_FULL": (
            previous_value,
            current_value
        )
    }


    scores = {
        mode:
        safe_log_distance(
            values[0],
            values[1]
        )
        for mode, values
        in candidates.items()
    }


    valid_scores = {
        mode: score
        for mode, score
        in scores.items()
        if not pd.isna(score)
    }

    if not valid_scores:
        continue


    ranked_modes = sorted(
        valid_scores.items(),
        key=lambda item: item[1]
    )


    best_mode = ranked_modes[0][0]
    best_score = ranked_modes[0][1]

    second_score = (
        ranked_modes[1][1]
        if len(ranked_modes) > 1
        else np.nan
    )


    score_margin = (
        second_score - best_score
        if not pd.isna(second_score)
        else np.nan
    )


    if best_mode in [
        "PREV_PRESENT_CURRENT_FULL",
        "PREV_FULL_CURRENT_FULL"
    ]:
        current_mode_vote = "FULL_NOMINAL"

    else:
        current_mode_vote = (
            "PRESENTATION_SCALED"
        )


    if best_mode in [
        "PREV_PRESENT_CURRENT_PRESENT",
        "PREV_PRESENT_CURRENT_FULL"
    ]:
        previous_mode_vote = (
            "PRESENTATION_SCALED"
        )

    else:
        previous_mode_vote = (
            "FULL_NOMINAL"
        )


    pairwise_results.append(
        {
            "ticker": row["ticker"],
            "metric": row["metric"],

            "previous_source_file":
                row[
                    "previous_source_file"
                ],

            "current_source_file":
                row[
                    "source_file"
                ],

            "best_mode":
                best_mode,

            "best_score":
                best_score,

            "score_margin":
                score_margin,

            "previous_mode_vote":
                previous_mode_vote,

            "current_mode_vote":
                current_mode_vote
        }
    )


pairwise_scale_mode_df = (
    pd.DataFrame(
        pairwise_results
    )
)


print(
    "Pairwise comparisons:",
    len(pairwise_scale_mode_df)
)


print("\nBEST MODE COUNTS")

print(
    pairwise_scale_mode_df[
        "best_mode"
    ].value_counts()
)


print("\nCURRENT MODE VOTES")

print(
    pairwise_scale_mode_df[
        "current_mode_vote"
    ].value_counts()
)


display(
    pairwise_scale_mode_df.head(50)
)

Pairwise comparisons: 82900

BEST MODE COUNTS
best_mode
PREV_PRESENT_CURRENT_PRESENT    81558
PREV_PRESENT_CURRENT_FULL         533
PREV_FULL_CURRENT_PRESENT         506
PREV_FULL_CURRENT_FULL            303
Name: count, dtype: int64

CURRENT MODE VOTES
current_mode_vote
PRESENTATION_SCALED    82064
FULL_NOMINAL             836
Name: count, dtype: int64


,ticker,metric,previous_source_file,current_source_file,best_mode,best_score,score_margin,previous_mode_vote,current_mode_vote
0,AADI,cash,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2025_Q1_FS.xlsx,PREV_PRESENT_CURRENT_PRESENT,0.048462,0.000000,PRESENTATION_SCALED,PRESENTATION_SCALED
1,AADI,gross_profit,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2025_Q1_FS.xlsx,PREV_PRESENT_CURRENT_PRESENT,0.625280,0.000000,PRESENTATION_SCALED,PRESENTATION_SCALED
2,AADI,operating_cash_flow,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2025_Q1_FS.xlsx,PREV_PRESENT_CURRENT_PRESENT,0.564554,0.000000,PRESENTATION_SCALED,PRESENTATION_SCALED
3,AADI,revenue,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2025_Q1_FS.xlsx,PREV_PRESENT_CURRENT_PRESENT,0.659762,0.000000,PRESENTATION_SCALED,PRESENTATION_SCALED
4,AADI,total_assets,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2025_Q1_FS.xlsx,PREV_PRESENT_CURRENT_PRESENT,0.012117,0.000000,PRESENTATION_SCALED,PRESENTATION_SCALED
5,AADI,total_liabilities,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,AADI_2025_Q1_FS.xlsx,PREV_PRESENT_CURRENT_PRESENT,0.050679,0.000000,PRESENTATION_SCALED,PRESENTATION_SCALED
6,AALI,cash,AALI_2020_Q1_FS.xlsx,AALI_2020_Q2_FS.xlsx,PREV_PRESENT_CURRENT_PRESENT,0.106077,0.000000,PRESENTATION_SCALED,PRESENTATION_SCALED
7,AALI,cash,AALI_2020_Q2_FS.xlsx,AALI_2020_Q3_FS.xlsx,PREV_PRESENT_CURRENT_PRESENT,0.000000,0.000000,PRESENTATION_SCALED,PRESENTATION_SCALED
8,AALI,cash,AALI_2020_Q3_FS.xlsx,AALI_2020_Q4_FS.xlsx,PREV_PRESENT_CURRENT_PRESENT,0.070762,0.000000,PRESENTATION_SCALED,PRESENTATION_SCALED
9,AALI,cash,AALI_2020_Q4_FS.xlsx,AALI_2021_Q1_FS.xlsx,PREV_PRESENT_CURRENT_PRESENT,0.267146,0.000000,PRESENTATION_SCALED,PRESENTATION_SCALED


In [62]:
# CELL 46 - AGGREGATE SCALE MODE VOTES PER SOURCE FILE

vote_rows = []


for _, row in pairwise_scale_mode_df.iterrows():

    # Vote for previous file
    vote_rows.append(
        {
            "source_file": row["previous_source_file"],
            "mode_vote": row["previous_mode_vote"],
            "score_margin": row["score_margin"]
        }
    )

    # Vote for current file
    vote_rows.append(
        {
            "source_file": row["current_source_file"],
            "mode_vote": row["current_mode_vote"],
            "score_margin": row["score_margin"]
        }
    )


source_file_votes_df = pd.DataFrame(
    vote_rows
)


source_file_vote_summary_df = (
    source_file_votes_df
    .groupby(
        [
            "source_file",
            "mode_vote"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)


# Make sure both columns exist
if "FULL_NOMINAL" not in source_file_vote_summary_df.columns:
    source_file_vote_summary_df["FULL_NOMINAL"] = 0

if "PRESENTATION_SCALED" not in source_file_vote_summary_df.columns:
    source_file_vote_summary_df["PRESENTATION_SCALED"] = 0


source_file_vote_summary_df[
    "total_votes"
] = (
    source_file_vote_summary_df[
        "FULL_NOMINAL"
    ]
    +
    source_file_vote_summary_df[
        "PRESENTATION_SCALED"
    ]
)


source_file_vote_summary_df[
    "full_nominal_share"
] = (
    source_file_vote_summary_df[
        "FULL_NOMINAL"
    ]
    /
    source_file_vote_summary_df[
        "total_votes"
    ]
)


source_file_vote_summary_df[
    "presentation_scaled_share"
] = (
    source_file_vote_summary_df[
        "PRESENTATION_SCALED"
    ]
    /
    source_file_vote_summary_df[
        "total_votes"
    ]
)


def classify_file_mode(row):

    full_votes = row["FULL_NOMINAL"]
    presentation_votes = row[
        "PRESENTATION_SCALED"
    ]
    total_votes = row["total_votes"]

    if total_votes == 0:
        return "NO_EVIDENCE"

    # Require reasonably strong agreement
    if (
        full_votes >= 2
        and row["full_nominal_share"] >= 0.80
    ):
        return "FULL_NOMINAL"

    if (
        presentation_votes >= 2
        and row[
            "presentation_scaled_share"
        ] >= 0.80
    ):
        return "PRESENTATION_SCALED"

    return "AMBIGUOUS"


source_file_vote_summary_df[
    "file_scale_mode"
] = (
    source_file_vote_summary_df.apply(
        classify_file_mode,
        axis=1
    )
)


print("FILE SCALE MODE")
print(
    source_file_vote_summary_df[
        "file_scale_mode"
    ].value_counts(
        dropna=False
    )
)


print("\nKNOWN EXAMPLE FILES")

example_files = [
    "ASII_2020_Q1_FS.xlsx",
    "AALI_2020_Q1_FS.xlsx",
    "ASII_2023_Q4_FS.xlsx",
    "BMRI_2023_Q4_FS.xlsx",
    "AALI_2023_Q4_FS.xlsx",
    "AALI_2024_Q2_FS.xlsx"
]

display(
    source_file_vote_summary_df[
        source_file_vote_summary_df[
            "source_file"
        ].isin(
            example_files
        )
    ]
    [
        [
            "source_file",
            "FULL_NOMINAL",
            "PRESENTATION_SCALED",
            "total_votes",
            "full_nominal_share",
            "presentation_scaled_share",
            "file_scale_mode"
        ]
    ]
)

FILE SCALE MODE
file_scale_mode
PRESENTATION_SCALED    15398
AMBIGUOUS                109
FULL_NOMINAL              88
Name: count, dtype: int64

KNOWN EXAMPLE FILES


mode_vote,source_file,FULL_NOMINAL,PRESENTATION_SCALED,total_votes,full_nominal_share,presentation_scaled_share,file_scale_mode
2,AALI_2020_Q1_FS.xlsx,0,6,6,0.0,1.0,PRESENTATION_SCALED
17,AALI_2023_Q4_FS.xlsx,12,0,12,1.0,0.0,FULL_NOMINAL
18,AALI_2024_Q2_FS.xlsx,0,12,12,0.0,1.0,PRESENTATION_SCALED
1121,ASII_2020_Q1_FS.xlsx,0,6,6,0.0,1.0,PRESENTATION_SCALED
1136,ASII_2023_Q4_FS.xlsx,10,0,10,1.0,0.0,FULL_NOMINAL
2511,BMRI_2023_Q4_FS.xlsx,6,0,6,1.0,0.0,FULL_NOMINAL


In [63]:
# CELL 47 - BUILD EXTERNAL VALIDATION SAMPLE

# Attach ticker/year/quarter/unit info to file-level classification
file_metadata_for_validation_df = (
    metrics_with_currency_scale_df[
        [
            "source_file",
            "ticker",
            "year",
            "quarter",
            "currency",
            "unit_scale",
            "multiplier"
        ]
    ]
    .drop_duplicates(
        subset=["source_file"]
    )
)


external_validation_candidates_df = (
    source_file_vote_summary_df
    .merge(
        file_metadata_for_validation_df,
        on="source_file",
        how="left",
        validate="one_to_one"
    )
)


# Prefer files that are meaningful for checking:
# - clear FULL_NOMINAL
# - clear PRESENTATION_SCALED
# - AMBIGUOUS
# - non-UNIT scales, because UNIT has no scaling issue
external_validation_candidates_df = (
    external_validation_candidates_df[
        external_validation_candidates_df[
            "unit_scale"
        ].isin(
            [
                "THOUSAND",
                "MILLION",
                "BILLION"
            ]
        )
    ]
    .copy()
)


print("CANDIDATES BY FILE SCALE MODE")

print(
    external_validation_candidates_df[
        "file_scale_mode"
    ].value_counts(
        dropna=False
    )
)


# Known / useful large-company cases
priority_tickers = [
    "ASII",
    "TLKM",
    "BMRI",
    "BBRI",
    "BBCA",
    "BBNI",
    "AALI"
]


priority_validation_df = (
    external_validation_candidates_df[
        external_validation_candidates_df[
            "ticker"
        ].isin(priority_tickers)
    ]
    .sort_values(
        [
            "ticker",
            "year",
            "quarter"
        ]
    )
)


print("\nPRIORITY VALIDATION FILES")

display(
    priority_validation_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "currency",
            "unit_scale",
            "FULL_NOMINAL",
            "PRESENTATION_SCALED",
            "full_nominal_share",
            "presentation_scaled_share",
            "file_scale_mode"
        ]
    ]
    .head(100)
)

CANDIDATES BY FILE SCALE MODE
file_scale_mode
PRESENTATION_SCALED    5273
AMBIGUOUS               109
FULL_NOMINAL             88
Name: count, dtype: int64

PRIORITY VALIDATION FILES


,ticker,year,quarter,source_file,currency,unit_scale,FULL_NOMINAL,PRESENTATION_SCALED,full_nominal_share,presentation_scaled_share,file_scale_mode
2,AALI,2020,Q1,AALI_2020_Q1_FS.xlsx,IDR,MILLION,0,6,0.0,1.0,PRESENTATION_SCALED
3,AALI,2020,Q2,AALI_2020_Q2_FS.xlsx,IDR,MILLION,0,12,0.0,1.0,PRESENTATION_SCALED
4,AALI,2020,Q3,AALI_2020_Q3_FS.xlsx,IDR,MILLION,0,12,0.0,1.0,PRESENTATION_SCALED
5,AALI,2020,Q4,AALI_2020_Q4_FS.xlsx,IDR,MILLION,0,12,0.0,1.0,PRESENTATION_SCALED
6,AALI,2021,Q1,AALI_2021_Q1_FS.xlsx,IDR,MILLION,0,12,0.0,1.0,PRESENTATION_SCALED
...,...,...,...,...,...,...,...,...,...,...,...
1736,BBRI,2025,Q1,BBRI_2025_Q1_FS.xlsx,IDR,MILLION,0,3,0.0,1.0,PRESENTATION_SCALED
2496,BMRI,2020,Q1,BMRI_2020_Q1_FS.xlsx,IDR,MILLION,0,3,0.0,1.0,PRESENTATION_SCALED
2497,BMRI,2020,Q2,BMRI_2020_Q2_FS.xlsx,IDR,MILLION,0,6,0.0,1.0,PRESENTATION_SCALED
2498,BMRI,2020,Q3,BMRI_2020_Q3_FS.xlsx,IDR,MILLION,0,6,0.0,1.0,PRESENTATION_SCALED


In [64]:
# CELL 48 - SELECT EXTERNAL VALIDATION SAMPLE

validation_metric_priority = [
    "total_assets",
    "revenue",
    "total_liabilities"
]


validation_rows_df = (
    metrics_with_currency_scale_df
    .merge(
        external_validation_candidates_df[
            [
                "source_file",
                "file_scale_mode"
            ]
        ],
        on="source_file",
        how="inner",
        validate="many_to_one"
    )
)


validation_rows_df = (
    validation_rows_df[
        validation_rows_df[
            "metric"
        ].isin(
            validation_metric_priority
        )
    ]
    .copy()
)


# Candidate normalized value according to classifier
validation_rows_df[
    "candidate_normalized_value"
] = np.where(
    validation_rows_df[
        "file_scale_mode"
    ] == "FULL_NOMINAL",
    validation_rows_df[
        "selected_value"
    ],
    np.where(
        validation_rows_df[
            "file_scale_mode"
        ] == "PRESENTATION_SCALED",
        validation_rows_df[
            "selected_value"
        ]
        *
        validation_rows_df[
            "multiplier"
        ],
        np.nan
    )
)


def take_sample(mode, n=10):
    df = (
        validation_rows_df[
            validation_rows_df[
                "file_scale_mode"
            ] == mode
        ]
        .sort_values(
            [
                "year",
                "ticker",
                "quarter",
                "metric"
            ],
            ascending=[
                False,
                True,
                True,
                True
            ]
        )
    )

    return (
        df
        .drop_duplicates(
            subset=[
                "source_file"
            ]
        )
        .head(n)
    )


full_nominal_sample_df = take_sample(
    "FULL_NOMINAL",
    10
)

presentation_sample_df = take_sample(
    "PRESENTATION_SCALED",
    10
)

ambiguous_sample_df = take_sample(
    "AMBIGUOUS",
    10
)


external_validation_sample_df = pd.concat(
    [
        full_nominal_sample_df,
        presentation_sample_df,
        ambiguous_sample_df
    ],
    ignore_index=True
)


print(
    "External validation sample rows:",
    len(external_validation_sample_df)
)

print("\nBY MODE")

print(
    external_validation_sample_df[
        "file_scale_mode"
    ].value_counts(
        dropna=False
    )
)


display(
    external_validation_sample_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "candidate_normalized_value",
            "file_scale_mode",
            "source_file"
        ]
    ]
)

External validation sample rows: 30

BY MODE
file_scale_mode
FULL_NOMINAL           10
PRESENTATION_SCALED    10
AMBIGUOUS              10
Name: count, dtype: int64


,ticker,year,quarter,metric,selected_value,currency,unit_scale,multiplier,candidate_normalized_value,file_scale_mode,source_file
0,ASII,2025,Q1,revenue,8.336100e+04,IDR,BILLION,1.000000e+09,8.336100e+04,FULL_NOMINAL,ASII_2025_Q1_FS.xlsx
1,IMJS,2025,Q1,revenue,1.392720e+06,IDR,MILLION,1.000000e+06,1.392720e+06,FULL_NOMINAL,IMJS_2025_Q1_FS.xlsx
2,ASII,2024,Q4,revenue,5.319582e+06,USD,THOUSAND,1.000000e+03,5.319582e+06,FULL_NOMINAL,ASII_2024_Q4_FS.xlsx
3,BALI,2024,Q4,revenue,5.319582e+06,USD,THOUSAND,1.000000e+03,5.319582e+06,FULL_NOMINAL,BALI_2024_Q4_FS.xlsx
4,AALI,2023,Q4,revenue,2.074547e+13,IDR,MILLION,1.000000e+06,2.074547e+13,FULL_NOMINAL,AALI_2023_Q4_FS.xlsx
5,ACST,2023,Q4,total_assets,2.608782e+12,IDR,MILLION,1.000000e+06,2.608782e+12,FULL_NOMINAL,ACST_2023_Q4_FS.xlsx
6,ADRO,2023,Q4,total_assets,1.047271e+10,USD,THOUSAND,1.000000e+03,1.047271e+10,FULL_NOMINAL,ADRO_2023_Q4_FS.xlsx
7,AMOR,2023,Q2,total_assets,3.612300e+11,IDR,MILLION,1.000000e+06,3.612300e+11,FULL_NOMINAL,AMOR_2023_Q2_FS.xlsx
8,ASGR,2023,Q4,revenue,2.968952e+12,IDR,MILLION,1.000000e+06,2.968952e+12,FULL_NOMINAL,ASGR_2023_Q4_FS.xlsx
9,ASII,2023,Q4,total_assets,4.456790e+14,IDR,BILLION,1.000000e+09,4.456790e+14,FULL_NOMINAL,ASII_2023_Q4_FS.xlsx


In [1]:
# RECOVERY CELL - RUN THIS AFTER RESTART
# Reload only the saved data needed to continue from Cell 49

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
from tqdm.auto import tqdm


FINAL_METRICS_FILE = Path(
    "data/idx_financial_current_metrics_final.csv"
)

FINAL_CURRENCY_SCALE_FILE = Path(
    "data/idx_financial_currency_scale_mapping_final.csv"
)


# Load saved final metric data
final_metrics_df = pd.read_csv(
    FINAL_METRICS_FILE,
    low_memory=False
)

# Load saved final currency / scale mapping
currency_scale_mapping_final_df = pd.read_csv(
    FINAL_CURRENCY_SCALE_FILE,
    low_memory=False
)


# Rebuild the unique source-file table
# This replaces the old source_files_df that disappeared after restart.
source_files_df = (
    final_metrics_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "source_path"
        ]
    ]
    .drop_duplicates(
        subset=["source_file"]
    )
    .reset_index(drop=True)
)


print(
    "Final metric rows:",
    len(final_metrics_df)
)

print(
    "Currency / scale mapping rows:",
    len(currency_scale_mapping_final_df)
)

print(
    "Unique source files:",
    len(source_files_df)
)

e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Final metric rows: 88400
Currency / scale mapping rows: 15602
Unique source files: 15602


In [2]:
# CELL 49 - FAST GLOBAL DUPLICATE WORKBOOK AUDIT WITH PROGRESS BAR

import hashlib
from tqdm.auto import tqdm


duplicate_audit_df = (
    source_files_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "source_path"
        ]
    ]
    .copy()
)


def get_file_size(path_value):
    try:
        path = Path(path_value)

        if not path.exists():
            return np.nan

        return path.stat().st_size

    except Exception:
        return np.nan


# Progress bar for file-size scan
file_sizes = []

for path_value in tqdm(
    duplicate_audit_df["source_path"],
    total=len(duplicate_audit_df),
    desc="Checking file sizes"
):
    file_sizes.append(
        get_file_size(path_value)
    )

duplicate_audit_df["file_size"] = file_sizes


print(
    "Total source files:",
    len(duplicate_audit_df)
)

print(
    "Files missing on disk:",
    duplicate_audit_df[
        "file_size"
    ].isna().sum()
)


# Only files with identical byte size can be exact duplicates
size_counts = (
    duplicate_audit_df[
        "file_size"
    ].value_counts()
)

duplicate_sizes = set(
    size_counts[
        size_counts > 1
    ].index
)


hash_candidates_df = (
    duplicate_audit_df[
        duplicate_audit_df[
            "file_size"
        ].isin(
            duplicate_sizes
        )
    ]
    .copy()
)


print(
    "Files requiring SHA-256:",
    len(hash_candidates_df)
)


def calculate_sha256(path_value):
    try:
        sha = hashlib.sha256()

        with open(
            Path(path_value),
            "rb"
        ) as f:

            while True:
                chunk = f.read(
                    1024 * 1024
                )

                if not chunk:
                    break

                sha.update(chunk)

        return sha.hexdigest()

    except Exception:
        return None


# Progress bar for hashing
sha256_results = []

for path_value in tqdm(
    hash_candidates_df["source_path"],
    total=len(hash_candidates_df),
    desc="Calculating SHA-256"
):
    sha256_results.append(
        calculate_sha256(path_value)
    )

hash_candidates_df["sha256"] = sha256_results


print(
    "Successfully hashed:",
    hash_candidates_df[
        "sha256"
    ].notna().sum()
)

Checking file sizes: 100%|██████████| 15602/15602 [00:09<00:00, 1643.40it/s]


Total source files: 15602
Files missing on disk: 0
Files requiring SHA-256: 6358


Calculating SHA-256: 100%|██████████| 6358/6358 [02:04<00:00, 51.06it/s]

Successfully hashed: 6358


In [3]:
# CELL 50 - FIND EXACT DUPLICATES ACROSS DIFFERENT TICKERS

hash_ticker_counts = (
    hash_candidates_df[
        hash_candidates_df[
            "sha256"
        ].notna()
    ]
    .groupby(
        "sha256"
    )[
        "ticker"
    ]
    .nunique()
)


cross_ticker_duplicate_hashes = set(
    hash_ticker_counts[
        hash_ticker_counts > 1
    ].index
)


cross_ticker_duplicates_df = (
    hash_candidates_df[
        hash_candidates_df[
            "sha256"
        ].isin(
            cross_ticker_duplicate_hashes
        )
    ]
    .copy()
    .sort_values(
        [
            "sha256",
            "ticker",
            "year",
            "quarter"
        ]
    )
)


print(
    "Exact duplicate hashes shared across tickers:",
    len(
        cross_ticker_duplicate_hashes
    )
)

print(
    "Affected source files:",
    len(
        cross_ticker_duplicates_df
    )
)

print(
    "Affected tickers:",
    cross_ticker_duplicates_df[
        "ticker"
    ].nunique()
)


display(
    cross_ticker_duplicates_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "file_size",
            "sha256"
        ]
    ]
    .head(200)
)

Exact duplicate hashes shared across tickers: 123
Affected source files: 968
Affected tickers: 88


,ticker,year,quarter,source_file,file_size,sha256
2863,BRMS,2024,Q2,BRMS_2024_Q2_FS.xlsx,446166,0146ce3551b1e904488a6a90e08cac50aefeb9a35be55a...
3210,BUMI,2024,Q2,BUMI_2024_Q2_FS.xlsx,446166,0146ce3551b1e904488a6a90e08cac50aefeb9a35be55a...
2233,BIPI,2024,Q4,BIPI_2024_Q4_FS.xlsx,355239,043337d2ca021666e95498e022dad8b6f81866a3ae6a1a...
9356,META,2024,Q4,META_2024_Q4_FS.xlsx,355239,043337d2ca021666e95498e022dad8b6f81866a3ae6a1a...
1748,BBRM,2024,Q2,BBRM_2024_Q2_FS.xlsx,345364,05c072baa91651ec0ef2aba8181585291458abc4b11807...
...,...,...,...,...,...,...
12272,RMKE,2023,Q2,RMKE_2023_Q2_FS.xlsx,1191269,46996ccf4d849ab55fe005ee68beb08b355b8a66248853...
12922,SKBM,2023,Q2,SKBM_2023_Q2_FS.xlsx,1191269,46996ccf4d849ab55fe005ee68beb08b355b8a66248853...
12941,SKLT,2023,Q2,SKLT_2023_Q2_FS.xlsx,1191269,46996ccf4d849ab55fe005ee68beb08b355b8a66248853...
13001,SMAR,2023,Q2,SMAR_2023_Q2_FS.xlsx,1191269,46996ccf4d849ab55fe005ee68beb08b355b8a66248853...


In [4]:
# CELL 51 - MARK GLOBAL CROSS-TICKER DUPLICATE SOURCE FILES

cross_ticker_duplicate_files = set(
    cross_ticker_duplicates_df[
        "source_file"
    ]
)

print(
    "Cross-ticker duplicate source files:",
    len(cross_ticker_duplicate_files)
)


final_metrics_global_duplicate_audit_df = (
    final_metrics_df
    .copy()
)

final_metrics_global_duplicate_audit_df[
    "global_cross_ticker_duplicate"
] = (
    final_metrics_global_duplicate_audit_df[
        "source_file"
    ].isin(
        cross_ticker_duplicate_files
    )
)


print(
    "Final metric rows affected:",
    final_metrics_global_duplicate_audit_df[
        "global_cross_ticker_duplicate"
    ].sum()
)

print(
    "Final metric rows unaffected:",
    (
        ~final_metrics_global_duplicate_audit_df[
            "global_cross_ticker_duplicate"
        ]
    ).sum()
)

print(
    "Affected tickers:",
    final_metrics_global_duplicate_audit_df[
        final_metrics_global_duplicate_audit_df[
            "global_cross_ticker_duplicate"
        ]
    ]["ticker"].nunique()
)

Cross-ticker duplicate source files: 968
Final metric rows affected: 5611
Final metric rows unaffected: 82789
Affected tickers: 88


In [5]:
# CELL 52 - AUDIT IMPACT OF GLOBAL DUPLICATES

affected_metrics_df = (
    final_metrics_global_duplicate_audit_df[
        final_metrics_global_duplicate_audit_df[
            "global_cross_ticker_duplicate"
        ]
    ]
    .copy()
)


print("AFFECTED ROWS BY METRIC")

print(
    affected_metrics_df[
        "metric"
    ].value_counts()
)


print("\nAFFECTED ROWS BY YEAR / QUARTER")

print(
    affected_metrics_df
    .groupby(
        [
            "year",
            "quarter"
        ]
    )
    .size()
    .sort_values(
        ascending=False
    )
)


print("\nMOST AFFECTED TICKERS")

print(
    affected_metrics_df[
        "ticker"
    ]
    .value_counts()
    .head(50)
)


display(
    affected_metrics_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "source_file",
            "source_sheet",
            "source_label"
        ]
    ]
    .head(200)
)

AFFECTED ROWS BY METRIC
metric
operating_cash_flow    968
total_assets           968
total_liabilities      968
cash                   913
revenue                913
gross_profit           881
Name: count, dtype: int64

AFFECTED ROWS BY YEAR / QUARTER
year  quarter
2023  Q2         337
      Q3         334
2024  Q3         327
2022  Q2         325
2024  Q4         325
2023  Q4         321
2024  Q2         321
2023  Q1         312
2022  Q3         310
      Q1         309
2025  Q1         307
2021  Q1         298
2024  Q1         295
2021  Q2         283
      Q3         274
2020  Q4         199
      Q1         199
      Q2         169
      Q3         169
2021  Q4         154
2022  Q4          43
dtype: int64

MOST AFFECTED TICKERS
ticker
BRMS     126
BUMI     126
RIGS     120
BFIN     114
CNTB     114
GPRA     114
INTD     114
JSMR     114
KBLI     114
MAMIP    114
MYRXP    114
SMMA     114
TMPO     114
AALI     102
SMAR     102
BPII      96
BPTR      96
FAPA      96
MLPL      96
PTR

,ticker,year,quarter,metric,selected_value,source_file,source_sheet,source_label
0,AADI,2024,Q4,cash,1518688.0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,1210000,Kas dan setara kas
1,AADI,2024,Q4,gross_profit,1465951.0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,1321000,Jumlah laba bruto
2,AADI,2024,Q4,operating_cash_flow,1198515.0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,1510000,Total net cash flows received from (used in) o...
3,AADI,2024,Q4,revenue,5319582.0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,1321000,Sales and revenue
4,AADI,2024,Q4,total_assets,5992658.0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,1210000,Jumlah aset
...,...,...,...,...,...,...,...,...
7411,AVIA,2022,Q2,revenue,10964778.0,AVIA_2022_Q2_FS.xlsx,1321000,Sales and revenue
7412,AVIA,2022,Q2,total_assets,30233993.0,AVIA_2022_Q2_FS.xlsx,1210000,Jumlah aset
7413,AVIA,2022,Q2,total_liabilities,8884801.0,AVIA_2022_Q2_FS.xlsx,1210000,Jumlah liabilitas
7414,AVIA,2022,Q3,cash,4873551.0,AVIA_2022_Q3_FS.xlsx,1210000,Kas dan setara kas


In [6]:
# CELL 53 - INSPECT INTERNAL IDENTITY FOR EACH DUPLICATE HASH GROUP

from openpyxl import load_workbook
from tqdm.auto import tqdm


duplicate_hash_representatives_df = (
    cross_ticker_duplicates_df
    .drop_duplicates(
        subset=["sha256"]
    )
    [
        [
            "sha256",
            "source_file",
            "source_path"
        ]
    ]
    .reset_index(drop=True)
)


def inspect_internal_identity(
    source_path,
    sheet_name="1000000",
    max_rows=40,
    max_columns=8
):
    result = {
        "internal_ticker": None,
        "internal_company_name": None,
        "identity_text": None,
        "identity_status": None
    }

    wb = None

    try:
        wb = load_workbook(
            source_path,
            read_only=True,
            data_only=True
        )

        if sheet_name not in wb.sheetnames:
            result["identity_status"] = (
                "METADATA_SHEET_NOT_FOUND"
            )
            return result

        ws = wb[sheet_name]

        text_parts = []

        for row in ws.iter_rows(
            min_row=1,
            max_row=max_rows,
            min_col=1,
            max_col=max_columns,
            values_only=True
        ):
            for value in row:
                if value is not None:
                    text_parts.append(
                        str(value).strip()
                    )

        full_text = " | ".join(
            text_parts
        )

        result["identity_text"] = full_text
        result["identity_status"] = "READ"

        return result

    except Exception as e:
        result["identity_status"] = (
            f"READ_ERROR:{type(e).__name__}"
        )
        return result

    finally:
        if wb is not None:
            wb.close()


identity_results = []


for _, row in tqdm(
    duplicate_hash_representatives_df.iterrows(),
    total=len(duplicate_hash_representatives_df),
    desc="Inspecting duplicate group identities"
):

    identity = inspect_internal_identity(
        Path(row["source_path"])
    )

    identity_results.append(
        {
            "sha256": row["sha256"],
            "representative_source_file":
                row["source_file"],
            **identity
        }
    )


duplicate_hash_identity_df = pd.DataFrame(
    identity_results
)


print(
    "Duplicate hash groups inspected:",
    len(duplicate_hash_identity_df)
)

print("\nIDENTITY STATUS")

print(
    duplicate_hash_identity_df[
        "identity_status"
    ].value_counts(
        dropna=False
    )
)

display(
    duplicate_hash_identity_df.head(20)
)

Inspecting duplicate group identities: 100%|██████████| 123/123 [00:53<00:00,  2.28it/s]

Duplicate hash groups inspected: 123

IDENTITY STATUS
identity_status
READ    123
Name: count, dtype: int64


,sha256,representative_source_file,internal_ticker,internal_company_name,identity_text,identity_status
0,0146ce3551b1e904488a6a90e08cac50aefeb9a35be55a...,BRMS_2024_Q2_FS.xlsx,None,None,[1000000] General information | Informasi umum...,READ
1,043337d2ca021666e95498e022dad8b6f81866a3ae6a1a...,BIPI_2024_Q4_FS.xlsx,None,None,[1000000] General information | Informasi umum...,READ
2,05c072baa91651ec0ef2aba8181585291458abc4b11807...,BBRM_2024_Q2_FS.xlsx,None,None,[1000000] General information | Informasi umum...,READ
3,07fc358e839863abe6ef0b544b9bef796c95a2c9296a4a...,BRMS_2023_Q3_FS.xlsx,None,None,[1000000] General information | Informasi umum...,READ
4,093904f29cb7ccea594f6f002b9fbc4dd0881b632a215e...,BPII_2020_Q3_FS.xlsx,None,None,[1000000] General information | Informasi umum...,READ
5,0c3ca4540a1fe763af0314e5c38a2b8f6fc135b9f2ee26...,BRMS_2021_Q2_FS.xlsx,None,None,[1000000] General information | Informasi umum...,READ
6,0c7298a8c08751ccffa02c396b576248384f0764c0a78f...,AALI_2021_Q2_FS.xlsx,None,None,[1000000] General information | Informasi umum...,READ
7,0dd4881cdd388408017f76ce1650b47d3c736f82fcad59...,BRMS_2020_Q1_FS.xlsx,None,None,[1000000] General information | Informasi umum...,READ
8,0fc998d2afc61754dab9cc67e5463d75802b85d6329c47...,BEKS_2023_Q3_FS.xlsx,None,None,[1000000] General information | Informasi umum...,READ
9,111fc145597bb8b8ff0f37b582ce62d3337ac02be64a61...,BBRM_2023_Q3_FS.xlsx,None,None,[1000000] General information | Informasi umum...,READ


In [7]:
# CELL 54 - SHOW DUPLICATE GROUPS WITH INTERNAL METADATA TEXT

duplicate_groups_with_identity_df = (
    cross_ticker_duplicates_df
    .merge(
        duplicate_hash_identity_df[
            [
                "sha256",
                "identity_text",
                "identity_status"
            ]
        ],
        on="sha256",
        how="left",
        validate="many_to_one"
    )
)


group_summary_df = (
    duplicate_groups_with_identity_df
    .groupby(
        "sha256"
    )
    .agg(
        tickers=(
            "ticker",
            lambda x: sorted(
                set(x)
            )
        ),
        file_count=(
            "source_file",
            "count"
        ),
        identity_text=(
            "identity_text",
            "first"
        )
    )
    .reset_index()
)


print(
    "Duplicate groups:",
    len(group_summary_df)
)


for _, row in group_summary_df.head(20).iterrows():

    print("\n" + "=" * 120)

    print(
        "Tickers:",
        row["tickers"]
    )

    print(
        "Files:",
        row["file_count"]
    )

    print("\nInternal metadata text:")

    identity_text = row[
        "identity_text"
    ]

    if pd.isna(identity_text):
        print("NO TEXT")
    else:
        print(
            str(identity_text)[:1500]
        )

Duplicate groups: 123

Tickers: ['BRMS', 'BUMI']
Files: 2

Internal metadata text:
[1000000] General information | Informasi umum | General information | CurrentYearInstant | Informasi umum | General information | Nama entitas | Bumi Resources Minerals Tbk | Entity name | Penjelasan perubahan nama dari akhir periode laporan sebelumnya | Explanation of change in name from the end of the preceding reporting period | Kode entitas | BRMS | Entity code | Nomor identifikasi entitas | AA567 | Entity identification number | Industri utama entitas | Umum / General | Entity main industry | Standar akutansi yang dipilih | PSAK | Selected accounting standards | Sektor | B. Basic Materials | Sector | Subsektor | B1. Basic Materials | Subsector | Industri | B14. Metals & Minerals | Industry | Subindustri | B146. Diversified Metals & Minerals | Subindustry | Informasi pemegang saham pengendali | National Corporation | Controlling shareholder information | Jenis entitas | Local Company - Indonesia Jur

In [8]:
# CELL 55 - PARSE INTERNAL TICKER FROM DUPLICATE WORKBOOK METADATA

import re


def extract_internal_ticker(identity_text):
    if pd.isna(identity_text):
        return None

    text = str(identity_text)

    patterns = [
        r"Kode entitas\s*\|\s*([A-Z0-9]+)",
        r"Entity code\s*\|\s*([A-Z0-9]+)",
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:
            return match.group(1).upper()

    return None


duplicate_hash_identity_df[
    "internal_ticker"
] = (
    duplicate_hash_identity_df[
        "identity_text"
    ].apply(
        extract_internal_ticker
    )
)


print("INTERNAL TICKER PARSE STATUS")

print(
    duplicate_hash_identity_df[
        "internal_ticker"
    ].notna().value_counts()
)


display(
    duplicate_hash_identity_df[
        [
            "representative_source_file",
            "internal_ticker",
            "identity_status"
        ]
    ]
    .head(30)
)

INTERNAL TICKER PARSE STATUS
internal_ticker
True    123
Name: count, dtype: int64


,representative_source_file,internal_ticker,identity_status
0,BRMS_2024_Q2_FS.xlsx,BRMS,READ
1,BIPI_2024_Q4_FS.xlsx,BIPI,READ
2,BBRM_2024_Q2_FS.xlsx,BBRM,READ
3,BRMS_2023_Q3_FS.xlsx,BRMS,READ
4,BPII_2020_Q3_FS.xlsx,BPII,READ
5,BRMS_2021_Q2_FS.xlsx,BRMS,READ
6,AALI_2021_Q2_FS.xlsx,AALI,READ
7,BRMS_2020_Q1_FS.xlsx,BRMS,READ
8,BEKS_2023_Q3_FS.xlsx,BEKS,READ
9,BBRM_2023_Q3_FS.xlsx,BBRM,READ


In [9]:
# CELL 56 - RESOLVE DUPLICATE GROUP OWNERSHIP USING INTERNAL TICKER

duplicate_resolution_df = (
    cross_ticker_duplicates_df
    .merge(
        duplicate_hash_identity_df[
            [
                "sha256",
                "internal_ticker"
            ]
        ],
        on="sha256",
        how="left",
        validate="many_to_one"
    )
)


duplicate_resolution_df[
    "ticker_matches_internal"
] = (
    duplicate_resolution_df[
        "ticker"
    ].astype(str).str.upper()
    ==
    duplicate_resolution_df[
        "internal_ticker"
    ].astype(str).str.upper()
)


group_resolution_summary_df = (
    duplicate_resolution_df
    .groupby(
        "sha256"
    )
    .agg(
        internal_ticker=(
            "internal_ticker",
            "first"
        ),
        ticker_count=(
            "ticker",
            "nunique"
        ),
        matching_files=(
            "ticker_matches_internal",
            "sum"
        )
    )
    .reset_index()
)


def classify_duplicate_group(row):
    if pd.isna(row["internal_ticker"]):
        return "NO_INTERNAL_TICKER"

    if row["matching_files"] == 1:
        return "RESOLVED_SINGLE_OWNER"

    if row["matching_files"] == 0:
        return "NO_MATCHING_OWNER"

    return "MULTIPLE_MATCHING_FILES"


group_resolution_summary_df[
    "resolution_status"
] = (
    group_resolution_summary_df.apply(
        classify_duplicate_group,
        axis=1
    )
)


print("DUPLICATE GROUP RESOLUTION STATUS")

print(
    group_resolution_summary_df[
        "resolution_status"
    ].value_counts(
        dropna=False
    )
)


display(
    group_resolution_summary_df.head(30)
)

DUPLICATE GROUP RESOLUTION STATUS
resolution_status
RESOLVED_SINGLE_OWNER      119
NO_MATCHING_OWNER            2
MULTIPLE_MATCHING_FILES      2
Name: count, dtype: int64


,sha256,internal_ticker,ticker_count,matching_files,resolution_status
0,0146ce3551b1e904488a6a90e08cac50aefeb9a35be55a...,BRMS,2,1,RESOLVED_SINGLE_OWNER
1,043337d2ca021666e95498e022dad8b6f81866a3ae6a1a...,BIPI,2,1,RESOLVED_SINGLE_OWNER
2,05c072baa91651ec0ef2aba8181585291458abc4b11807...,BBRM,2,1,RESOLVED_SINGLE_OWNER
3,07fc358e839863abe6ef0b544b9bef796c95a2c9296a4a...,BRMS,2,1,RESOLVED_SINGLE_OWNER
4,093904f29cb7ccea594f6f002b9fbc4dd0881b632a215e...,BPII,2,1,RESOLVED_SINGLE_OWNER
5,0c3ca4540a1fe763af0314e5c38a2b8f6fc135b9f2ee26...,BRMS,2,1,RESOLVED_SINGLE_OWNER
6,0c7298a8c08751ccffa02c396b576248384f0764c0a78f...,AALI,40,1,RESOLVED_SINGLE_OWNER
7,0dd4881cdd388408017f76ce1650b47d3c736f82fcad59...,BRMS,2,1,RESOLVED_SINGLE_OWNER
8,0fc998d2afc61754dab9cc67e5463d75802b85d6329c47...,BEKS,2,1,RESOLVED_SINGLE_OWNER
9,111fc145597bb8b8ff0f37b582ce62d3337ac02be64a61...,BBRM,2,1,RESOLVED_SINGLE_OWNER


In [10]:
# CELL 57 - MARK TRUSTED AND BAD DUPLICATE SOURCE FILES

duplicate_resolution_df = (
    duplicate_resolution_df
    .merge(
        group_resolution_summary_df[
            [
                "sha256",
                "resolution_status"
            ]
        ],
        on="sha256",
        how="left",
        validate="many_to_one"
    )
)


duplicate_resolution_df[
    "duplicate_source_status"
] = np.select(
    [
        (
            duplicate_resolution_df[
                "resolution_status"
            ] == "RESOLVED_SINGLE_OWNER"
        )
        &
        (
            duplicate_resolution_df[
                "ticker_matches_internal"
            ]
        ),

        (
            duplicate_resolution_df[
                "resolution_status"
            ] == "RESOLVED_SINGLE_OWNER"
        )
        &
        (
            ~duplicate_resolution_df[
                "ticker_matches_internal"
            ]
        ),
    ],
    [
        "TRUSTED_DUPLICATE_OWNER",
        "BAD_DUPLICATE_COPY",
    ],
    default="UNRESOLVED_DUPLICATE"
)


print("DUPLICATE SOURCE STATUS")

print(
    duplicate_resolution_df[
        "duplicate_source_status"
    ].value_counts(
        dropna=False
    )
)


display(
    duplicate_resolution_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "internal_ticker",
            "ticker_matches_internal",
            "resolution_status",
            "duplicate_source_status"
        ]
    ]
    .head(100)
)

DUPLICATE SOURCE STATUS
duplicate_source_status
BAD_DUPLICATE_COPY         818
TRUSTED_DUPLICATE_OWNER    119
UNRESOLVED_DUPLICATE        31
Name: count, dtype: int64


,ticker,year,quarter,source_file,internal_ticker,ticker_matches_internal,resolution_status,duplicate_source_status
0,BRMS,2024,Q2,BRMS_2024_Q2_FS.xlsx,BRMS,True,RESOLVED_SINGLE_OWNER,TRUSTED_DUPLICATE_OWNER
1,BUMI,2024,Q2,BUMI_2024_Q2_FS.xlsx,BRMS,False,RESOLVED_SINGLE_OWNER,BAD_DUPLICATE_COPY
2,BIPI,2024,Q4,BIPI_2024_Q4_FS.xlsx,BIPI,True,RESOLVED_SINGLE_OWNER,TRUSTED_DUPLICATE_OWNER
3,META,2024,Q4,META_2024_Q4_FS.xlsx,BIPI,False,RESOLVED_SINGLE_OWNER,BAD_DUPLICATE_COPY
4,BBRM,2024,Q2,BBRM_2024_Q2_FS.xlsx,BBRM,True,RESOLVED_SINGLE_OWNER,TRUSTED_DUPLICATE_OWNER
...,...,...,...,...,...,...,...,...
95,BJBR,2023,Q2,BJBR_2023_Q2_FS.xlsx,BEKS,False,RESOLVED_SINGLE_OWNER,BAD_DUPLICATE_COPY
96,BJTM,2023,Q2,BJTM_2023_Q2_FS.xlsx,BEKS,False,RESOLVED_SINGLE_OWNER,BAD_DUPLICATE_COPY
97,BRMS,2020,Q3,BRMS_2020_Q3_FS.xlsx,BRMS,True,RESOLVED_SINGLE_OWNER,TRUSTED_DUPLICATE_OWNER
98,BUMI,2020,Q3,BUMI_2020_Q3_FS.xlsx,BRMS,False,RESOLVED_SINGLE_OWNER,BAD_DUPLICATE_COPY


In [11]:
# CELL 58 - INSPECT UNRESOLVED DUPLICATE GROUPS

unresolved_duplicate_df = (
    duplicate_resolution_df[
        duplicate_resolution_df[
            "duplicate_source_status"
        ] == "UNRESOLVED_DUPLICATE"
    ]
    .copy()
)


print(
    "Unresolved duplicate files:",
    len(unresolved_duplicate_df)
)

print(
    "Unresolved duplicate groups:",
    unresolved_duplicate_df[
        "sha256"
    ].nunique()
)


unresolved_group_summary_df = (
    unresolved_duplicate_df
    .groupby(
        "sha256"
    )
    .agg(
        internal_ticker=(
            "internal_ticker",
            "first"
        ),
        tickers=(
            "ticker",
            lambda x: sorted(
                set(x)
            )
        ),
        source_files=(
            "source_file",
            lambda x: list(x)
        ),
        resolution_status=(
            "resolution_status",
            "first"
        )
    )
    .reset_index()
)


display(
    unresolved_group_summary_df
)

Unresolved duplicate files: 31
Unresolved duplicate groups: 4


,sha256,internal_ticker,tickers,source_files,resolution_status
0,15e53f1c37a182ac2ec7e7ba6b920f69f9f54b4de12cab...,AALI,"[AIMS, BFIN, CNTB, GPRA, INTD, JSMR, KBLI, MAM...","[AIMS_2020_Q3_FS.xlsx, BFIN_2020_Q3_FS.xlsx, C...",NO_MATCHING_OWNER
1,1c0c7889b52804c509f391291b9d84cc1e3b1bd95df406...,FREN,"[FREN, SMAR]","[FREN_2020_Q3_FS.xlsx, FREN_2020_Q4_FS.xlsx, S...",MULTIPLE_MATCHING_FILES
2,d2c771d471563bf4c3af290414de30ddef35ab542968b3...,AALI,"[AALI, AIMS, BFIN, CNTB, GPRA, INTD, JSMR, KBL...","[AALI_2020_Q2_FS.xlsx, AALI_2020_Q3_FS.xlsx, A...",MULTIPLE_MATCHING_FILES
3,dd157d72cc2594932898f28e731f4235358fd798d40884...,BEKS,"[BJBR, BJTM]","[BJBR_2021_Q4_FS.xlsx, BJTM_2021_Q4_FS.xlsx]",NO_MATCHING_OWNER


In [ ]:
# CELL 59 - SHOW FULL DETAILS FOR UNRESOLVED DUPLICATE FILES

display(
    unresolved_duplicate_df[
        [
            "sha256",
            "ticker",
            "year",
            "quarter",
            "source_file",
            "internal_ticker",
            "ticker_matches_internal",
            "resolution_status",
            "duplicate_source_status"
        ]
    ]
    .sort_values(
        [
            "sha256",
            "ticker",
            "year",
            "quarter"
        ]
    )
)

In [12]:
# CELL 60 - PARSE INTERNAL REPORT PERIOD FROM METADATA

import re


def extract_internal_period(identity_text):

    result = {
        "internal_year": None,
        "internal_quarter": None,
        "internal_period_end": None
    }

    if pd.isna(identity_text):
        return result

    text = str(identity_text)


    # Prefer the explicit current-period end date
    pattern = (
        r"Tanggal akhir periode berjalan"
        r"\s*\|\s*"
        r"(\d{4}-\d{2}-\d{2})"
    )

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if not match:
        return result


    period_end = match.group(1)

    try:
        period_date = pd.to_datetime(
            period_end
        )

        year = int(
            period_date.year
        )

        month = int(
            period_date.month
        )


        quarter_map = {
            3: "Q1",
            6: "Q2",
            9: "Q3",
            12: "Q4"
        }


        quarter = quarter_map.get(
            month
        )


        result = {
            "internal_year": year,
            "internal_quarter": quarter,
            "internal_period_end": period_end
        }

        return result

    except Exception:
        return result


period_results = (
    duplicate_hash_identity_df[
        "identity_text"
    ]
    .apply(
        extract_internal_period
    )
)


duplicate_hash_identity_df[
    "internal_year"
] = period_results.apply(
    lambda x: x["internal_year"]
)

duplicate_hash_identity_df[
    "internal_quarter"
] = period_results.apply(
    lambda x: x["internal_quarter"]
)

duplicate_hash_identity_df[
    "internal_period_end"
] = period_results.apply(
    lambda x: x["internal_period_end"]
)


print("INTERNAL PERIOD PARSE")

print(
    duplicate_hash_identity_df[
        "internal_period_end"
    ].notna().value_counts()
)


display(
    duplicate_hash_identity_df[
        [
            "representative_source_file",
            "internal_ticker",
            "internal_year",
            "internal_quarter",
            "internal_period_end"
        ]
    ]
    .head(30)
)

INTERNAL PERIOD PARSE
internal_period_end
True    123
Name: count, dtype: int64


,representative_source_file,internal_ticker,internal_year,internal_quarter,internal_period_end
0,BRMS_2024_Q2_FS.xlsx,BRMS,2024,Q2,2024-06-30
1,BIPI_2024_Q4_FS.xlsx,BIPI,2024,Q4,2024-12-31
2,BBRM_2024_Q2_FS.xlsx,BBRM,2024,Q2,2024-06-30
3,BRMS_2023_Q3_FS.xlsx,BRMS,2023,Q3,2023-09-30
4,BPII_2020_Q3_FS.xlsx,BPII,2020,Q3,2020-09-30
5,BRMS_2021_Q2_FS.xlsx,BRMS,2021,Q2,2021-06-30
6,AALI_2021_Q2_FS.xlsx,AALI,2021,Q2,2021-06-30
7,BRMS_2020_Q1_FS.xlsx,BRMS,2020,Q1,2020-03-31
8,BEKS_2023_Q3_FS.xlsx,BEKS,2023,Q3,2023-09-30
9,BBRM_2023_Q3_FS.xlsx,BBRM,2023,Q3,2023-09-30


In [14]:
# CELL 61 - FINAL DUPLICATE OWNERSHIP RESOLUTION
# Match internal ticker + internal year + internal quarter

duplicate_resolution_final_df = (
    cross_ticker_duplicates_df
    .merge(
        duplicate_hash_identity_df[
            [
                "sha256",
                "internal_ticker",
                "internal_year",
                "internal_quarter",
                "internal_period_end"
            ]
        ],
        on="sha256",
        how="left",
        validate="many_to_one"
    )
)


duplicate_resolution_final_df[
    "ticker_matches_internal"
] = (
    duplicate_resolution_final_df[
        "ticker"
    ].astype(str).str.upper()
    ==
    duplicate_resolution_final_df[
        "internal_ticker"
    ].astype(str).str.upper()
)


duplicate_resolution_final_df[
    "year_matches_internal"
] = (
    pd.to_numeric(
        duplicate_resolution_final_df[
            "year"
        ],
        errors="coerce"
    )
    ==
    pd.to_numeric(
        duplicate_resolution_final_df[
            "internal_year"
        ],
        errors="coerce"
    )
)


duplicate_resolution_final_df[
    "quarter_matches_internal"
] = (
    duplicate_resolution_final_df[
        "quarter"
    ].astype(str).str.upper()
    ==
    duplicate_resolution_final_df[
        "internal_quarter"
    ].astype(str).str.upper()
)


duplicate_resolution_final_df[
    "full_identity_match"
] = (
    duplicate_resolution_final_df[
        "ticker_matches_internal"
    ]
    &
    duplicate_resolution_final_df[
        "year_matches_internal"
    ]
    &
    duplicate_resolution_final_df[
        "quarter_matches_internal"
    ]
)


final_group_summary_df = (
    duplicate_resolution_final_df
    .groupby(
        "sha256"
    )
    .agg(
        internal_ticker=(
            "internal_ticker",
            "first"
        ),
        internal_year=(
            "internal_year",
            "first"
        ),
        internal_quarter=(
            "internal_quarter",
            "first"
        ),
        exact_owner_matches=(
            "full_identity_match",
            "sum"
        )
    )
    .reset_index()
)


def classify_final_duplicate_group(row):

    if row["exact_owner_matches"] == 1:
        return "RESOLVED_EXACT_OWNER"

    if row["exact_owner_matches"] == 0:
        return "NO_EXACT_OWNER"

    return "MULTIPLE_EXACT_OWNERS"


final_group_summary_df[
    "final_resolution_status"
] = (
    final_group_summary_df.apply(
        classify_final_duplicate_group,
        axis=1
    )
)


print("FINAL DUPLICATE GROUP STATUS")

print(
    final_group_summary_df[
        "final_resolution_status"
    ].value_counts(
        dropna=False
    )
)


display(
    final_group_summary_df[
        final_group_summary_df[
            "final_resolution_status"
        ] != "RESOLVED_EXACT_OWNER"
    ]
)

FINAL DUPLICATE GROUP STATUS
final_resolution_status
RESOLVED_EXACT_OWNER    121
NO_EXACT_OWNER            2
Name: count, dtype: int64


,sha256,internal_ticker,internal_year,internal_quarter,exact_owner_matches,final_resolution_status
12,15e53f1c37a182ac2ec7e7ba6b920f69f9f54b4de12cab...,AALI,2020,Q3,0,NO_EXACT_OWNER
97,dd157d72cc2594932898f28e731f4235358fd798d40884...,BEKS,2021,Q4,0,NO_EXACT_OWNER


In [15]:
# CELL 62 - ASSIGN FINAL DUPLICATE SOURCE STATUS

duplicate_resolution_final_df = (
    duplicate_resolution_final_df
    .merge(
        final_group_summary_df[
            [
                "sha256",
                "final_resolution_status"
            ]
        ],
        on="sha256",
        how="left",
        validate="many_to_one"
    )
)


duplicate_resolution_final_df[
    "final_duplicate_source_status"
] = np.select(
    [
        (
            duplicate_resolution_final_df[
                "final_resolution_status"
            ] == "RESOLVED_EXACT_OWNER"
        )
        &
        (
            duplicate_resolution_final_df[
                "full_identity_match"
            ]
        ),

        (
            duplicate_resolution_final_df[
                "final_resolution_status"
            ] == "RESOLVED_EXACT_OWNER"
        )
        &
        (
            ~duplicate_resolution_final_df[
                "full_identity_match"
            ]
        ),

        (
            duplicate_resolution_final_df[
                "final_resolution_status"
            ] == "NO_EXACT_OWNER"
        ),
    ],

    [
        "TRUSTED_DUPLICATE_OWNER",
        "BAD_DUPLICATE_COPY",
        "BAD_DUPLICATE_COPY",
    ],

    default="UNRESOLVED_DUPLICATE"
)


print("FINAL DUPLICATE SOURCE STATUS")

print(
    duplicate_resolution_final_df[
        "final_duplicate_source_status"
    ].value_counts(
        dropna=False
    )
)


print("\nSTILL UNRESOLVED")

display(
    duplicate_resolution_final_df[
        duplicate_resolution_final_df[
            "final_duplicate_source_status"
        ] == "UNRESOLVED_DUPLICATE"
    ][
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "internal_ticker",
            "internal_year",
            "internal_quarter",
            "final_resolution_status"
        ]
    ]
)

FINAL DUPLICATE SOURCE STATUS
final_duplicate_source_status
BAD_DUPLICATE_COPY         847
TRUSTED_DUPLICATE_OWNER    121
Name: count, dtype: int64

STILL UNRESOLVED


,ticker,year,quarter,source_file,internal_ticker,internal_year,internal_quarter,final_resolution_status


In [16]:
# CELL 63 - REMOVE BAD GLOBAL DUPLICATE SOURCE FILES

bad_duplicate_source_files = set(
    duplicate_resolution_final_df.loc[
        duplicate_resolution_final_df[
            "final_duplicate_source_status"
        ] == "BAD_DUPLICATE_COPY",
        "source_file"
    ]
)

trusted_duplicate_source_files = set(
    duplicate_resolution_final_df.loc[
        duplicate_resolution_final_df[
            "final_duplicate_source_status"
        ] == "TRUSTED_DUPLICATE_OWNER",
        "source_file"
    ]
)


print(
    "Bad duplicate source files:",
    len(bad_duplicate_source_files)
)

print(
    "Trusted duplicate owner files:",
    len(trusted_duplicate_source_files)
)


clean_final_metrics_df = (
    final_metrics_df[
        ~final_metrics_df[
            "source_file"
        ].isin(
            bad_duplicate_source_files
        )
    ]
    .copy()
)


print(
    "\nOriginal final metric rows:",
    len(final_metrics_df)
)

print(
    "Clean final metric rows:",
    len(clean_final_metrics_df)
)

print(
    "Removed metric rows:",
    len(final_metrics_df)
    - len(clean_final_metrics_df)
)

print(
    "Clean unique tickers:",
    clean_final_metrics_df[
        "ticker"
    ].nunique()
)

print(
    "Duplicate ticker/year/quarter/metric:",
    clean_final_metrics_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    ].duplicated().sum()
)

Bad duplicate source files: 847
Trusted duplicate owner files: 121

Original final metric rows: 88400
Clean final metric rows: 83442
Removed metric rows: 4958
Clean unique tickers: 924
Duplicate ticker/year/quarter/metric: 0


In [17]:
# CELL 64 - AUDIT REMOVED METRIC ROWS

removed_bad_duplicate_metrics_df = (
    final_metrics_df[
        final_metrics_df[
            "source_file"
        ].isin(
            bad_duplicate_source_files
        )
    ]
    .copy()
)


print("REMOVED ROWS BY METRIC")

print(
    removed_bad_duplicate_metrics_df[
        "metric"
    ].value_counts()
)


print("\nREMOVED ROWS BY YEAR / QUARTER")

print(
    removed_bad_duplicate_metrics_df
    .groupby(
        [
            "year",
            "quarter"
        ]
    )
    .size()
    .sort_values(
        ascending=False
    )
)


print("\nMOST AFFECTED TICKERS")

print(
    removed_bad_duplicate_metrics_df[
        "ticker"
    ]
    .value_counts()
    .head(50)
)

REMOVED ROWS BY METRIC
metric
operating_cash_flow    847
total_assets           847
total_liabilities      847
cash                   811
revenue                811
gross_profit           795
Name: count, dtype: int64

REMOVED ROWS BY YEAR / QUARTER
year  quarter
2023  Q2         311
      Q3         308
2024  Q4         305
2023  Q4         300
2024  Q3         300
      Q2         294
2022  Q2         293
2023  Q1         285
2022  Q3         284
      Q1         282
2025  Q1         281
2021  Q1         275
2024  Q1         275
2021  Q2         257
      Q3         248
2020  Q1         149
      Q3         125
      Q4         125
      Q2         119
2021  Q4         119
2022  Q4          23
dtype: int64

MOST AFFECTED TICKERS
ticker
BUMI     126
RIGS     120
BFIN     114
CNTB     114
GPRA     114
INTD     114
JSMR     114
KBLI     114
MAMIP    114
MYRXP    114
SMMA     114
TMPO     114
SMAR     102
BPTR      96
FAPA      96
MLPL      96
PTRO      96
RMKE      96
SKBM      96
UANG 

In [18]:
# CELL 65 - SAVE CLEAN FINAL METRICS AFTER GLOBAL DUPLICATE REMOVAL

CLEAN_FINAL_METRICS_FILE = Path(
    "data/idx_financial_current_metrics_clean.csv"
)


clean_final_metrics_df.to_csv(
    CLEAN_FINAL_METRICS_FILE,
    index=False
)


print(
    "Saved clean final metrics to:",
    CLEAN_FINAL_METRICS_FILE
)

print(
    "Total rows:",
    len(clean_final_metrics_df)
)

print(
    "Unique tickers:",
    clean_final_metrics_df[
        "ticker"
    ].nunique()
)

Saved clean final metrics to: data\idx_financial_current_metrics_clean.csv
Total rows: 83442
Unique tickers: 924


In [19]:
# CELL 66 - LOAD CLEAN METRICS AND MERGE FINAL CURRENCY / SCALE

CLEAN_FINAL_METRICS_FILE = Path(
    "data/idx_financial_current_metrics_clean.csv"
)

FINAL_CURRENCY_SCALE_FILE = Path(
    "data/idx_financial_currency_scale_mapping_final.csv"
)


clean_metrics_df = pd.read_csv(
    CLEAN_FINAL_METRICS_FILE,
    low_memory=False
)

currency_scale_mapping_final_df = pd.read_csv(
    FINAL_CURRENCY_SCALE_FILE,
    low_memory=False
)


currency_scale_for_clean_merge_df = (
    currency_scale_mapping_final_df[
        [
            "source_file",
            "currency",
            "unit_scale",
            "multiplier",
            "mapping_status"
        ]
    ]
    .copy()
    .rename(
        columns={
            "mapping_status": "currency_scale_status"
        }
    )
)


clean_metrics_with_scale_df = (
    clean_metrics_df
    .merge(
        currency_scale_for_clean_merge_df,
        on="source_file",
        how="left",
        validate="many_to_one"
    )
)


print(
    "Clean metric rows:",
    len(clean_metrics_df)
)

print(
    "Rows after merge:",
    len(clean_metrics_with_scale_df)
)

print(
    "Missing mapping:",
    clean_metrics_with_scale_df[
        "currency_scale_status"
    ].isna().sum()
)

print("\nCURRENCY / SCALE STATUS")

print(
    clean_metrics_with_scale_df[
        "currency_scale_status"
    ].value_counts(
        dropna=False
    )
)

Clean metric rows: 83442
Rows after merge: 83442
Missing mapping: 0

CURRENCY / SCALE STATUS
currency_scale_status
FOUND                           83438
CURRENCY_AND_SCALE_NOT_FOUND        4
Name: count, dtype: int64


In [20]:
# CELL 67 - REBUILD SCALE MODE DATA USING CLEAN METRICS

clean_scale_mode_df = (
    clean_metrics_with_scale_df[
        (
            clean_metrics_with_scale_df[
                "currency_scale_status"
            ] == "FOUND"
        )
        &
        (
            clean_metrics_with_scale_df[
                "selected_value"
            ].notna()
        )
    ]
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "source_file"
        ]
    ]
    .copy()
)


quarter_order = {
    "Q1": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4": 4
}


clean_scale_mode_df[
    "quarter_order"
] = (
    clean_scale_mode_df[
        "quarter"
    ].map(
        quarter_order
    )
)


clean_scale_mode_df = (
    clean_scale_mode_df
    .sort_values(
        [
            "ticker",
            "metric",
            "year",
            "quarter_order"
        ]
    )
    .reset_index(drop=True)
)


clean_grouped = (
    clean_scale_mode_df
    .groupby(
        [
            "ticker",
            "metric"
        ]
    )
)


clean_scale_mode_df[
    "previous_value"
] = (
    clean_grouped[
        "selected_value"
    ].shift(1)
)


clean_scale_mode_df[
    "previous_multiplier"
] = (
    clean_grouped[
        "multiplier"
    ].shift(1)
)


clean_scale_mode_df[
    "previous_source_file"
] = (
    clean_grouped[
        "source_file"
    ].shift(1)
)


print(
    "Rows available for clean scale analysis:",
    len(clean_scale_mode_df)
)

display(
    clean_scale_mode_df.head()
)

Rows available for clean scale analysis: 83438


,ticker,year,quarter,metric,selected_value,currency,unit_scale,multiplier,source_file,quarter_order,previous_value,previous_multiplier,previous_source_file
0,AADI,2024,Q4,cash,1518688.0,USD,THOUSAND,1000.0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,4,NaN,NaN,NaN
1,AADI,2025,Q1,cash,1358333.0,USD,THOUSAND,1000.0,AADI_2025_Q1_FS.xlsx,1,1518688.0,1000.0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...
2,AADI,2024,Q4,gross_profit,1465951.0,USD,THOUSAND,1000.0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,4,NaN,NaN,NaN
3,AADI,2025,Q1,gross_profit,347408.0,USD,THOUSAND,1000.0,AADI_2025_Q1_FS.xlsx,1,1465951.0,1000.0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...
4,AADI,2024,Q4,operating_cash_flow,1198515.0,USD,THOUSAND,1000.0,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,4,NaN,NaN,NaN


In [21]:
# CELL 68 - CLEAN PAIRWISE SCALE MODE CLASSIFICATION

def safe_log_distance(
    previous_value,
    current_value
):
    if (
        pd.isna(previous_value)
        or pd.isna(current_value)
    ):
        return np.nan

    previous_value = abs(
        float(previous_value)
    )

    current_value = abs(
        float(current_value)
    )

    if (
        previous_value == 0
        or current_value == 0
    ):
        return np.nan

    return abs(
        np.log10(
            current_value
            /
            previous_value
        )
    )


clean_pairwise_results = []


for _, row in clean_scale_mode_df.iterrows():

    previous_value = row[
        "previous_value"
    ]

    current_value = row[
        "selected_value"
    ]

    previous_multiplier = row[
        "previous_multiplier"
    ]

    current_multiplier = row[
        "multiplier"
    ]


    if (
        pd.isna(previous_value)
        or pd.isna(previous_multiplier)
        or pd.isna(current_multiplier)
    ):
        continue


    candidates = {

        "PREV_PRESENT_CURRENT_PRESENT": (
            previous_value
            * previous_multiplier,
            current_value
            * current_multiplier
        ),

        "PREV_PRESENT_CURRENT_FULL": (
            previous_value
            * previous_multiplier,
            current_value
        ),

        "PREV_FULL_CURRENT_PRESENT": (
            previous_value,
            current_value
            * current_multiplier
        ),

        "PREV_FULL_CURRENT_FULL": (
            previous_value,
            current_value
        )
    }


    scores = {
        mode: safe_log_distance(
            values[0],
            values[1]
        )
        for mode, values
        in candidates.items()
    }


    valid_scores = {
        mode: score
        for mode, score
        in scores.items()
        if not pd.isna(score)
    }


    if not valid_scores:
        continue


    ranked_modes = sorted(
        valid_scores.items(),
        key=lambda item: item[1]
    )


    best_mode = ranked_modes[0][0]


    if best_mode in [
        "PREV_PRESENT_CURRENT_FULL",
        "PREV_FULL_CURRENT_FULL"
    ]:
        current_mode_vote = (
            "FULL_NOMINAL"
        )

    else:
        current_mode_vote = (
            "PRESENTATION_SCALED"
        )


    if best_mode in [
        "PREV_PRESENT_CURRENT_PRESENT",
        "PREV_PRESENT_CURRENT_FULL"
    ]:
        previous_mode_vote = (
            "PRESENTATION_SCALED"
        )

    else:
        previous_mode_vote = (
            "FULL_NOMINAL"
        )


    clean_pairwise_results.append(
        {
            "ticker": row[
                "ticker"
            ],

            "metric": row[
                "metric"
            ],

            "previous_source_file":
                row[
                    "previous_source_file"
                ],

            "current_source_file":
                row[
                    "source_file"
                ],

            "best_mode":
                best_mode,

            "previous_mode_vote":
                previous_mode_vote,

            "current_mode_vote":
                current_mode_vote
        }
    )


clean_pairwise_scale_mode_df = (
    pd.DataFrame(
        clean_pairwise_results
    )
)


print(
    "Pairwise comparisons:",
    len(
        clean_pairwise_scale_mode_df
    )
)

print("\nBEST MODE COUNTS")

print(
    clean_pairwise_scale_mode_df[
        "best_mode"
    ].value_counts()
)

print("\nCURRENT MODE VOTES")

print(
    clean_pairwise_scale_mode_df[
        "current_mode_vote"
    ].value_counts()
)

Pairwise comparisons: 78092

BEST MODE COUNTS
best_mode
PREV_PRESENT_CURRENT_PRESENT    77660
PREV_PRESENT_CURRENT_FULL         211
PREV_FULL_CURRENT_PRESENT         210
PREV_FULL_CURRENT_FULL             11
Name: count, dtype: int64

CURRENT MODE VOTES
current_mode_vote
PRESENTATION_SCALED    77870
FULL_NOMINAL             222
Name: count, dtype: int64


In [22]:
# CELL 69 - AGGREGATE CLEAN SCALE MODE VOTES PER SOURCE FILE

clean_vote_rows = []


for _, row in clean_pairwise_scale_mode_df.iterrows():

    clean_vote_rows.append(
        {
            "source_file": row["previous_source_file"],
            "mode_vote": row["previous_mode_vote"]
        }
    )

    clean_vote_rows.append(
        {
            "source_file": row["current_source_file"],
            "mode_vote": row["current_mode_vote"]
        }
    )


clean_source_file_votes_df = pd.DataFrame(
    clean_vote_rows
)


clean_source_file_vote_summary_df = (
    clean_source_file_votes_df
    .groupby(
        [
            "source_file",
            "mode_vote"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)


if "FULL_NOMINAL" not in clean_source_file_vote_summary_df.columns:
    clean_source_file_vote_summary_df[
        "FULL_NOMINAL"
    ] = 0

if "PRESENTATION_SCALED" not in clean_source_file_vote_summary_df.columns:
    clean_source_file_vote_summary_df[
        "PRESENTATION_SCALED"
    ] = 0


clean_source_file_vote_summary_df[
    "total_votes"
] = (
    clean_source_file_vote_summary_df[
        "FULL_NOMINAL"
    ]
    +
    clean_source_file_vote_summary_df[
        "PRESENTATION_SCALED"
    ]
)


clean_source_file_vote_summary_df[
    "full_nominal_share"
] = (
    clean_source_file_vote_summary_df[
        "FULL_NOMINAL"
    ]
    /
    clean_source_file_vote_summary_df[
        "total_votes"
    ]
)


clean_source_file_vote_summary_df[
    "presentation_scaled_share"
] = (
    clean_source_file_vote_summary_df[
        "PRESENTATION_SCALED"
    ]
    /
    clean_source_file_vote_summary_df[
        "total_votes"
    ]
)


def classify_clean_file_mode(row):

    if row["total_votes"] == 0:
        return "NO_EVIDENCE"

    if (
        row["FULL_NOMINAL"] >= 2
        and row["full_nominal_share"] >= 0.80
    ):
        return "FULL_NOMINAL"

    if (
        row["PRESENTATION_SCALED"] >= 2
        and row["presentation_scaled_share"] >= 0.80
    ):
        return "PRESENTATION_SCALED"

    return "AMBIGUOUS"


clean_source_file_vote_summary_df[
    "file_scale_mode"
] = (
    clean_source_file_vote_summary_df.apply(
        classify_clean_file_mode,
        axis=1
    )
)


print("CLEAN FILE SCALE MODE")

print(
    clean_source_file_vote_summary_df[
        "file_scale_mode"
    ].value_counts(
        dropna=False
    )
)


print("\nKNOWN CHECK FILES")

known_check_files = [
    "ASII_2020_Q1_FS.xlsx",
    "AALI_2020_Q1_FS.xlsx",
    "ASII_2023_Q4_FS.xlsx",
    "BMRI_2023_Q4_FS.xlsx",
    "AALI_2023_Q4_FS.xlsx",
    "AALI_2024_Q2_FS.xlsx"
]


display(
    clean_source_file_vote_summary_df[
        clean_source_file_vote_summary_df[
            "source_file"
        ].isin(
            known_check_files
        )
    ]
    [
        [
            "source_file",
            "FULL_NOMINAL",
            "PRESENTATION_SCALED",
            "total_votes",
            "full_nominal_share",
            "presentation_scaled_share",
            "file_scale_mode"
        ]
    ]
)

CLEAN FILE SCALE MODE
file_scale_mode
PRESENTATION_SCALED    14703
FULL_NOMINAL              37
AMBIGUOUS                  7
Name: count, dtype: int64

KNOWN CHECK FILES


mode_vote,source_file,FULL_NOMINAL,PRESENTATION_SCALED,total_votes,full_nominal_share,presentation_scaled_share,file_scale_mode
2,AALI_2020_Q1_FS.xlsx,0,6,6,0.0,1.0,PRESENTATION_SCALED
16,AALI_2023_Q4_FS.xlsx,12,0,12,1.0,0.0,FULL_NOMINAL
17,AALI_2024_Q2_FS.xlsx,0,12,12,0.0,1.0,PRESENTATION_SCALED
1116,ASII_2020_Q1_FS.xlsx,0,6,6,0.0,1.0,PRESENTATION_SCALED
1131,ASII_2023_Q4_FS.xlsx,10,0,10,1.0,0.0,FULL_NOMINAL
2388,BMRI_2023_Q4_FS.xlsx,6,0,6,1.0,0.0,FULL_NOMINAL


In [23]:
# CELL 70 - INSPECT FULL_NOMINAL AND AMBIGUOUS FILES

special_scale_files_df = (
    clean_source_file_vote_summary_df[
        clean_source_file_vote_summary_df[
            "file_scale_mode"
        ].isin(
            [
                "FULL_NOMINAL",
                "AMBIGUOUS"
            ]
        )
    ]
    .copy()
)


print(
    "Special files total:",
    len(special_scale_files_df)
)

print("\nBY MODE")

print(
    special_scale_files_df[
        "file_scale_mode"
    ].value_counts()
)


special_scale_files_with_meta_df = (
    special_scale_files_df
    .merge(
        clean_metrics_with_scale_df[
            [
                "source_file",
                "ticker",
                "year",
                "quarter",
                "currency",
                "unit_scale",
                "multiplier"
            ]
        ]
        .drop_duplicates(
            subset=["source_file"]
        ),
        on="source_file",
        how="left",
        validate="one_to_one"
    )
)


display(
    special_scale_files_with_meta_df[
        [
            "source_file",
            "ticker",
            "year",
            "quarter",
            "currency",
            "unit_scale",
            "multiplier",
            "FULL_NOMINAL",
            "PRESENTATION_SCALED",
            "total_votes",
            "full_nominal_share",
            "presentation_scaled_share",
            "file_scale_mode"
        ]
    ]
    .sort_values(
        [
            "file_scale_mode",
            "ticker",
            "year",
            "quarter"
        ]
    )
)

Special files total: 44

BY MODE
file_scale_mode
FULL_NOMINAL    37
AMBIGUOUS        7
Name: count, dtype: int64


,source_file,ticker,year,quarter,currency,unit_scale,multiplier,FULL_NOMINAL,PRESENTATION_SCALED,total_votes,full_nominal_share,presentation_scaled_share,file_scale_mode
3,AMAG_2020_Q1_FS.xlsx,AMAG,2020,Q1,IDR,THOUSAND,1.000000e+03,1,3,4,0.250000,0.750000,AMBIGUOUS
5,ASBI_2021_Q3_FS.xlsx,ASBI,2021,Q3,IDR,THOUSAND,1.000000e+03,2,6,8,0.250000,0.750000,AMBIGUOUS
17,DEWA_2023_Q1_FS.xlsx,DEWA,2023,Q1,IDR,THOUSAND,1.000000e+03,6,6,12,0.500000,0.500000,AMBIGUOUS
20,HDFA_2020_Q1_FS.xlsx,HDFA,2020,Q1,IDR,THOUSAND,1.000000e+03,1,3,4,0.250000,0.750000,AMBIGUOUS
30,PANI_2022_Q3_FS.xlsx,PANI,2022,Q3,IDR,THOUSAND,1.000000e+03,3,9,12,0.250000,0.750000,AMBIGUOUS
39,TINS_2024_Q2_FS.xlsx,TINS,2024,Q2,IDR,MILLION,1.000000e+06,6,6,12,0.500000,0.500000,AMBIGUOUS
43,YOII_2024_Q4_FS.xlsx,YOII,2024,Q4,IDR,THOUSAND,1.000000e+03,1,3,4,0.250000,0.750000,AMBIGUOUS
0,AALI_2023_Q4_FS.xlsx,AALI,2023,Q4,IDR,MILLION,1.000000e+06,12,0,12,1.000000,0.000000,FULL_NOMINAL
1,ACST_2023_Q4_FS.xlsx,ACST,2023,Q4,IDR,MILLION,1.000000e+06,8,0,8,1.000000,0.000000,FULL_NOMINAL
2,ADRO_2023_Q4_FS.xlsx,ADRO,2023,Q4,USD,THOUSAND,1.000000e+03,10,0,10,1.000000,0.000000,FULL_NOMINAL


In [24]:
# CELL 71 - INSPECT AMBIGUOUS SCALE FILES

ambiguous_scale_files_df = (
    special_scale_files_with_meta_df[
        special_scale_files_with_meta_df[
            "file_scale_mode"
        ] == "AMBIGUOUS"
    ]
    .copy()
)


print(
    "Ambiguous files:",
    len(ambiguous_scale_files_df)
)


display(
    ambiguous_scale_files_df[
        [
            "source_file",
            "ticker",
            "year",
            "quarter",
            "currency",
            "unit_scale",
            "multiplier",
            "FULL_NOMINAL",
            "PRESENTATION_SCALED",
            "total_votes",
            "full_nominal_share",
            "presentation_scaled_share"
        ]
    ]
)


ambiguous_metric_details_df = (
    clean_metrics_with_scale_df[
        clean_metrics_with_scale_df[
            "source_file"
        ].isin(
            ambiguous_scale_files_df[
                "source_file"
            ]
        )
    ]
    .copy()
)


display(
    ambiguous_metric_details_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "source_file"
        ]
    ]
    .sort_values(
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    )
)

Ambiguous files: 7


,source_file,ticker,year,quarter,currency,unit_scale,multiplier,FULL_NOMINAL,PRESENTATION_SCALED,total_votes,full_nominal_share,presentation_scaled_share
3,AMAG_2020_Q1_FS.xlsx,AMAG,2020,Q1,IDR,THOUSAND,1000.0,1,3,4,0.25,0.75
5,ASBI_2021_Q3_FS.xlsx,ASBI,2021,Q3,IDR,THOUSAND,1000.0,2,6,8,0.25,0.75
17,DEWA_2023_Q1_FS.xlsx,DEWA,2023,Q1,IDR,THOUSAND,1000.0,6,6,12,0.50,0.50
20,HDFA_2020_Q1_FS.xlsx,HDFA,2020,Q1,IDR,THOUSAND,1000.0,1,3,4,0.25,0.75
30,PANI_2022_Q3_FS.xlsx,PANI,2022,Q3,IDR,THOUSAND,1000.0,3,9,12,0.25,0.75
39,TINS_2024_Q2_FS.xlsx,TINS,2024,Q2,IDR,MILLION,1000000.0,6,6,12,0.50,0.50
43,YOII_2024_Q4_FS.xlsx,YOII,2024,Q4,IDR,THOUSAND,1000.0,1,3,4,0.25,0.75


,ticker,year,quarter,metric,selected_value,currency,unit_scale,multiplier,source_file
3107,AMAG,2020,Q1,cash,1.236253e+08,IDR,THOUSAND,1000.0,AMAG_2020_Q1_FS.xlsx
3108,AMAG,2020,Q1,operating_cash_flow,8.594881e+07,IDR,THOUSAND,1000.0,AMAG_2020_Q1_FS.xlsx
3109,AMAG,2020,Q1,total_assets,5.092165e+09,IDR,THOUSAND,1000.0,AMAG_2020_Q1_FS.xlsx
3110,AMAG,2020,Q1,total_liabilities,3.218048e+09,IDR,THOUSAND,1000.0,AMAG_2020_Q1_FS.xlsx
5828,ASBI,2021,Q3,cash,1.615856e+07,IDR,THOUSAND,1000.0,ASBI_2021_Q3_FS.xlsx
5829,ASBI,2021,Q3,operating_cash_flow,-2.197129e+07,IDR,THOUSAND,1000.0,ASBI_2021_Q3_FS.xlsx
5830,ASBI,2021,Q3,total_assets,8.646677e+08,IDR,THOUSAND,1000.0,ASBI_2021_Q3_FS.xlsx
5831,ASBI,2021,Q3,total_liabilities,5.578020e+08,IDR,THOUSAND,1000.0,ASBI_2021_Q3_FS.xlsx
21196,DEWA,2023,Q1,cash,3.540602e+08,IDR,THOUSAND,1000.0,DEWA_2023_Q1_FS.xlsx
21197,DEWA,2023,Q1,gross_profit,1.233788e+08,IDR,THOUSAND,1000.0,DEWA_2023_Q1_FS.xlsx


In [25]:
# CELL 72 - SHOW NEIGHBOR PERIODS FOR AMBIGUOUS FILES

ambiguous_files = set(
    ambiguous_scale_files_df[
        "source_file"
    ]
)

ambiguous_rows = (
    clean_scale_mode_df[
        clean_scale_mode_df[
            "source_file"
        ].isin(
            ambiguous_files
        )
    ]
    .copy()
)

neighbor_rows = []

for _, row in ambiguous_rows.iterrows():

    ticker = row["ticker"]
    metric = row["metric"]
    year = row["year"]
    quarter_order_value = row["quarter_order"]

    group = (
        clean_scale_mode_df[
            (clean_scale_mode_df["ticker"] == ticker)
            &
            (clean_scale_mode_df["metric"] == metric)
        ]
        .sort_values(
            [
                "year",
                "quarter_order"
            ]
        )
        .reset_index(drop=True)
    )

    idx_matches = group[
        group["source_file"] == row["source_file"]
    ].index

    if len(idx_matches) == 0:
        continue

    idx = idx_matches[0]

    prev_row = (
        group.iloc[idx - 1]
        if idx > 0
        else None
    )

    next_row = (
        group.iloc[idx + 1]
        if idx < len(group) - 1
        else None
    )

    neighbor_rows.append(
        {
            "ticker": ticker,
            "metric": metric,
            "target_source_file": row["source_file"],
            "target_year": year,
            "target_quarter": row["quarter"],
            "target_value": row["selected_value"],
            "target_multiplier": row["multiplier"],

            "prev_source_file":
                None if prev_row is None
                else prev_row["source_file"],

            "prev_value":
                None if prev_row is None
                else prev_row["selected_value"],

            "prev_multiplier":
                None if prev_row is None
                else prev_row["multiplier"],

            "next_source_file":
                None if next_row is None
                else next_row["source_file"],

            "next_value":
                None if next_row is None
                else next_row["selected_value"],

            "next_multiplier":
                None if next_row is None
                else next_row["multiplier"],
        }
    )


ambiguous_neighbor_df = pd.DataFrame(
    neighbor_rows
)


display(
    ambiguous_neighbor_df[
        [
            "ticker",
            "metric",
            "target_source_file",
            "target_value",
            "target_multiplier",
            "prev_source_file",
            "prev_value",
            "prev_multiplier",
            "next_source_file",
            "next_value",
            "next_multiplier"
        ]
    ]
)

,ticker,metric,target_source_file,target_value,target_multiplier,prev_source_file,prev_value,prev_multiplier,next_source_file,next_value,next_multiplier
0,AMAG,cash,AMAG_2020_Q1_FS.xlsx,1.236253e+08,1000.0,NaN,NaN,NaN,AMAG_2020_Q2_FS.xlsx,9.811080e+07,1000.0
1,AMAG,operating_cash_flow,AMAG_2020_Q1_FS.xlsx,8.594881e+07,1000.0,NaN,NaN,NaN,AMAG_2020_Q2_FS.xlsx,6.244680e+05,1000.0
2,AMAG,total_assets,AMAG_2020_Q1_FS.xlsx,5.092165e+09,1000.0,NaN,NaN,NaN,AMAG_2020_Q2_FS.xlsx,4.935622e+09,1000.0
3,AMAG,total_liabilities,AMAG_2020_Q1_FS.xlsx,3.218048e+09,1000.0,NaN,NaN,NaN,AMAG_2020_Q2_FS.xlsx,2.951492e+09,1000.0
4,ASBI,cash,ASBI_2021_Q3_FS.xlsx,1.615856e+07,1000.0,ASBI_2021_Q2_FS.xlsx,20337227.0,1000.0,ASBI_2021_Q4_FS.xlsx,1.919089e+07,1000.0
5,ASBI,operating_cash_flow,ASBI_2021_Q3_FS.xlsx,-2.197129e+07,1000.0,ASBI_2021_Q2_FS.xlsx,-7636870.0,1000.0,ASBI_2021_Q4_FS.xlsx,-8.248747e+06,1000.0
6,ASBI,total_assets,ASBI_2021_Q3_FS.xlsx,8.646677e+08,1000.0,ASBI_2021_Q2_FS.xlsx,885837493.0,1000.0,ASBI_2021_Q4_FS.xlsx,9.546572e+08,1000.0
7,ASBI,total_liabilities,ASBI_2021_Q3_FS.xlsx,5.578020e+08,1000.0,ASBI_2021_Q2_FS.xlsx,573724858.0,1000.0,ASBI_2021_Q4_FS.xlsx,5.989147e+08,1000.0
8,DEWA,cash,DEWA_2023_Q1_FS.xlsx,3.540602e+08,1000.0,DEWA_2022_Q4_FS.xlsx,19318721.0,1.0,DEWA_2023_Q2_FS.xlsx,3.720212e+08,1000.0
9,DEWA,gross_profit,DEWA_2023_Q1_FS.xlsx,1.233788e+08,1000.0,DEWA_2022_Q4_FS.xlsx,558214.0,1.0,DEWA_2023_Q2_FS.xlsx,2.566706e+08,1000.0


In [26]:
# CELL 73 - SCORE AMBIGUOUS FILES USING BOTH NEIGHBORS

def log_distance(a, b):

    if (
        pd.isna(a)
        or pd.isna(b)
    ):
        return np.nan

    a = abs(float(a))
    b = abs(float(b))

    if a == 0 or b == 0:
        return np.nan

    return abs(
        np.log10(
            a / b
        )
    )


ambiguous_neighbor_scores = []


for _, row in ambiguous_neighbor_df.iterrows():

    target_raw = row["target_value"]
    target_multiplier = row["target_multiplier"]

    target_present = (
        target_raw
        * target_multiplier
    )

    target_full = target_raw


    present_scores = []
    full_scores = []


    # Previous neighbor
    if not pd.isna(row["prev_value"]):

        prev_candidates = [
            row["prev_value"],
            row["prev_value"]
            * row["prev_multiplier"]
        ]

        present_scores.append(
            min(
                log_distance(
                    target_present,
                    x
                )
                for x in prev_candidates
            )
        )

        full_scores.append(
            min(
                log_distance(
                    target_full,
                    x
                )
                for x in prev_candidates
            )
        )


    # Next neighbor
    if not pd.isna(row["next_value"]):

        next_candidates = [
            row["next_value"],
            row["next_value"]
            * row["next_multiplier"]
        ]

        present_scores.append(
            min(
                log_distance(
                    target_present,
                    x
                )
                for x in next_candidates
            )
        )

        full_scores.append(
            min(
                log_distance(
                    target_full,
                    x
                )
                for x in next_candidates
            )
        )


    present_score = (
        np.nanmean(present_scores)
        if present_scores
        else np.nan
    )

    full_score = (
        np.nanmean(full_scores)
        if full_scores
        else np.nan
    )


    if (
        pd.isna(present_score)
        or pd.isna(full_score)
    ):
        preferred_mode = "NO_EVIDENCE"

    elif present_score < full_score:
        preferred_mode = "PRESENTATION_SCALED"

    elif full_score < present_score:
        preferred_mode = "FULL_NOMINAL"

    else:
        preferred_mode = "TIE"


    ambiguous_neighbor_scores.append(
        {
            "ticker": row["ticker"],
            "metric": row["metric"],
            "target_source_file":
                row["target_source_file"],
            "present_score":
                present_score,
            "full_score":
                full_score,
            "preferred_mode":
                preferred_mode
        }
    )


ambiguous_neighbor_score_df = pd.DataFrame(
    ambiguous_neighbor_scores
)


print("METRIC-LEVEL AMBIGUOUS MODE PREFERENCE")

print(
    ambiguous_neighbor_score_df[
        "preferred_mode"
    ].value_counts(
        dropna=False
    )
)


display(
    ambiguous_neighbor_score_df
    .sort_values(
        [
            "target_source_file",
            "metric"
        ]
    )
)

METRIC-LEVEL AMBIGUOUS MODE PREFERENCE
preferred_mode
FULL_NOMINAL    19
TIE             15
Name: count, dtype: int64


,ticker,metric,target_source_file,present_score,full_score,preferred_mode
0,AMAG,cash,AMAG_2020_Q1_FS.xlsx,0.100391,0.100391,TIE
1,AMAG,operating_cash_flow,AMAG_2020_Q1_FS.xlsx,2.138730,0.861270,FULL_NOMINAL
2,AMAG,total_assets,AMAG_2020_Q1_FS.xlsx,0.013561,0.013561,TIE
3,AMAG,total_liabilities,AMAG_2020_Q1_FS.xlsx,0.037551,0.037551,TIE
4,ASBI,cash,ASBI_2021_Q3_FS.xlsx,0.087291,0.087291,TIE
5,ASBI,operating_cash_flow,ASBI_2021_Q3_FS.xlsx,0.442204,0.442204,TIE
6,ASBI,total_assets,ASBI_2021_Q3_FS.xlsx,0.026752,0.026752,TIE
7,ASBI,total_liabilities,ASBI_2021_Q3_FS.xlsx,0.021554,0.021554,FULL_NOMINAL
8,DEWA,cash,DEWA_2023_Q1_FS.xlsx,2.142295,0.642295,FULL_NOMINAL
9,DEWA,gross_profit,DEWA_2023_Q1_FS.xlsx,2.831288,1.331288,FULL_NOMINAL


In [27]:
# CELL 74 - FINAL VOTE FOR AMBIGUOUS FILES

ambiguous_file_resolution_df = (
    ambiguous_neighbor_score_df
    .groupby(
        "target_source_file"
    )
    .agg(
        presentation_votes=(
            "preferred_mode",
            lambda x:
                (x == "PRESENTATION_SCALED").sum()
        ),

        full_nominal_votes=(
            "preferred_mode",
            lambda x:
                (x == "FULL_NOMINAL").sum()
        ),

        no_evidence_votes=(
            "preferred_mode",
            lambda x:
                x.isin(
                    [
                        "NO_EVIDENCE",
                        "TIE"
                    ]
                ).sum()
        ),

        total_metric_votes=(
            "preferred_mode",
            "count"
        )
    )
    .reset_index()
)


def resolve_ambiguous_file(row):

    if (
        row["presentation_votes"]
        >
        row["full_nominal_votes"]
    ):
        return "PRESENTATION_SCALED"

    if (
        row["full_nominal_votes"]
        >
        row["presentation_votes"]
    ):
        return "FULL_NOMINAL"

    return "UNRESOLVED"


ambiguous_file_resolution_df[
    "resolved_scale_mode"
] = (
    ambiguous_file_resolution_df.apply(
        resolve_ambiguous_file,
        axis=1
    )
)


display(
    ambiguous_file_resolution_df
)


print("\nRESOLUTION COUNTS")

print(
    ambiguous_file_resolution_df[
        "resolved_scale_mode"
    ].value_counts(
        dropna=False
    )
)

,target_source_file,presentation_votes,full_nominal_votes,no_evidence_votes,total_metric_votes,resolved_scale_mode
0,AMAG_2020_Q1_FS.xlsx,0,1,3,4,FULL_NOMINAL
1,ASBI_2021_Q3_FS.xlsx,0,1,3,4,FULL_NOMINAL
2,DEWA_2023_Q1_FS.xlsx,0,6,0,6,FULL_NOMINAL
3,HDFA_2020_Q1_FS.xlsx,0,1,3,4,FULL_NOMINAL
4,PANI_2022_Q3_FS.xlsx,0,3,3,6,FULL_NOMINAL
5,TINS_2024_Q2_FS.xlsx,0,6,0,6,FULL_NOMINAL
6,YOII_2024_Q4_FS.xlsx,0,1,3,4,FULL_NOMINAL



RESOLUTION COUNTS
resolved_scale_mode
FULL_NOMINAL    7
Name: count, dtype: int64


In [28]:
# CELL 75 - RESOLVE AMBIGUOUS FILES USING ONLY CLASSIFIED NEIGHBORS
# IMPORTANT:
# Neighbor values are normalized according to the neighbor's EXISTING
# file-level classification from Cell 69.
# We do NOT let the algorithm freely choose neighbor scale mode.

known_file_mode_map = (
    clean_source_file_vote_summary_df
    .set_index("source_file")[
        "file_scale_mode"
    ]
    .to_dict()
)


def normalize_by_known_mode(
    raw_value,
    multiplier,
    file_mode
):
    if pd.isna(raw_value):
        return np.nan

    if file_mode == "PRESENTATION_SCALED":
        if pd.isna(multiplier):
            return np.nan

        return (
            float(raw_value)
            * float(multiplier)
        )

    if file_mode == "FULL_NOMINAL":
        return float(raw_value)

    # AMBIGUOUS / missing / unknown neighbor
    return np.nan


def safe_log_distance_v2(a, b):

    if pd.isna(a) or pd.isna(b):
        return np.nan

    a = abs(float(a))
    b = abs(float(b))

    if a == 0 or b == 0:
        return np.nan

    return abs(
        np.log10(
            a / b
        )
    )


resolved_metric_rows = []


for _, row in ambiguous_neighbor_df.iterrows():

    target_raw = row["target_value"]
    target_multiplier = row["target_multiplier"]

    if (
        pd.isna(target_raw)
        or pd.isna(target_multiplier)
    ):
        continue


    # Two possible interpretations for TARGET only
    target_as_presentation = (
        float(target_raw)
        * float(target_multiplier)
    )

    target_as_full = float(
        target_raw
    )


    neighbor_nominal_values = []


    # -------------------------
    # PREVIOUS NEIGHBOR
    # -------------------------

    prev_file = row["prev_source_file"]

    if pd.notna(prev_file):

        prev_mode = known_file_mode_map.get(
            prev_file
        )

        prev_nominal = normalize_by_known_mode(
            row["prev_value"],
            row["prev_multiplier"],
            prev_mode
        )

        if not pd.isna(prev_nominal):
            neighbor_nominal_values.append(
                prev_nominal
            )


    # -------------------------
    # NEXT NEIGHBOR
    # -------------------------

    next_file = row["next_source_file"]

    if pd.notna(next_file):

        next_mode = known_file_mode_map.get(
            next_file
        )

        next_nominal = normalize_by_known_mode(
            row["next_value"],
            row["next_multiplier"],
            next_mode
        )

        if not pd.isna(next_nominal):
            neighbor_nominal_values.append(
                next_nominal
            )


    if len(neighbor_nominal_values) == 0:

        presentation_score = np.nan
        full_score = np.nan
        preferred_mode = "NO_EVIDENCE"

    else:

        presentation_distances = [
            safe_log_distance_v2(
                target_as_presentation,
                neighbor_value
            )
            for neighbor_value
            in neighbor_nominal_values
        ]

        full_distances = [
            safe_log_distance_v2(
                target_as_full,
                neighbor_value
            )
            for neighbor_value
            in neighbor_nominal_values
        ]


        presentation_score = (
            np.nanmean(
                presentation_distances
            )
        )

        full_score = (
            np.nanmean(
                full_distances
            )
        )


        if (
            pd.isna(presentation_score)
            or pd.isna(full_score)
        ):
            preferred_mode = "NO_EVIDENCE"

        elif presentation_score < full_score:
            preferred_mode = (
                "PRESENTATION_SCALED"
            )

        elif full_score < presentation_score:
            preferred_mode = (
                "FULL_NOMINAL"
            )

        else:
            preferred_mode = "TIE"


    resolved_metric_rows.append(
        {
            "ticker":
                row["ticker"],

            "metric":
                row["metric"],

            "target_source_file":
                row["target_source_file"],

            "known_neighbor_count":
                len(
                    neighbor_nominal_values
                ),

            "presentation_score":
                presentation_score,

            "full_score":
                full_score,

            "preferred_mode":
                preferred_mode
        }
    )


ambiguous_metric_resolution_v2_df = (
    pd.DataFrame(
        resolved_metric_rows
    )
)


print(
    "METRIC-LEVEL RESOLUTION V2"
)

print(
    ambiguous_metric_resolution_v2_df[
        "preferred_mode"
    ].value_counts(
        dropna=False
    )
)


display(
    ambiguous_metric_resolution_v2_df
    .sort_values(
        [
            "target_source_file",
            "metric"
        ]
    )
)

METRIC-LEVEL RESOLUTION V2
preferred_mode
PRESENTATION_SCALED    19
FULL_NOMINAL            9
TIE                     6
Name: count, dtype: int64


,ticker,metric,target_source_file,known_neighbor_count,presentation_score,full_score,preferred_mode
0,AMAG,cash,AMAG_2020_Q1_FS.xlsx,1,0.100391,2.899609,PRESENTATION_SCALED
1,AMAG,operating_cash_flow,AMAG_2020_Q1_FS.xlsx,1,2.138730,0.861270,FULL_NOMINAL
2,AMAG,total_assets,AMAG_2020_Q1_FS.xlsx,1,0.013561,2.986439,PRESENTATION_SCALED
3,AMAG,total_liabilities,AMAG_2020_Q1_FS.xlsx,1,0.037551,2.962449,PRESENTATION_SCALED
4,ASBI,cash,ASBI_2021_Q3_FS.xlsx,2,0.087291,3.087291,PRESENTATION_SCALED
5,ASBI,operating_cash_flow,ASBI_2021_Q3_FS.xlsx,2,0.442204,2.557796,PRESENTATION_SCALED
6,ASBI,total_assets,ASBI_2021_Q3_FS.xlsx,2,0.026752,3.026752,PRESENTATION_SCALED
7,ASBI,total_liabilities,ASBI_2021_Q3_FS.xlsx,2,0.021554,3.021554,PRESENTATION_SCALED
8,DEWA,cash,DEWA_2023_Q1_FS.xlsx,2,2.142295,2.142295,TIE
9,DEWA,gross_profit,DEWA_2023_Q1_FS.xlsx,2,2.831288,2.831288,TIE


In [29]:
# CELL 76 - AGGREGATE CORRECTED AMBIGUOUS RESOLUTION PER FILE

ambiguous_file_resolution_v2_df = (
    ambiguous_metric_resolution_v2_df
    .groupby(
        "target_source_file"
    )
    .agg(
        presentation_votes=(
            "preferred_mode",
            lambda x:
                (
                    x
                    == "PRESENTATION_SCALED"
                ).sum()
        ),

        full_nominal_votes=(
            "preferred_mode",
            lambda x:
                (
                    x
                    == "FULL_NOMINAL"
                ).sum()
        ),

        no_evidence_votes=(
            "preferred_mode",
            lambda x:
                x.isin(
                    [
                        "NO_EVIDENCE",
                        "TIE"
                    ]
                ).sum()
        ),

        metrics_with_known_neighbor=(
            "known_neighbor_count",
            lambda x:
                (x > 0).sum()
        ),

        total_metrics=(
            "metric",
            "count"
        )
    )
    .reset_index()
)


def resolve_file_v2(row):

    p = row[
        "presentation_votes"
    ]

    f = row[
        "full_nominal_votes"
    ]

    if p > f:
        return "PRESENTATION_SCALED"

    if f > p:
        return "FULL_NOMINAL"

    return "UNRESOLVED"


ambiguous_file_resolution_v2_df[
    "resolved_scale_mode"
] = (
    ambiguous_file_resolution_v2_df
    .apply(
        resolve_file_v2,
        axis=1
    )
)


display(
    ambiguous_file_resolution_v2_df
)


print("\nCORRECTED RESOLUTION COUNTS")

print(
    ambiguous_file_resolution_v2_df[
        "resolved_scale_mode"
    ].value_counts(
        dropna=False
    )
)

,target_source_file,presentation_votes,full_nominal_votes,no_evidence_votes,metrics_with_known_neighbor,total_metrics,resolved_scale_mode
0,AMAG_2020_Q1_FS.xlsx,3,1,0,4,4,PRESENTATION_SCALED
1,ASBI_2021_Q3_FS.xlsx,4,0,0,4,4,PRESENTATION_SCALED
2,DEWA_2023_Q1_FS.xlsx,0,0,6,6,6,UNRESOLVED
3,HDFA_2020_Q1_FS.xlsx,3,1,0,4,4,PRESENTATION_SCALED
4,PANI_2022_Q3_FS.xlsx,6,0,0,6,6,PRESENTATION_SCALED
5,TINS_2024_Q2_FS.xlsx,0,6,0,6,6,FULL_NOMINAL
6,YOII_2024_Q4_FS.xlsx,3,1,0,4,4,PRESENTATION_SCALED



CORRECTED RESOLUTION COUNTS
resolved_scale_mode
PRESENTATION_SCALED    5
UNRESOLVED             1
FULL_NOMINAL           1
Name: count, dtype: int64


In [30]:
# CELL 77 - INSPECT DEWA 2023 Q1 WITH NEIGHBOR MODES

DEWA_TARGET_FILE = "DEWA_2023_Q1_FS.xlsx"


dewa_target_rows_df = (
    clean_scale_mode_df[
        clean_scale_mode_df[
            "source_file"
        ] == DEWA_TARGET_FILE
    ]
    .copy()
)


dewa_neighbor_audit_rows = []


for _, row in dewa_target_rows_df.iterrows():

    ticker = row["ticker"]
    metric = row["metric"]

    metric_history = (
        clean_scale_mode_df[
            (clean_scale_mode_df["ticker"] == ticker)
            &
            (clean_scale_mode_df["metric"] == metric)
        ]
        .sort_values(
            [
                "year",
                "quarter_order"
            ]
        )
        .reset_index(drop=True)
    )


    target_matches = metric_history[
        metric_history[
            "source_file"
        ] == DEWA_TARGET_FILE
    ].index


    if len(target_matches) == 0:
        continue


    idx = target_matches[0]


    start_idx = max(
        0,
        idx - 2
    )

    end_idx = min(
        len(metric_history),
        idx + 3
    )


    history_slice = (
        metric_history.iloc[
            start_idx:end_idx
        ]
        .copy()
    )


    for _, history_row in history_slice.iterrows():

        file_mode = known_file_mode_map.get(
            history_row[
                "source_file"
            ]
        )


        if file_mode == "PRESENTATION_SCALED":

            normalized_candidate = (
                history_row[
                    "selected_value"
                ]
                *
                history_row[
                    "multiplier"
                ]
            )

        elif file_mode == "FULL_NOMINAL":

            normalized_candidate = (
                history_row[
                    "selected_value"
                ]
            )

        else:

            normalized_candidate = np.nan


        dewa_neighbor_audit_rows.append(
            {
                "metric":
                    metric,

                "source_file":
                    history_row[
                        "source_file"
                    ],

                "year":
                    history_row[
                        "year"
                    ],

                "quarter":
                    history_row[
                        "quarter"
                    ],

                "selected_value":
                    history_row[
                        "selected_value"
                    ],

                "unit_scale":
                    history_row[
                        "unit_scale"
                    ],

                "multiplier":
                    history_row[
                        "multiplier"
                    ],

                "file_scale_mode":
                    file_mode,

                "normalized_if_known":
                    normalized_candidate,

                "is_target":
                    history_row[
                        "source_file"
                    ] == DEWA_TARGET_FILE
            }
        )


dewa_neighbor_audit_df = pd.DataFrame(
    dewa_neighbor_audit_rows
)


display(
    dewa_neighbor_audit_df[
        [
            "metric",
            "year",
            "quarter",
            "source_file",
            "selected_value",
            "unit_scale",
            "multiplier",
            "file_scale_mode",
            "normalized_if_known",
            "is_target"
        ]
    ]
    .sort_values(
        [
            "metric",
            "year",
            "quarter"
        ]
    )
)

,metric,year,quarter,source_file,selected_value,unit_scale,multiplier,file_scale_mode,normalized_if_known,is_target
0,cash,2022,Q3,DEWA_2022_Q3_FS.xlsx,3.956341e+07,UNIT,1.0,PRESENTATION_SCALED,3.956341e+07,False
1,cash,2022,Q4,DEWA_2022_Q4_FS.xlsx,1.931872e+07,UNIT,1.0,PRESENTATION_SCALED,1.931872e+07,False
2,cash,2023,Q1,DEWA_2023_Q1_FS.xlsx,3.540602e+08,THOUSAND,1000.0,AMBIGUOUS,NaN,True
3,cash,2023,Q2,DEWA_2023_Q2_FS.xlsx,3.720212e+08,THOUSAND,1000.0,PRESENTATION_SCALED,3.720212e+11,False
4,cash,2023,Q3,DEWA_2023_Q3_FS.xlsx,4.404680e+08,THOUSAND,1000.0,PRESENTATION_SCALED,4.404680e+11,False
5,gross_profit,2022,Q3,DEWA_2022_Q3_FS.xlsx,2.574168e+06,UNIT,1.0,PRESENTATION_SCALED,2.574168e+06,False
6,gross_profit,2022,Q4,DEWA_2022_Q4_FS.xlsx,5.582140e+05,UNIT,1.0,PRESENTATION_SCALED,5.582140e+05,False
7,gross_profit,2023,Q1,DEWA_2023_Q1_FS.xlsx,1.233788e+08,THOUSAND,1000.0,AMBIGUOUS,NaN,True
8,gross_profit,2023,Q2,DEWA_2023_Q2_FS.xlsx,2.566706e+08,THOUSAND,1000.0,PRESENTATION_SCALED,2.566706e+11,False
9,gross_profit,2023,Q3,DEWA_2023_Q3_FS.xlsx,3.153330e+08,THOUSAND,1000.0,PRESENTATION_SCALED,3.153330e+11,False


In [36]:
# CELL 78 - FINALIZE SCALE MODE FOR ALL CLASSIFIED CLEAN SOURCE FILES
# UPDATED: DEWA 2023 Q1 resolved as PRESENTATION_SCALED

final_scale_mode_df = (
    clean_source_file_vote_summary_df[
        [
            "source_file",
            "file_scale_mode"
        ]
    ]
    .copy()
    .rename(
        columns={
            "file_scale_mode": "final_scale_mode"
        }
    )
)


# Override previously ambiguous files using corrected resolution
ambiguous_resolution_map = (
    ambiguous_file_resolution_v2_df
    .set_index(
        "target_source_file"
    )[
        "resolved_scale_mode"
    ]
    .to_dict()
)


final_scale_mode_df[
    "final_scale_mode"
] = (
    final_scale_mode_df.apply(
        lambda row:
            ambiguous_resolution_map.get(
                row["source_file"],
                row["final_scale_mode"]
            ),
        axis=1
    )
)


# DEWA recheck (Cell 81):
# 12 votes PRESENTATION_SCALED vs 6 FULL_NOMINAL
final_scale_mode_df.loc[
    final_scale_mode_df[
        "source_file"
    ] == "DEWA_2023_Q1_FS.xlsx",
    "final_scale_mode"
] = "PRESENTATION_SCALED"


print("FINAL SCALE MODE COUNTS")

print(
    final_scale_mode_df[
        "final_scale_mode"
    ].value_counts(
        dropna=False
    )
)


print("\nSTILL UNRESOLVED")

display(
    final_scale_mode_df[
        ~final_scale_mode_df[
            "final_scale_mode"
        ].isin(
            [
                "PRESENTATION_SCALED",
                "FULL_NOMINAL"
            ]
        )
    ]
)


print(
    "\nTotal classified source files:",
    len(final_scale_mode_df)
)

FINAL SCALE MODE COUNTS
final_scale_mode
PRESENTATION_SCALED    14709
FULL_NOMINAL              38
Name: count, dtype: int64

STILL UNRESOLVED


mode_vote,source_file,final_scale_mode



Total classified source files: 14747


In [55]:
# CELL 79 - APPLY FINAL SCALE MODE AND AUDIT NORMALIZED VALUES
# UPDATED AFTER CELL 88

clean_metrics_normalized_df = (
    clean_metrics_with_scale_df
    .merge(
        final_scale_mode_df,
        on="source_file",
        how="left",
        validate="many_to_one"
    )
)


print(
    "Rows after scale-mode merge:",
    len(clean_metrics_normalized_df)
)

print(
    "Missing final scale mode:",
    clean_metrics_normalized_df[
        "final_scale_mode"
    ].isna().sum()
)


def calculate_final_normalized_value(row):

    # Kalau metadata currency/scale tidak ditemukan,
    # jangan tebak hasil normalisasi.
    if row["currency_scale_status"] != "FOUND":
        return np.nan

    if pd.isna(row["selected_value"]):
        return np.nan

    if row["final_scale_mode"] == "PRESENTATION_SCALED":

        if pd.isna(row["multiplier"]):
            return np.nan

        return (
            float(row["selected_value"])
            * float(row["multiplier"])
        )

    if row["final_scale_mode"] == "FULL_NOMINAL":

        return float(
            row["selected_value"]
        )

    return np.nan


clean_metrics_normalized_df[
    "normalized_value"
] = (
    clean_metrics_normalized_df.apply(
        calculate_final_normalized_value,
        axis=1
    )
)


print("\nNORMALIZATION STATUS")

print(
    clean_metrics_normalized_df[
        "normalized_value"
    ].notna().value_counts()
)


print("\nFINAL SCALE MODE BY METRIC ROW")

print(
    clean_metrics_normalized_df[
        "final_scale_mode"
    ].value_counts(
        dropna=False
    )
)


print("\nNORMALIZED VALUE SUMMARY")

print(
    clean_metrics_normalized_df[
        "normalized_value"
    ].describe()
)


print("\nKNOWN CASE SANITY CHECK")

known_files = [
    "AALI_2020_Q1_FS.xlsx",
    "AALI_2023_Q4_FS.xlsx",
    "AALI_2024_Q2_FS.xlsx",
    "ASII_2020_Q1_FS.xlsx",
    "ASII_2023_Q4_FS.xlsx",
    "BMRI_2023_Q4_FS.xlsx",
    "TINS_2024_Q2_FS.xlsx",
    "DEWA_2023_Q1_FS.xlsx",
    "CBDK_2024_Q4_FS.xlsx",
    "HGII_2025_Q1_FS.xlsx",
    "MDIY_2025_Q1_FS.xlsx",
]


display(
    clean_metrics_normalized_df[
        clean_metrics_normalized_df[
            "source_file"
        ].isin(
            known_files
        )
    ][
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "final_scale_mode",
            "normalized_value",
            "source_file"
        ]
    ]
    .sort_values(
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    )
)

Rows after scale-mode merge: 83442
Missing final scale mode: 4

NORMALIZATION STATUS
normalized_value
True     83438
False        4
Name: count, dtype: int64

FINAL SCALE MODE BY METRIC ROW
final_scale_mode
PRESENTATION_SCALED    83269
FULL_NOMINAL             169
NaN                        4
Name: count, dtype: int64

NORMALIZED VALUE SUMMARY
count    8.343800e+04
mean     5.968938e+12
std      6.286793e+13
min     -1.211337e+14
25%      4.230170e+09
50%      1.191094e+11
75%      9.445622e+11
max      2.463659e+15
Name: normalized_value, dtype: float64

KNOWN CASE SANITY CHECK


,ticker,year,quarter,metric,selected_value,currency,unit_scale,multiplier,final_scale_mode,normalized_value,source_file
12,AALI,2020,Q1,cash,1470866.0,IDR,MILLION,1000000.0,PRESENTATION_SCALED,1.470866e+12,AALI_2020_Q1_FS.xlsx
13,AALI,2020,Q1,gross_profit,926791.0,IDR,MILLION,1000000.0,PRESENTATION_SCALED,9.267910e+11,AALI_2020_Q1_FS.xlsx
14,AALI,2020,Q1,operating_cash_flow,660610.0,IDR,MILLION,1000000.0,PRESENTATION_SCALED,6.606100e+11,AALI_2020_Q1_FS.xlsx
15,AALI,2020,Q1,revenue,4796084.0,IDR,MILLION,1000000.0,PRESENTATION_SCALED,4.796084e+12,AALI_2020_Q1_FS.xlsx
16,AALI,2020,Q1,total_assets,29218599.0,IDR,MILLION,1000000.0,PRESENTATION_SCALED,2.921860e+13,AALI_2020_Q1_FS.xlsx
...,...,...,...,...,...,...,...,...,...,...,...
75545,TINS,2024,Q2,gross_profit,1214050.0,IDR,MILLION,1000000.0,FULL_NOMINAL,1.214050e+06,TINS_2024_Q2_FS.xlsx
75546,TINS,2024,Q2,operating_cash_flow,889524.0,IDR,MILLION,1000000.0,FULL_NOMINAL,8.895240e+05,TINS_2024_Q2_FS.xlsx
75547,TINS,2024,Q2,revenue,5211895.0,IDR,MILLION,1000000.0,FULL_NOMINAL,5.211895e+06,TINS_2024_Q2_FS.xlsx
75548,TINS,2024,Q2,total_assets,13250259.0,IDR,MILLION,1000000.0,FULL_NOMINAL,1.325026e+07,TINS_2024_Q2_FS.xlsx


In [56]:
# CELL 89 - RECHECK TINS 2024 Q2 BEFORE FINAL NORMALIZATION

TINS_TARGET_FILE = "TINS_2024_Q2_FS.xlsx"

tins_metrics = (
    clean_metrics_with_scale_df[
        clean_metrics_with_scale_df[
            "source_file"
        ] == TINS_TARGET_FILE
    ]["metric"]
    .unique()
)


tins_audit_rows = []


for metric in tins_metrics:

    history = (
        clean_metrics_with_scale_df[
            (
                clean_metrics_with_scale_df["ticker"] == "TINS"
            )
            &
            (
                clean_metrics_with_scale_df["metric"] == metric
            )
        ]
        .merge(
            final_scale_mode_df,
            on="source_file",
            how="left"
        )
        .copy()
    )

    history["quarter_order"] = (
        history["quarter"].map(
            {
                "Q1": 1,
                "Q2": 2,
                "Q3": 3,
                "Q4": 4
            }
        )
    )

    history = (
        history
        .sort_values(
            [
                "year",
                "quarter_order"
            ]
        )
        .reset_index(drop=True)
    )


    for _, row in history.iterrows():

        if row["final_scale_mode"] == "PRESENTATION_SCALED":

            normalized = (
                row["selected_value"]
                * row["multiplier"]
            )

        elif row["final_scale_mode"] == "FULL_NOMINAL":

            normalized = row["selected_value"]

        else:
            normalized = np.nan


        tins_audit_rows.append(
            {
                "metric": metric,
                "year": row["year"],
                "quarter": row["quarter"],
                "source_file": row["source_file"],
                "selected_value": row["selected_value"],
                "unit_scale": row["unit_scale"],
                "multiplier": row["multiplier"],
                "final_scale_mode": row["final_scale_mode"],
                "normalized_current_rule": normalized,
                "is_target": (
                    row["source_file"]
                    == TINS_TARGET_FILE
                )
            }
        )


tins_history_audit_df = pd.DataFrame(
    tins_audit_rows
)


display(
    tins_history_audit_df[
        [
            "metric",
            "year",
            "quarter",
            "source_file",
            "selected_value",
            "unit_scale",
            "multiplier",
            "final_scale_mode",
            "normalized_current_rule",
            "is_target"
        ]
    ]
    .sort_values(
        [
            "metric",
            "year",
            "quarter"
        ]
    )
)

,metric,year,quarter,source_file,selected_value,unit_scale,multiplier,final_scale_mode,normalized_current_rule,is_target
0,cash,2021,Q1,TINS_2021_Q1_FS.xlsx,715102.0,MILLION,1000000.0,PRESENTATION_SCALED,7.151020e+11,False
1,cash,2021,Q2,TINS_2021_Q2_FS.xlsx,1286157.0,MILLION,1000000.0,PRESENTATION_SCALED,1.286157e+12,False
2,cash,2021,Q3,TINS_2021_Q3_FS.xlsx,1101560.0,MILLION,1000000.0,PRESENTATION_SCALED,1.101560e+12,False
3,cash,2021,Q4,TINS_2021_Q4_FS.xlsx,1782262.0,MILLION,1000000.0,PRESENTATION_SCALED,1.782262e+12,False
4,cash,2022,Q1,TINS_2022_Q1_FS.xlsx,2115098.0,MILLION,1000000.0,PRESENTATION_SCALED,2.115098e+12,False
...,...,...,...,...,...,...,...,...,...,...
85,total_liabilities,2023,Q3,TINS_2023_Q3_FS.xlsx,6088916.0,MILLION,1000000.0,PRESENTATION_SCALED,6.088916e+12,False
86,total_liabilities,2023,Q4,TINS_2023_Q4_FS.xlsx,6610928.0,MILLION,1000000.0,PRESENTATION_SCALED,6.610928e+12,False
87,total_liabilities,2024,Q1,TINS_2024_Q1_FS.xlsx,6455352.0,MILLION,1000000.0,PRESENTATION_SCALED,6.455352e+12,False
88,total_liabilities,2024,Q2,TINS_2024_Q2_FS.xlsx,6477679.0,MILLION,1000000.0,FULL_NOMINAL,6.477679e+06,True


In [57]:
# CELL 90 - FIX TINS 2024 Q2 AND AUDIT TINS 2025 Q1

# --------------------------------------------------
# 1. FIX TINS 2024 Q2
# Cell 89 clearly shows it is presentation-scaled.
# --------------------------------------------------

final_scale_mode_df.loc[
    final_scale_mode_df["source_file"]
    == "TINS_2024_Q2_FS.xlsx",
    "final_scale_mode"
] = "PRESENTATION_SCALED"


print("TINS 2024 Q2 updated mode:")

display(
    final_scale_mode_df[
        final_scale_mode_df["source_file"]
        == "TINS_2024_Q2_FS.xlsx"
    ]
)


# --------------------------------------------------
# 2. AUDIT TINS 2025 Q1
# Its metadata says UNIT, but raw values look similar
# to earlier MILLION-scaled reports.
# --------------------------------------------------

TINS_2025_FILE = "TINS_2025_Q1_FS.xlsx"


tins_2025_audit_df = (
    clean_metrics_with_scale_df[
        clean_metrics_with_scale_df["source_file"]
        == TINS_2025_FILE
    ]
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "source_file",
            "source_path",
            "source_sheet",
            "source_label"
        ]
    ]
    .copy()
)


print("\nTINS 2025 Q1 CURRENT MAPPING")

display(
    tins_2025_audit_df[
        [
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "source_sheet",
            "source_label"
        ]
    ]
)

TINS 2024 Q2 updated mode:


,source_file,final_scale_mode
13371,TINS_2024_Q2_FS.xlsx,PRESENTATION_SCALED



TINS 2025 Q1 CURRENT MAPPING


,metric,selected_value,currency,unit_scale,multiplier,source_sheet,source_label
75550,cash,1619417.0,IDR,UNIT,1.0,1210000,Kas dan setara kas
75551,gross_profit,382432.0,IDR,UNIT,1.0,1311000,Jumlah laba bruto
75552,operating_cash_flow,297253.0,IDR,UNIT,1.0,1510000,Total net cash flows received from (used in) o...
75553,revenue,2097892.0,IDR,UNIT,1.0,1311000,Sales and revenue
75554,total_assets,12488575.0,IDR,UNIT,1.0,1210000,Jumlah aset
75555,total_liabilities,4851112.0,IDR,UNIT,1.0,1210000,Jumlah liabilitas


In [58]:
# CELL 91 - INSPECT TINS 2025 Q1 WORKBOOK SCALE TEXT

from openpyxl import load_workbook


source_row = (
    tins_2025_audit_df
    .iloc[0]
)

source_path = source_row["source_path"]


wb = None

try:

    wb = load_workbook(
        source_path,
        read_only=True,
        data_only=True
    )


    print(
        "FILE:",
        TINS_2025_FILE
    )

    print(
        "CURRENT MAPPING:",
        source_row["unit_scale"],
        "| multiplier:",
        source_row["multiplier"]
    )


    relevant_sheets = (
        tins_2025_audit_df[
            "source_sheet"
        ]
        .dropna()
        .astype(str)
        .unique()
    )


    for sheet_name in relevant_sheets:

        if sheet_name not in wb.sheetnames:
            continue

        ws = wb[sheet_name]

        text_values = []

        for row in ws.iter_rows(
            min_row=1,
            max_row=min(
                ws.max_row,
                25
            ),
            min_col=1,
            max_col=min(
                ws.max_column,
                15
            ),
            values_only=True
        ):

            for value in row:

                if value is not None:

                    text = str(
                        value
                    ).strip()

                    if text:
                        text_values.append(
                            text
                        )


        print(
            "\n" + "=" * 120
        )

        print(
            "SHEET:",
            sheet_name
        )

        print(
            "HEADER / TOP TEXT:"
        )

        print(
            " | ".join(
                text_values
            )[:4000]
        )


finally:

    if wb is not None:
        wb.close()

FILE: TINS_2025_Q1_FS.xlsx
CURRENT MAPPING: UNIT | multiplier: 1.0

SHEET: 1210000
HEADER / TOP TEXT:
[1210000] Statement of financial position presented using current and non-current - General Industry | Laporan posisi keuangan | Statement of financial position | CurrentYearInstant | PriorEndYearInstant | Laporan posisi keuangan | Statement of financial position | Aset | Assets | Aset lancar | Current assets | Kas dan setara kas | 1619417 | 1988254 | Cash and cash equivalents | Wesel tagih | Notes receivable | Investasi jangka pendek | Short-term investments | Dana yang dibatasi penggunaannya lancar | Current restricted funds | Aset keuangan lancar | Current financial assets | Aset keuangan lancar yang diukur pada nilai wajar melalui laba rugi | Current financial assets at fair value through profit or loss | Aset keuangan lancar nilai wajar melalui pendapatan komprehensif lainnya | Current financial assets fair value through other comprehensive income | Aset keuangan biaya perolehan d

In [59]:
# CELL 92 - INSPECT TINS 2025 Q1 METADATA SCALE DIRECTLY

from openpyxl import load_workbook

TINS_2025_FILE = "TINS_2025_Q1_FS.xlsx"

source_path = (
    tins_2025_audit_df
    .iloc[0]["source_path"]
)

wb = None

try:

    wb = load_workbook(
        source_path,
        read_only=True,
        data_only=True
    )

    print("FILE:", TINS_2025_FILE)

    if "1000000" not in wb.sheetnames:

        print("Metadata sheet 1000000 not found.")

    else:

        ws = wb["1000000"]

        print("\nMETADATA SHEET 1000000")
        print("=" * 120)

        for row_idx in range(
            1,
            min(ws.max_row, 60) + 1
        ):

            values = []

            for col_idx in range(
                1,
                min(ws.max_column, 12) + 1
            ):

                value = ws.cell(
                    row=row_idx,
                    column=col_idx
                ).value

                if value is not None:

                    values.append(
                        str(value).strip()
                    )

            if values:

                row_text = " | ".join(values)

                lower_text = row_text.lower()

                # Print rows relevant to currency / presentation scale
                keywords = [
                    "pembulatan",
                    "rounding",
                    "mata uang",
                    "currency",
                    "satuan",
                    "unit",
                    "ribu",
                    "thousand",
                    "juta",
                    "million",
                    "miliar",
                    "billion"
                ]

                if any(
                    keyword in lower_text
                    for keyword in keywords
                ):

                    print(
                        f"ROW {row_idx}:",
                        row_text
                    )

finally:

    if wb is not None:
        wb.close()

FILE: TINS_2025_Q1_FS.xlsx

METADATA SHEET 1000000
ROW 29: Mata uang pelaporan | Rupiah / IDR | Description of presentation currency
ROW 30: Kurs konversi pada tanggal pelaporan jika mata uang penyajian selain rupiah | Conversion rate at reporting date if presentation currency is other than rupiah
ROW 31: Pembulatan yang digunakan dalam penyajian jumlah dalam laporan keuangan | Satuan Penuh / Full Amount | Level of rounding used in financial statements


In [60]:
# CELL 93 - COMPARE TINS 2025 Q1 WITH TINS 2024 Q1
# Goal:
# Determine whether TINS 2025 Q1 numeric values still behave like MILLION
# despite metadata saying Full Amount.

TINS_2024_FILE = "TINS_2024_Q1_FS.xlsx"
TINS_2025_FILE = "TINS_2025_Q1_FS.xlsx"


tins_compare_df = (
    clean_metrics_with_scale_df[
        clean_metrics_with_scale_df[
            "source_file"
        ].isin(
            [
                TINS_2024_FILE,
                TINS_2025_FILE
            ]
        )
    ]
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "source_file"
        ]
    ]
    .copy()
)


tins_compare_df = (
    tins_compare_df
    .merge(
        final_scale_mode_df,
        on="source_file",
        how="left"
    )
)


print("TINS 2024 Q1 VS TINS 2025 Q1")

display(
    tins_compare_df[
        [
            "year",
            "quarter",
            "metric",
            "selected_value",
            "unit_scale",
            "multiplier",
            "final_scale_mode",
            "source_file"
        ]
    ]
    .sort_values(
        [
            "metric",
            "year",
            "quarter"
        ]
    )
)


print("\nRAW VALUE RATIO 2025 Q1 / 2024 Q1")

pivot = (
    tins_compare_df
    .pivot_table(
        index="metric",
        columns="source_file",
        values="selected_value",
        aggfunc="first"
    )
)


if (
    TINS_2024_FILE in pivot.columns
    and TINS_2025_FILE in pivot.columns
):

    pivot[
        "raw_ratio_2025_vs_2024"
    ] = (
        pivot[
            TINS_2025_FILE
        ]
        /
        pivot[
            TINS_2024_FILE
        ]
    )


display(pivot)

TINS 2024 Q1 VS TINS 2025 Q1


,year,quarter,metric,selected_value,unit_scale,multiplier,final_scale_mode,source_file
0,2024,Q1,cash,907241.0,MILLION,1000000.0,PRESENTATION_SCALED,TINS_2024_Q1_FS.xlsx
6,2025,Q1,cash,1619417.0,UNIT,1.0,PRESENTATION_SCALED,TINS_2025_Q1_FS.xlsx
1,2024,Q1,gross_profit,295391.0,MILLION,1000000.0,PRESENTATION_SCALED,TINS_2024_Q1_FS.xlsx
7,2025,Q1,gross_profit,382432.0,UNIT,1.0,PRESENTATION_SCALED,TINS_2025_Q1_FS.xlsx
2,2024,Q1,operating_cash_flow,-202832.0,MILLION,1000000.0,PRESENTATION_SCALED,TINS_2024_Q1_FS.xlsx
8,2025,Q1,operating_cash_flow,297253.0,UNIT,1.0,PRESENTATION_SCALED,TINS_2025_Q1_FS.xlsx
3,2024,Q1,revenue,2056597.0,MILLION,1000000.0,PRESENTATION_SCALED,TINS_2024_Q1_FS.xlsx
9,2025,Q1,revenue,2097892.0,UNIT,1.0,PRESENTATION_SCALED,TINS_2025_Q1_FS.xlsx
4,2024,Q1,total_assets,12823232.0,MILLION,1000000.0,PRESENTATION_SCALED,TINS_2024_Q1_FS.xlsx
10,2025,Q1,total_assets,12488575.0,UNIT,1.0,PRESENTATION_SCALED,TINS_2025_Q1_FS.xlsx



RAW VALUE RATIO 2025 Q1 / 2024 Q1


source_file,TINS_2024_Q1_FS.xlsx,TINS_2025_Q1_FS.xlsx,raw_ratio_2025_vs_2024
metric,,,
cash,907241.0,1619417.0,1.784991
gross_profit,295391.0,382432.0,1.294664
operating_cash_flow,-202832.0,297253.0,-1.465513
revenue,2056597.0,2097892.0,1.020079
total_assets,12823232.0,12488575.0,0.973902
total_liabilities,6455352.0,4851112.0,0.751487


In [61]:
# CELL 94 - CONSOLIDATE FINAL SCALE DECISIONS AND EXCEPTIONS

# ---------------------------------------------------------
# FINAL FILE-SPECIFIC SCALE EXCEPTIONS
# ---------------------------------------------------------

final_scale_exceptions = {
    # Rechecks showed these are presentation-scaled
    "DEWA_2023_Q1_FS.xlsx": {
        "final_scale_mode": "PRESENTATION_SCALED",
        "override_multiplier": None,
        "reason": "Temporal neighbor recheck supports presentation-scaled."
    },

    "TINS_2024_Q2_FS.xlsx": {
        "final_scale_mode": "PRESENTATION_SCALED",
        "override_multiplier": None,
        "reason": "Raw values align with surrounding MILLION-scaled TINS periods."
    },

    # Metadata says UNIT / Full Amount, but raw values align almost exactly
    # with prior MILLION-scaled TINS statements.
    "TINS_2025_Q1_FS.xlsx": {
        "final_scale_mode": "PRESENTATION_SCALED",
        "override_multiplier": 1_000_000.0,
        "reason": "Metadata UNIT conflicts with numeric payload; comparative scale matches MILLION."
    },

    # Single-period files manually resolved from workbook/header evidence
    "CBDK_2024_Q4_FS.xlsx": {
        "final_scale_mode": "PRESENTATION_SCALED",
        "override_multiplier": None,
        "reason": "Workbook evidence supports reported presentation scale."
    },

    "HGII_2025_Q1_FS.xlsx": {
        "final_scale_mode": "PRESENTATION_SCALED",
        "override_multiplier": None,
        "reason": "Workbook evidence supports reported presentation scale."
    },

    "MDIY_2025_Q1_FS.xlsx": {
        "final_scale_mode": "PRESENTATION_SCALED",
        "override_multiplier": None,
        "reason": "Workbook evidence supports reported presentation scale."
    },

    # UNIT multiplier = 1, therefore numeric result is unchanged either way
    "KAQI_2025_Q1_FS.xlsx": {
        "final_scale_mode": "PRESENTATION_SCALED",
        "override_multiplier": None,
        "reason": "UNIT scale; numeric result unchanged."
    },

    "SMAR_2020_Q4_FS.xlsx": {
        "final_scale_mode": "PRESENTATION_SCALED",
        "override_multiplier": None,
        "reason": "UNIT scale; numeric result unchanged."
    },

    "YUPI_2025_Q1_FS.xlsx": {
        "final_scale_mode": "PRESENTATION_SCALED",
        "override_multiplier": None,
        "reason": "UNIT scale; numeric result unchanged."
    },
}


# ---------------------------------------------------------
# APPLY FINAL MODE OVERRIDES
# ---------------------------------------------------------

for source_file, decision in final_scale_exceptions.items():

    mask = (
        final_scale_mode_df["source_file"]
        == source_file
    )

    if mask.any():

        final_scale_mode_df.loc[
            mask,
            "final_scale_mode"
        ] = decision[
            "final_scale_mode"
        ]

    else:

        final_scale_mode_df = pd.concat(
            [
                final_scale_mode_df,
                pd.DataFrame(
                    [
                        {
                            "source_file": source_file,
                            "final_scale_mode":
                                decision["final_scale_mode"]
                        }
                    ]
                )
            ],
            ignore_index=True
        )


# ---------------------------------------------------------
# EXCEPTION TABLE FOR AUDIT / PROVENANCE
# ---------------------------------------------------------

final_scale_exception_df = pd.DataFrame(
    [
        {
            "source_file": source_file,
            "final_scale_mode":
                decision["final_scale_mode"],
            "override_multiplier":
                decision["override_multiplier"],
            "reason":
                decision["reason"]
        }
        for source_file, decision
        in final_scale_exceptions.items()
    ]
)


print("FINAL SCALE MODE COUNTS")

print(
    final_scale_mode_df[
        "final_scale_mode"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nDuplicate source_file rows:",
    final_scale_mode_df[
        "source_file"
    ].duplicated().sum()
)


print("\nFINAL EXCEPTIONS")

display(
    final_scale_exception_df
)

FINAL SCALE MODE COUNTS
final_scale_mode
PRESENTATION_SCALED    14716
FULL_NOMINAL              37
Name: count, dtype: int64

Duplicate source_file rows: 0

FINAL EXCEPTIONS


,source_file,final_scale_mode,override_multiplier,reason
0,DEWA_2023_Q1_FS.xlsx,PRESENTATION_SCALED,NaN,Temporal neighbor recheck supports presentatio...
1,TINS_2024_Q2_FS.xlsx,PRESENTATION_SCALED,NaN,Raw values align with surrounding MILLION-scal...
2,TINS_2025_Q1_FS.xlsx,PRESENTATION_SCALED,1000000.0,Metadata UNIT conflicts with numeric payload; ...
3,CBDK_2024_Q4_FS.xlsx,PRESENTATION_SCALED,NaN,Workbook evidence supports reported presentati...
4,HGII_2025_Q1_FS.xlsx,PRESENTATION_SCALED,NaN,Workbook evidence supports reported presentati...
5,MDIY_2025_Q1_FS.xlsx,PRESENTATION_SCALED,NaN,Workbook evidence supports reported presentati...
6,KAQI_2025_Q1_FS.xlsx,PRESENTATION_SCALED,NaN,UNIT scale; numeric result unchanged.
7,SMAR_2020_Q4_FS.xlsx,PRESENTATION_SCALED,NaN,UNIT scale; numeric result unchanged.
8,YUPI_2025_Q1_FS.xlsx,PRESENTATION_SCALED,NaN,UNIT scale; numeric result unchanged.


In [63]:
# CELL 95 - FINAL NORMALIZATION WITH FILE-SPECIFIC SCALE EXCEPTIONS

final_normalized_df = (
    clean_metrics_with_scale_df
    .merge(
        final_scale_mode_df[
            [
                "source_file",
                "final_scale_mode"
            ]
        ],
        on="source_file",
        how="left",
        validate="many_to_one"
    )
    .merge(
        final_scale_exception_df[
            [
                "source_file",
                "override_multiplier",
                "reason"
            ]
        ],
        on="source_file",
        how="left",
        validate="many_to_one"
    )
)


def calculate_final_value(row):

    if row["currency_scale_status"] != "FOUND":
        return np.nan

    if pd.isna(row["selected_value"]):
        return np.nan


    # Default multiplier from metadata
    effective_multiplier = row["multiplier"]


    # File-specific metadata correction
    if pd.notna(row["override_multiplier"]):
        effective_multiplier = row[
            "override_multiplier"
        ]


    if row["final_scale_mode"] == "PRESENTATION_SCALED":

        if pd.isna(effective_multiplier):
            return np.nan

        return (
            float(row["selected_value"])
            * float(effective_multiplier)
        )


    if row["final_scale_mode"] == "FULL_NOMINAL":

        return float(
            row["selected_value"]
        )


    return np.nan


final_normalized_df[
    "effective_multiplier"
] = np.where(
    final_normalized_df[
        "override_multiplier"
    ].notna(),
    final_normalized_df[
        "override_multiplier"
    ],
    final_normalized_df[
        "multiplier"
    ]
)


final_normalized_df[
    "normalized_value"
] = (
    final_normalized_df.apply(
        calculate_final_value,
        axis=1
    )
)


print(
    "Final rows:",
    len(final_normalized_df)
)

print(
    "Missing final scale mode:",
    final_normalized_df[
        "final_scale_mode"
    ].isna().sum()
)

print(
    "Missing normalized value:",
    final_normalized_df[
        "normalized_value"
    ].isna().sum()
)


print("\nNORMALIZED VALUE SUMMARY")

print(
    final_normalized_df[
        "normalized_value"
    ].describe()
)


print("\nTINS FINAL CHECK")

display(
    final_normalized_df[
        final_normalized_df[
            "ticker"
        ] == "TINS"
    ][
        [
            "year",
            "quarter",
            "metric",
            "selected_value",
            "unit_scale",
            "multiplier",
            "effective_multiplier",
            "final_scale_mode",
            "normalized_value",
            "source_file"
        ]
    ]
    .sort_values(
        [
            "metric",
            "year",
            "quarter"
        ]
    )
)

Final rows: 83442
Missing final scale mode: 4
Missing normalized value: 4

NORMALIZED VALUE SUMMARY
count    8.343800e+04
mean     5.969545e+12
std      6.286791e+13
min     -1.211337e+14
25%      4.249710e+09
50%      1.192769e+11
75%      9.455465e+11
max      2.463659e+15
Name: normalized_value, dtype: float64

TINS FINAL CHECK


,year,quarter,metric,selected_value,unit_scale,multiplier,effective_multiplier,final_scale_mode,normalized_value,source_file
75466,2021,Q1,cash,715102.0,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,7.151020e+11,TINS_2021_Q1_FS.xlsx
75472,2021,Q2,cash,1286157.0,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,1.286157e+12,TINS_2021_Q2_FS.xlsx
75478,2021,Q3,cash,1101560.0,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,1.101560e+12,TINS_2021_Q3_FS.xlsx
75484,2021,Q4,cash,1782262.0,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,1.782262e+12,TINS_2021_Q4_FS.xlsx
75490,2022,Q1,cash,2115098.0,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,2.115098e+12,TINS_2022_Q1_FS.xlsx
...,...,...,...,...,...,...,...,...,...,...
75531,2023,Q3,total_liabilities,6088916.0,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,6.088916e+12,TINS_2023_Q3_FS.xlsx
75537,2023,Q4,total_liabilities,6610928.0,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,6.610928e+12,TINS_2023_Q4_FS.xlsx
75543,2024,Q1,total_liabilities,6455352.0,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,6.455352e+12,TINS_2024_Q1_FS.xlsx
75549,2024,Q2,total_liabilities,6477679.0,MILLION,1000000.0,1000000.0,PRESENTATION_SCALED,6.477679e+12,TINS_2024_Q2_FS.xlsx


In [64]:
# CELL 96 - FINAL NORMALIZATION AUDIT

print("FINAL DATASET AUDIT")
print("=" * 80)

print(
    "Total rows:",
    len(final_normalized_df)
)

print(
    "Unique tickers:",
    final_normalized_df[
        "ticker"
    ].nunique()
)

print(
    "Unique source files:",
    final_normalized_df[
        "source_file"
    ].nunique()
)

print(
    "Missing normalized values:",
    final_normalized_df[
        "normalized_value"
    ].isna().sum()
)

print(
    "Duplicate ticker/year/quarter/metric:",
    final_normalized_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    ].duplicated().sum()
)


print("\nMISSING NORMALIZED ROWS")

display(
    final_normalized_df[
        final_normalized_df[
            "normalized_value"
        ].isna()
    ][
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "currency_scale_status",
            "final_scale_mode",
            "source_file"
        ]
    ]
)


print("\nFINAL SCALE MODE COUNTS")

print(
    final_normalized_df[
        "final_scale_mode"
    ].value_counts(
        dropna=False
    )
)


print("\nNORMALIZED VALUE SUMMARY")

print(
    final_normalized_df[
        "normalized_value"
    ].describe()
)

FINAL DATASET AUDIT
Total rows: 83442
Unique tickers: 924
Unique source files: 14755
Missing normalized values: 4
Duplicate ticker/year/quarter/metric: 0

MISSING NORMALIZED ROWS


,ticker,year,quarter,metric,selected_value,currency,unit_scale,multiplier,currency_scale_status,final_scale_mode,source_file
20265,CTRA,2023,Q2,operating_cash_flow,1.453784e+06,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND,NaN,CTRA_2023_Q2_FS.xlsx
74674,TEBE,2023,Q3,cash,3.876731e+08,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND,NaN,TEBE_2023_Q3_FS.xlsx
74675,TEBE,2023,Q3,total_assets,1.184340e+09,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND,NaN,TEBE_2023_Q3_FS.xlsx
74676,TEBE,2023,Q3,total_liabilities,1.137427e+08,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND,NaN,TEBE_2023_Q3_FS.xlsx



FINAL SCALE MODE COUNTS
final_scale_mode
PRESENTATION_SCALED    83275
FULL_NOMINAL             163
NaN                        4
Name: count, dtype: int64

NORMALIZED VALUE SUMMARY
count    8.343800e+04
mean     5.969545e+12
std      6.286791e+13
min     -1.211337e+14
25%      4.249710e+09
50%      1.192769e+11
75%      9.455465e+11
max      2.463659e+15
Name: normalized_value, dtype: float64


In [65]:
# CELL 97 - SAVE FINAL NORMALIZED DATASET

FINAL_NORMALIZED_FILE = Path(
    "data/idx_financial_metrics_normalized_final.csv"
)

FINAL_SCALE_MODE_FILE = Path(
    "data/idx_financial_scale_mode_final.csv"
)

FINAL_SCALE_EXCEPTION_FILE = Path(
    "data/idx_financial_scale_exceptions.csv"
)


final_normalized_df.to_csv(
    FINAL_NORMALIZED_FILE,
    index=False
)

final_scale_mode_df.to_csv(
    FINAL_SCALE_MODE_FILE,
    index=False
)

final_scale_exception_df.to_csv(
    FINAL_SCALE_EXCEPTION_FILE,
    index=False
)


print(
    "Saved final normalized dataset:",
    FINAL_NORMALIZED_FILE
)

print(
    "Saved final scale mode mapping:",
    FINAL_SCALE_MODE_FILE
)

print(
    "Saved scale exceptions:",
    FINAL_SCALE_EXCEPTION_FILE
)

print(
    "\nFinal normalized rows:",
    len(final_normalized_df)
)

print(
    "Normalized values available:",
    final_normalized_df[
        "normalized_value"
    ].notna().sum()
)

print(
    "Normalized values missing:",
    final_normalized_df[
        "normalized_value"
    ].isna().sum()
)

Saved final normalized dataset: data\idx_financial_metrics_normalized_final.csv
Saved final scale mode mapping: data\idx_financial_scale_mode_final.csv
Saved scale exceptions: data\idx_financial_scale_exceptions.csv

Final normalized rows: 83442
Normalized values available: 83438
Normalized values missing: 4


In [39]:
print("\nKNOWN CASE SANITY CHECK")

known_files = [
    "AALI_2020_Q1_FS.xlsx",
    "AALI_2023_Q4_FS.xlsx",
    "AALI_2024_Q2_FS.xlsx",
    "ASII_2020_Q1_FS.xlsx",
    "ASII_2023_Q4_FS.xlsx",
    "BMRI_2023_Q4_FS.xlsx",
    "TINS_2024_Q2_FS.xlsx",
    "DEWA_2023_Q1_FS.xlsx"
]


display(
    clean_metrics_normalized_df[
        clean_metrics_normalized_df[
            "source_file"
        ].isin(
            known_files
        )
    ][
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "final_scale_mode",
            "normalized_value",
            "source_file"
        ]
    ]
    .sort_values(
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    )
)


KNOWN CASE SANITY CHECK


,ticker,year,quarter,metric,selected_value,currency,unit_scale,multiplier,final_scale_mode,normalized_value,source_file
12,AALI,2020,Q1,cash,1.470866e+06,IDR,MILLION,1.000000e+06,PRESENTATION_SCALED,1.470866e+12,AALI_2020_Q1_FS.xlsx
13,AALI,2020,Q1,gross_profit,9.267910e+05,IDR,MILLION,1.000000e+06,PRESENTATION_SCALED,9.267910e+11,AALI_2020_Q1_FS.xlsx
14,AALI,2020,Q1,operating_cash_flow,6.606100e+05,IDR,MILLION,1.000000e+06,PRESENTATION_SCALED,6.606100e+11,AALI_2020_Q1_FS.xlsx
15,AALI,2020,Q1,revenue,4.796084e+06,IDR,MILLION,1.000000e+06,PRESENTATION_SCALED,4.796084e+12,AALI_2020_Q1_FS.xlsx
16,AALI,2020,Q1,total_assets,2.921860e+07,IDR,MILLION,1.000000e+06,PRESENTATION_SCALED,2.921860e+13,AALI_2020_Q1_FS.xlsx
17,AALI,2020,Q1,total_liabilities,9.854949e+06,IDR,MILLION,1.000000e+06,PRESENTATION_SCALED,9.854949e+12,AALI_2020_Q1_FS.xlsx
96,AALI,2023,Q4,cash,2.089508e+12,IDR,MILLION,1.000000e+06,FULL_NOMINAL,2.089508e+12,AALI_2023_Q4_FS.xlsx
97,AALI,2023,Q4,gross_profit,2.770980e+12,IDR,MILLION,1.000000e+06,FULL_NOMINAL,2.770980e+12,AALI_2023_Q4_FS.xlsx
98,AALI,2023,Q4,operating_cash_flow,2.538738e+12,IDR,MILLION,1.000000e+06,FULL_NOMINAL,2.538738e+12,AALI_2023_Q4_FS.xlsx
99,AALI,2023,Q4,revenue,2.074547e+13,IDR,MILLION,1.000000e+06,FULL_NOMINAL,2.074547e+13,AALI_2023_Q4_FS.xlsx


In [40]:
# CELL 80 - AUDIT THE 40 ROWS WITHOUT FINAL SCALE MODE

missing_scale_mode_rows_df = (
    clean_metrics_normalized_df[
        clean_metrics_normalized_df[
            "final_scale_mode"
        ].isna()
    ]
    .copy()
)


print(
    "Missing scale-mode metric rows:",
    len(missing_scale_mode_rows_df)
)

print(
    "Missing scale-mode source files:",
    missing_scale_mode_rows_df[
        "source_file"
    ].nunique()
)


print("\nCURRENCY / SCALE STATUS")

print(
    missing_scale_mode_rows_df[
        "currency_scale_status"
    ].value_counts(
        dropna=False
    )
)


print("\nSOURCE FILES")

display(
    missing_scale_mode_rows_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "currency_scale_status",
            "source_file"
        ]
    ]
    .sort_values(
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    )
)

Missing scale-mode metric rows: 40
Missing scale-mode source files: 8

CURRENCY / SCALE STATUS
currency_scale_status
FOUND                           36
CURRENCY_AND_SCALE_NOT_FOUND     4
Name: count, dtype: int64

SOURCE FILES


,ticker,year,quarter,metric,selected_value,currency,unit_scale,multiplier,currency_scale_status,source_file
16815,CBDK,2024,Q4,cash,3.457910e+09,IDR,THOUSAND,1000.0,FOUND,CBDK_2024_Q4_FS.xlsx
16816,CBDK,2024,Q4,gross_profit,1.272365e+09,IDR,THOUSAND,1000.0,FOUND,CBDK_2024_Q4_FS.xlsx
16817,CBDK,2024,Q4,operating_cash_flow,1.841415e+09,IDR,THOUSAND,1000.0,FOUND,CBDK_2024_Q4_FS.xlsx
16818,CBDK,2024,Q4,revenue,2.248978e+09,IDR,THOUSAND,1000.0,FOUND,CBDK_2024_Q4_FS.xlsx
16819,CBDK,2024,Q4,total_assets,1.908160e+10,IDR,THOUSAND,1000.0,FOUND,CBDK_2024_Q4_FS.xlsx
16820,CBDK,2024,Q4,total_liabilities,1.075256e+10,IDR,THOUSAND,1000.0,FOUND,CBDK_2024_Q4_FS.xlsx
20265,CTRA,2023,Q2,operating_cash_flow,1.453784e+06,NaN,NaN,NaN,CURRENCY_AND_SCALE_NOT_FOUND,CTRA_2023_Q2_FS.xlsx
32162,HGII,2025,Q1,cash,2.742347e+08,IDR,THOUSAND,1000.0,FOUND,HGII_2025_Q1_FS.xlsx
32163,HGII,2025,Q1,gross_profit,1.735247e+07,IDR,THOUSAND,1000.0,FOUND,HGII_2025_Q1_FS.xlsx
32164,HGII,2025,Q1,operating_cash_flow,1.337655e+07,IDR,THOUSAND,1000.0,FOUND,HGII_2025_Q1_FS.xlsx


In [41]:
# CELL 81 - RECHECK DEWA 2023 Q1 SCALE MODE

DEWA_TARGET_FILE = "DEWA_2023_Q1_FS.xlsx"


dewa_compare_rows = []


for metric in (
    clean_metrics_with_scale_df[
        clean_metrics_with_scale_df[
            "source_file"
        ] == DEWA_TARGET_FILE
    ]["metric"].unique()
):

    history = (
        clean_metrics_with_scale_df[
            (
                clean_metrics_with_scale_df[
                    "ticker"
                ] == "DEWA"
            )
            &
            (
                clean_metrics_with_scale_df[
                    "metric"
                ] == metric
            )
        ]
        .merge(
            final_scale_mode_df,
            on="source_file",
            how="left"
        )
        .copy()
    )

    history[
        "quarter_order"
    ] = history[
        "quarter"
    ].map(
        {
            "Q1": 1,
            "Q2": 2,
            "Q3": 3,
            "Q4": 4
        }
    )

    history = (
        history
        .sort_values(
            [
                "year",
                "quarter_order"
            ]
        )
        .reset_index(drop=True)
    )


    target_matches = history[
        history[
            "source_file"
        ] == DEWA_TARGET_FILE
    ].index

    if len(target_matches) == 0:
        continue

    target_idx = target_matches[0]

    target = history.iloc[
        target_idx
    ]


    target_raw = float(
        target[
            "selected_value"
        ]
    )

    target_multiplier = float(
        target[
            "multiplier"
        ]
    )


    target_if_full = (
        target_raw
    )

    target_if_presentation = (
        target_raw
        *
        target_multiplier
    )


    for neighbor_position in [
        target_idx - 1,
        target_idx + 1,
        target_idx + 2
    ]:

        if (
            neighbor_position < 0
            or neighbor_position >= len(history)
        ):
            continue

        neighbor = history.iloc[
            neighbor_position
        ]

        neighbor_mode = neighbor[
            "final_scale_mode"
        ]

        if neighbor_mode == "PRESENTATION_SCALED":

            neighbor_normalized = (
                float(
                    neighbor[
                        "selected_value"
                    ]
                )
                *
                float(
                    neighbor[
                        "multiplier"
                    ]
                )
            )

        elif neighbor_mode == "FULL_NOMINAL":

            neighbor_normalized = float(
                neighbor[
                    "selected_value"
                ]
            )

        else:
            continue


        full_distance = abs(
            np.log10(
                abs(
                    target_if_full
                    /
                    neighbor_normalized
                )
            )
        )

        presentation_distance = abs(
            np.log10(
                abs(
                    target_if_presentation
                    /
                    neighbor_normalized
                )
            )
        )


        dewa_compare_rows.append(
            {
                "metric": metric,

                "neighbor_file":
                    neighbor[
                        "source_file"
                    ],

                "neighbor_mode":
                    neighbor_mode,

                "neighbor_normalized":
                    neighbor_normalized,

                "target_if_full":
                    target_if_full,

                "target_if_presentation":
                    target_if_presentation,

                "full_distance":
                    full_distance,

                "presentation_distance":
                    presentation_distance,

                "better_target_mode":
                    (
                        "FULL_NOMINAL"
                        if full_distance
                        <
                        presentation_distance
                        else
                        "PRESENTATION_SCALED"
                    )
            }
        )


dewa_recheck_df = pd.DataFrame(
    dewa_compare_rows
)


print("DEWA RECHECK VOTES")

print(
    dewa_recheck_df[
        "better_target_mode"
    ].value_counts()
)


display(
    dewa_recheck_df[
        [
            "metric",
            "neighbor_file",
            "neighbor_mode",
            "neighbor_normalized",
            "target_if_full",
            "target_if_presentation",
            "full_distance",
            "presentation_distance",
            "better_target_mode"
        ]
    ]
)

DEWA RECHECK VOTES
better_target_mode
PRESENTATION_SCALED    12
FULL_NOMINAL            6
Name: count, dtype: int64


,metric,neighbor_file,neighbor_mode,neighbor_normalized,target_if_full,target_if_presentation,full_distance,presentation_distance,better_target_mode
0,cash,DEWA_2022_Q4_FS.xlsx,PRESENTATION_SCALED,1.931872e+07,3.540602e+08,3.540602e+11,1.263099,4.263099,FULL_NOMINAL
1,cash,DEWA_2023_Q2_FS.xlsx,PRESENTATION_SCALED,3.720212e+11,3.540602e+08,3.540602e+11,3.021491,0.021491,PRESENTATION_SCALED
2,cash,DEWA_2023_Q3_FS.xlsx,PRESENTATION_SCALED,4.404680e+11,3.540602e+08,3.540602e+11,3.094837,0.094837,PRESENTATION_SCALED
3,gross_profit,DEWA_2022_Q4_FS.xlsx,PRESENTATION_SCALED,5.582140e+05,1.233788e+08,1.233788e+11,2.344440,5.344440,FULL_NOMINAL
4,gross_profit,DEWA_2023_Q2_FS.xlsx,PRESENTATION_SCALED,2.566706e+11,1.233788e+08,1.233788e+11,3.318135,0.318135,PRESENTATION_SCALED
5,gross_profit,DEWA_2023_Q3_FS.xlsx,PRESENTATION_SCALED,3.153330e+11,1.233788e+08,1.233788e+11,3.407529,0.407529,PRESENTATION_SCALED
6,operating_cash_flow,DEWA_2022_Q4_FS.xlsx,PRESENTATION_SCALED,1.424606e+07,4.330034e+08,4.330034e+11,1.482797,4.482797,FULL_NOMINAL
7,operating_cash_flow,DEWA_2023_Q2_FS.xlsx,PRESENTATION_SCALED,5.818073e+11,4.330034e+08,4.330034e+11,3.128288,0.128288,PRESENTATION_SCALED
8,operating_cash_flow,DEWA_2023_Q3_FS.xlsx,PRESENTATION_SCALED,9.307875e+11,4.330034e+08,4.330034e+11,3.332359,0.332359,PRESENTATION_SCALED
9,revenue,DEWA_2022_Q4_FS.xlsx,PRESENTATION_SCALED,4.068246e+08,1.761394e+09,1.761394e+12,0.636449,3.636449,FULL_NOMINAL


In [42]:
# CELL 82 - AUDIT FOUND-SCALE FILES WITHOUT FINAL SCALE MODE

unclassified_found_rows_df = (
    missing_scale_mode_rows_df[
        missing_scale_mode_rows_df[
            "currency_scale_status"
        ] == "FOUND"
    ]
    .copy()
)


unclassified_found_files_df = (
    unclassified_found_rows_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "currency",
            "unit_scale",
            "multiplier"
        ]
    ]
    .drop_duplicates(
        subset=["source_file"]
    )
    .sort_values(
        [
            "ticker",
            "year",
            "quarter"
        ]
    )
    .reset_index(drop=True)
)


print(
    "FOUND-scale metric rows without mode:",
    len(unclassified_found_rows_df)
)

print(
    "FOUND-scale source files without mode:",
    len(unclassified_found_files_df)
)


display(
    unclassified_found_files_df
)

FOUND-scale metric rows without mode: 36
FOUND-scale source files without mode: 6


,ticker,year,quarter,source_file,currency,unit_scale,multiplier
0,CBDK,2024,Q4,CBDK_2024_Q4_FS.xlsx,IDR,THOUSAND,1000.0
1,HGII,2025,Q1,HGII_2025_Q1_FS.xlsx,IDR,THOUSAND,1000.0
2,KAQI,2025,Q1,KAQI_2025_Q1_FS.xlsx,IDR,UNIT,1.0
3,MDIY,2025,Q1,MDIY_2025_Q1_FS.xlsx,IDR,MILLION,1000000.0
4,SMAR,2020,Q4,SMAR_2020_Q4_FS.xlsx,IDR,UNIT,1.0
5,YUPI,2025,Q1,YUPI_2025_Q1_FS.xlsx,IDR,UNIT,1.0


In [48]:
# CELL 83 - AUDIT REMAINING NON-UNIT UNCLASSIFIED FILES

remaining_nonunit_files = [
    "CBDK_2024_Q4_FS.xlsx",
    "HGII_2025_Q1_FS.xlsx",
    "MDIY_2025_Q1_FS.xlsx",
]


remaining_nonunit_rows_df = (
    clean_scale_mode_df[
        clean_scale_mode_df[
            "source_file"
        ].isin(
            remaining_nonunit_files
        )
    ]
    .copy()
)


remaining_neighbor_rows = []


for _, row in remaining_nonunit_rows_df.iterrows():

    ticker = row["ticker"]
    metric = row["metric"]
    target_file = row["source_file"]

    history = (
        clean_scale_mode_df[
            (clean_scale_mode_df["ticker"] == ticker)
            &
            (clean_scale_mode_df["metric"] == metric)
        ]
        .sort_values(
            [
                "year",
                "quarter_order"
            ]
        )
        .reset_index(drop=True)
    )


    target_idx_list = history[
        history["source_file"] == target_file
    ].index

    if len(target_idx_list) == 0:
        continue

    target_idx = target_idx_list[0]


    for idx in range(
        max(0, target_idx - 2),
        min(len(history), target_idx + 3)
    ):

        hist_row = history.iloc[idx]

        file_mode = known_file_mode_map.get(
            hist_row["source_file"]
        )

        if file_mode == "PRESENTATION_SCALED":

            normalized_if_known = (
                hist_row["selected_value"]
                *
                hist_row["multiplier"]
            )

        elif file_mode == "FULL_NOMINAL":

            normalized_if_known = (
                hist_row["selected_value"]
            )

        else:

            normalized_if_known = np.nan


        remaining_neighbor_rows.append(
            {
                "ticker": ticker,
                "metric": metric,
                "target_source_file": target_file,

                "source_file":
                    hist_row["source_file"],

                "year":
                    hist_row["year"],

                "quarter":
                    hist_row["quarter"],

                "selected_value":
                    hist_row["selected_value"],

                "unit_scale":
                    hist_row["unit_scale"],

                "multiplier":
                    hist_row["multiplier"],

                "file_scale_mode":
                    file_mode,

                "normalized_if_known":
                    normalized_if_known,

                "is_target":
                    hist_row["source_file"]
                    == target_file
            }
        )


remaining_neighbor_audit_df = pd.DataFrame(
    remaining_neighbor_rows
)


display(
    remaining_neighbor_audit_df[
        [
            "ticker",
            "metric",
            "year",
            "quarter",
            "source_file",
            "selected_value",
            "unit_scale",
            "multiplier",
            "file_scale_mode",
            "normalized_if_known",
            "is_target"
        ]
    ]
    .sort_values(
        [
            "ticker",
            "metric",
            "year",
            "quarter"
        ]
    )
)

,ticker,metric,year,quarter,source_file,selected_value,unit_scale,multiplier,file_scale_mode,normalized_if_known,is_target
0,CBDK,cash,2024,Q4,CBDK_2024_Q4_FS.xlsx,3.457910e+09,THOUSAND,1000.0,None,NaN,True
1,CBDK,gross_profit,2024,Q4,CBDK_2024_Q4_FS.xlsx,1.272365e+09,THOUSAND,1000.0,None,NaN,True
2,CBDK,operating_cash_flow,2024,Q4,CBDK_2024_Q4_FS.xlsx,1.841415e+09,THOUSAND,1000.0,None,NaN,True
3,CBDK,revenue,2024,Q4,CBDK_2024_Q4_FS.xlsx,2.248978e+09,THOUSAND,1000.0,None,NaN,True
4,CBDK,total_assets,2024,Q4,CBDK_2024_Q4_FS.xlsx,1.908160e+10,THOUSAND,1000.0,None,NaN,True
5,CBDK,total_liabilities,2024,Q4,CBDK_2024_Q4_FS.xlsx,1.075256e+10,THOUSAND,1000.0,None,NaN,True
6,HGII,cash,2025,Q1,HGII_2025_Q1_FS.xlsx,2.742347e+08,THOUSAND,1000.0,None,NaN,True
7,HGII,gross_profit,2025,Q1,HGII_2025_Q1_FS.xlsx,1.735247e+07,THOUSAND,1000.0,None,NaN,True
8,HGII,operating_cash_flow,2025,Q1,HGII_2025_Q1_FS.xlsx,1.337655e+07,THOUSAND,1000.0,None,NaN,True
9,HGII,revenue,2025,Q1,HGII_2025_Q1_FS.xlsx,2.080590e+07,THOUSAND,1000.0,None,NaN,True


In [49]:
# CELL 86 - INSPECT RAW WORKBOOK CONTEXT FOR 3 SINGLE-PERIOD FILES
# FIXED VERSION - DOES NOT REQUIRE source_row/source_column

from openpyxl import load_workbook


single_period_files = [
    "CBDK_2024_Q4_FS.xlsx",
    "HGII_2025_Q1_FS.xlsx",
    "MDIY_2025_Q1_FS.xlsx",
]


single_period_source_df = (
    clean_metrics_with_scale_df[
        clean_metrics_with_scale_df[
            "source_file"
        ].isin(single_period_files)
    ]
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "selected_value",
            "source_file",
            "source_path",
            "source_sheet",
            "source_label",
            "currency",
            "unit_scale",
            "multiplier"
        ]
    ]
    .copy()
)


raw_check_rows = []


for _, row in single_period_source_df.iterrows():

    wb = None

    try:
        wb = load_workbook(
            row["source_path"],
            read_only=True,
            data_only=True
        )

        if row["source_sheet"] not in wb.sheetnames:
            raw_check_rows.append(
                {
                    "ticker": row["ticker"],
                    "metric": row["metric"],
                    "source_file": row["source_file"],
                    "source_sheet": row["source_sheet"],
                    "source_label": row["source_label"],
                    "selected_value": row["selected_value"],
                    "workbook_matches": [],
                    "currency": row["currency"],
                    "unit_scale": row["unit_scale"],
                    "multiplier": row["multiplier"],
                    "status": "SHEET_NOT_FOUND"
                }
            )
            continue


        ws = wb[row["source_sheet"]]

        label_normalized = (
            str(row["source_label"])
            .strip()
            .lower()
        )

        matches = []

        # Search label text in first 250 rows / 20 columns
        for sheet_row in ws.iter_rows(
            min_row=1,
            max_row=min(ws.max_row, 250),
            min_col=1,
            max_col=min(ws.max_column, 20),
            values_only=False
        ):

            for cell in sheet_row:

                if cell.value is None:
                    continue

                cell_text = (
                    str(cell.value)
                    .strip()
                    .lower()
                )

                if cell_text == label_normalized:

                    # collect nearby values on the same row
                    nearby_values = []

                    start_col = max(
                        1,
                        cell.column
                    )

                    end_col = min(
                        ws.max_column,
                        cell.column + 6
                    )

                    for col_idx in range(
                        start_col,
                        end_col + 1
                    ):

                        nearby_cell = ws.cell(
                            row=cell.row,
                            column=col_idx
                        )

                        nearby_values.append(
                            {
                                "coordinate":
                                    nearby_cell.coordinate,

                                "value":
                                    nearby_cell.value
                            }
                        )

                    matches.append(
                        {
                            "label_coordinate":
                                cell.coordinate,

                            "nearby_values":
                                nearby_values
                        }
                    )


        raw_check_rows.append(
            {
                "ticker": row["ticker"],
                "metric": row["metric"],
                "source_file": row["source_file"],
                "source_sheet": row["source_sheet"],
                "source_label": row["source_label"],
                "selected_value": row["selected_value"],
                "workbook_matches": matches,
                "currency": row["currency"],
                "unit_scale": row["unit_scale"],
                "multiplier": row["multiplier"],
                "status":
                    "FOUND_LABEL"
                    if matches
                    else "LABEL_NOT_FOUND"
            }
        )


    except Exception as e:

        raw_check_rows.append(
            {
                "ticker": row["ticker"],
                "metric": row["metric"],
                "source_file": row["source_file"],
                "source_sheet": row["source_sheet"],
                "source_label": row["source_label"],
                "selected_value": row["selected_value"],
                "workbook_matches": [],
                "currency": row["currency"],
                "unit_scale": row["unit_scale"],
                "multiplier": row["multiplier"],
                "status":
                    f"READ_ERROR:{type(e).__name__}"
            }
        )


    finally:

        if wb is not None:
            wb.close()


single_period_raw_check_df = pd.DataFrame(
    raw_check_rows
)


print("STATUS")

print(
    single_period_raw_check_df[
        "status"
    ].value_counts(
        dropna=False
    )
)


display(
    single_period_raw_check_df[
        [
            "ticker",
            "metric",
            "source_file",
            "source_sheet",
            "source_label",
            "selected_value",
            "currency",
            "unit_scale",
            "multiplier",
            "status",
            "workbook_matches"
        ]
    ]
    .sort_values(
        [
            "ticker",
            "metric"
        ]
    )
)

STATUS
status
FOUND_LABEL    18
Name: count, dtype: int64


,ticker,metric,source_file,source_sheet,source_label,selected_value,currency,unit_scale,multiplier,status,workbook_matches
0,CBDK,cash,CBDK_2024_Q4_FS.xlsx,2210000,Kas dan setara kas,3.457910e+09,IDR,THOUSAND,1000.0,FOUND_LABEL,"[{'label_coordinate': 'A8', 'nearby_values': [..."
1,CBDK,gross_profit,CBDK_2024_Q4_FS.xlsx,2311000,Jumlah laba bruto,1.272365e+09,IDR,THOUSAND,1000.0,FOUND_LABEL,"[{'label_coordinate': 'A8', 'nearby_values': [..."
2,CBDK,operating_cash_flow,CBDK_2024_Q4_FS.xlsx,2510000,Total net cash flows received from (used in) o...,1.841415e+09,IDR,THOUSAND,1000.0,FOUND_LABEL,"[{'label_coordinate': 'D30', 'nearby_values': ..."
3,CBDK,revenue,CBDK_2024_Q4_FS.xlsx,2311000,Sales and revenue,2.248978e+09,IDR,THOUSAND,1000.0,FOUND_LABEL,"[{'label_coordinate': 'D6', 'nearby_values': [..."
4,CBDK,total_assets,CBDK_2024_Q4_FS.xlsx,2210000,Jumlah aset,1.908160e+10,IDR,THOUSAND,1000.0,FOUND_LABEL,"[{'label_coordinate': 'A90', 'nearby_values': ..."
5,CBDK,total_liabilities,CBDK_2024_Q4_FS.xlsx,2210000,Jumlah liabilitas,1.075256e+10,IDR,THOUSAND,1000.0,FOUND_LABEL,"[{'label_coordinate': 'A179', 'nearby_values':..."
6,HGII,cash,HGII_2025_Q1_FS.xlsx,3210000,Kas dan setara kas,2.742347e+08,IDR,THOUSAND,1000.0,FOUND_LABEL,"[{'label_coordinate': 'A8', 'nearby_values': [..."
7,HGII,gross_profit,HGII_2025_Q1_FS.xlsx,3311000,Jumlah laba bruto,1.735247e+07,IDR,THOUSAND,1000.0,FOUND_LABEL,"[{'label_coordinate': 'A8', 'nearby_values': [..."
8,HGII,operating_cash_flow,HGII_2025_Q1_FS.xlsx,3510000,Total net cash flows received from (used in) o...,1.337655e+07,IDR,THOUSAND,1000.0,FOUND_LABEL,"[{'label_coordinate': 'D32', 'nearby_values': ..."
9,HGII,revenue,HGII_2025_Q1_FS.xlsx,3311000,Sales and revenue,2.080590e+07,IDR,THOUSAND,1000.0,FOUND_LABEL,"[{'label_coordinate': 'D6', 'nearby_values': [..."


In [51]:
# CELL 87 - INSPECT STATEMENT HEADER / SCALE TEXT FOR 3 UNRESOLVED FILES

from openpyxl import load_workbook


unresolved_single_period_files = [
    "CBDK_2024_Q4_FS.xlsx",
    "HGII_2025_Q1_FS.xlsx",
    "MDIY_2025_Q1_FS.xlsx",
]


header_audit_rows = []


for source_file in unresolved_single_period_files:

    source_match = (
        clean_metrics_with_scale_df[
            clean_metrics_with_scale_df[
                "source_file"
            ] == source_file
        ]
        .iloc[0]
    )

    source_path = source_match[
        "source_path"
    ]

    wb = None

    try:
        wb = load_workbook(
            source_path,
            read_only=True,
            data_only=True
        )

        relevant_sheets = (
            clean_metrics_with_scale_df[
                clean_metrics_with_scale_df[
                    "source_file"
                ] == source_file
            ]["source_sheet"]
            .dropna()
            .astype(str)
            .unique()
        )


        for sheet_name in relevant_sheets:

            if sheet_name not in wb.sheetnames:
                continue

            ws = wb[
                sheet_name
            ]

            header_values = []

            for row in ws.iter_rows(
                min_row=1,
                max_row=min(
                    ws.max_row,
                    15
                ),
                min_col=1,
                max_col=min(
                    ws.max_column,
                    12
                ),
                values_only=True
            ):

                for value in row:

                    if value is not None:

                        text_value = (
                            str(value)
                            .strip()
                        )

                        if text_value:
                            header_values.append(
                                text_value
                            )


            header_text = " | ".join(
                header_values
            )


            header_audit_rows.append(
                {
                    "source_file":
                        source_file,

                    "ticker":
                        source_match[
                            "ticker"
                        ],

                    "source_sheet":
                        sheet_name,

                    "currency":
                        source_match[
                            "currency"
                        ],

                    "unit_scale":
                        source_match[
                            "unit_scale"
                        ],

                    "multiplier":
                        source_match[
                            "multiplier"
                        ],

                    "header_text":
                        header_text
                }
            )


    except Exception as e:

        header_audit_rows.append(
            {
                "source_file":
                    source_file,

                "ticker":
                    source_match[
                        "ticker"
                    ],

                "source_sheet":
                    None,

                "currency":
                    source_match[
                        "currency"
                    ],

                "unit_scale":
                    source_match[
                        "unit_scale"
                    ],

                "multiplier":
                    source_match[
                        "multiplier"
                    ],

                "header_text":
                    f"READ_ERROR: {e}"
            }
        )


    finally:

        if wb is not None:
            wb.close()


single_period_header_audit_df = (
    pd.DataFrame(
        header_audit_rows
    )
)


for _, row in (
    single_period_header_audit_df
    .sort_values(
        [
            "source_file",
            "source_sheet"
        ]
    )
    .iterrows()
):

    print("\n" + "=" * 120)

    print(
        "FILE:",
        row[
            "source_file"
        ]
    )

    print(
        "SHEET:",
        row[
            "source_sheet"
        ]
    )

    print(
        "METADATA SCALE:",
        row[
            "unit_scale"
        ],
        "| multiplier:",
        row[
            "multiplier"
        ]
    )

    print("\nHEADER TEXT:")

    print(
        row[
            "header_text"
        ][:2500]
    )


FILE: CBDK_2024_Q4_FS.xlsx
SHEET: 2210000
METADATA SCALE: THOUSAND | multiplier: 1000.0

HEADER TEXT:
[2210000] Statement of financial position presented using current and non-current - Property Industry | Laporan posisi keuangan | Statement of financial position | CurrentYearInstant | PriorEndYearInstant | Laporan posisi keuangan | Statement of financial position | Aset | Assets | Aset lancar | Current assets | Kas dan setara kas | 3457909945 | 287859225 | Cash and cash equivalents | Wesel tagih | Notes receivable | Investasi jangka pendek | Short-term investments | Dana yang dibatasi penggunaannya lancar | Current restricted funds | Aset keuangan lancar | Current financial assets | Aset keuangan lancar yang diukur pada nilai wajar melalui laba rugi | Current financial assets at fair value through profit or loss | Aset keuangan lancar nilai wajar melalui pendapatan komprehensif lainnya | Current financial assets fair value through other comprehensive income | Aset keuangan biaya pero

In [54]:
# CELL 88 - ADD FINAL SCALE MODE FOR PREVIOUSLY UNCLASSIFIED FILES

manual_final_scale_modes = {
    "CBDK_2024_Q4_FS.xlsx": "PRESENTATION_SCALED",
    "HGII_2025_Q1_FS.xlsx": "PRESENTATION_SCALED",
    "MDIY_2025_Q1_FS.xlsx": "PRESENTATION_SCALED",

    # UNIT multiplier = 1, so presentation/full gives same numeric result
    "KAQI_2025_Q1_FS.xlsx": "PRESENTATION_SCALED",
    "SMAR_2020_Q4_FS.xlsx": "PRESENTATION_SCALED",
    "YUPI_2025_Q1_FS.xlsx": "PRESENTATION_SCALED",
}


manual_scale_mode_df = pd.DataFrame(
    [
        {
            "source_file": source_file,
            "final_scale_mode": scale_mode
        }
        for source_file, scale_mode
        in manual_final_scale_modes.items()
    ]
)


# Remove them first if somehow already present
final_scale_mode_df = (
    final_scale_mode_df[
        ~final_scale_mode_df[
            "source_file"
        ].isin(
            manual_final_scale_modes.keys()
        )
    ]
    .copy()
)


# Add resolved files
final_scale_mode_df = pd.concat(
    [
        final_scale_mode_df,
        manual_scale_mode_df
    ],
    ignore_index=True
)


print("UPDATED FINAL SCALE MODE COUNTS")

print(
    final_scale_mode_df[
        "final_scale_mode"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nTotal classified source files:",
    final_scale_mode_df[
        "source_file"
    ].nunique()
)


print(
    "Duplicate source_file rows:",
    final_scale_mode_df[
        "source_file"
    ].duplicated().sum()
)

UPDATED FINAL SCALE MODE COUNTS
final_scale_mode
PRESENTATION_SCALED    14715
FULL_NOMINAL              38
Name: count, dtype: int64

Total classified source files: 14753
Duplicate source_file rows: 0


In [52]:
# CELL 84 - SCORE REMAINING NON-UNIT FILES USING KNOWN NEIGHBORS

remaining_resolution_rows = []


for target_file in remaining_nonunit_files:

    target_rows = (
        remaining_neighbor_audit_df[
            (
                remaining_neighbor_audit_df[
                    "target_source_file"
                ] == target_file
            )
            &
            (
                remaining_neighbor_audit_df[
                    "is_target"
                ]
            )
        ]
    )


    for _, target_row in target_rows.iterrows():

        metric = target_row["metric"]

        target_raw = float(
            target_row["selected_value"]
        )

        target_multiplier = float(
            target_row["multiplier"]
        )


        target_if_full = target_raw

        target_if_presentation = (
            target_raw
            *
            target_multiplier
        )


        neighbors = (
            remaining_neighbor_audit_df[
                (
                    remaining_neighbor_audit_df[
                        "target_source_file"
                    ] == target_file
                )
                &
                (
                    remaining_neighbor_audit_df[
                        "metric"
                    ] == metric
                )
                &
                (
                    ~remaining_neighbor_audit_df[
                        "is_target"
                    ]
                )
                &
                (
                    remaining_neighbor_audit_df[
                        "normalized_if_known"
                    ].notna()
                )
            ]
        )


        if len(neighbors) == 0:

            remaining_resolution_rows.append(
                {
                    "target_source_file":
                        target_file,

                    "metric":
                        metric,

                    "known_neighbors":
                        0,

                    "presentation_score":
                        np.nan,

                    "full_score":
                        np.nan,

                    "preferred_mode":
                        "NO_EVIDENCE"
                }
            )

            continue


        presentation_distances = []
        full_distances = []


        for _, neighbor in neighbors.iterrows():

            neighbor_value = float(
                neighbor[
                    "normalized_if_known"
                ]
            )


            if (
                target_if_presentation != 0
                and neighbor_value != 0
            ):

                presentation_distances.append(
                    abs(
                        np.log10(
                            abs(
                                target_if_presentation
                                /
                                neighbor_value
                            )
                        )
                    )
                )


            if (
                target_if_full != 0
                and neighbor_value != 0
            ):

                full_distances.append(
                    abs(
                        np.log10(
                            abs(
                                target_if_full
                                /
                                neighbor_value
                            )
                        )
                    )
                )


        presentation_score = (
            np.nanmean(
                presentation_distances
            )
            if presentation_distances
            else np.nan
        )

        full_score = (
            np.nanmean(
                full_distances
            )
            if full_distances
            else np.nan
        )


        if (
            pd.isna(presentation_score)
            or pd.isna(full_score)
        ):

            preferred_mode = "NO_EVIDENCE"

        elif presentation_score < full_score:

            preferred_mode = (
                "PRESENTATION_SCALED"
            )

        elif full_score < presentation_score:

            preferred_mode = (
                "FULL_NOMINAL"
            )

        else:

            preferred_mode = "TIE"


        remaining_resolution_rows.append(
            {
                "target_source_file":
                    target_file,

                "metric":
                    metric,

                "known_neighbors":
                    len(neighbors),

                "presentation_score":
                    presentation_score,

                "full_score":
                    full_score,

                "preferred_mode":
                    preferred_mode
            }
        )


remaining_resolution_metric_df = pd.DataFrame(
    remaining_resolution_rows
)


print("METRIC-LEVEL RESULT")

print(
    remaining_resolution_metric_df[
        "preferred_mode"
    ].value_counts(
        dropna=False
    )
)


display(
    remaining_resolution_metric_df
    .sort_values(
        [
            "target_source_file",
            "metric"
        ]
    )
)

METRIC-LEVEL RESULT
preferred_mode
NO_EVIDENCE    18
Name: count, dtype: int64


,target_source_file,metric,known_neighbors,presentation_score,full_score,preferred_mode
0,CBDK_2024_Q4_FS.xlsx,cash,0,NaN,NaN,NO_EVIDENCE
1,CBDK_2024_Q4_FS.xlsx,gross_profit,0,NaN,NaN,NO_EVIDENCE
2,CBDK_2024_Q4_FS.xlsx,operating_cash_flow,0,NaN,NaN,NO_EVIDENCE
3,CBDK_2024_Q4_FS.xlsx,revenue,0,NaN,NaN,NO_EVIDENCE
4,CBDK_2024_Q4_FS.xlsx,total_assets,0,NaN,NaN,NO_EVIDENCE
5,CBDK_2024_Q4_FS.xlsx,total_liabilities,0,NaN,NaN,NO_EVIDENCE
6,HGII_2025_Q1_FS.xlsx,cash,0,NaN,NaN,NO_EVIDENCE
7,HGII_2025_Q1_FS.xlsx,gross_profit,0,NaN,NaN,NO_EVIDENCE
8,HGII_2025_Q1_FS.xlsx,operating_cash_flow,0,NaN,NaN,NO_EVIDENCE
9,HGII_2025_Q1_FS.xlsx,revenue,0,NaN,NaN,NO_EVIDENCE


In [53]:
# CELL 85 - FINAL RESOLUTION FOR REMAINING NON-UNIT FILES

remaining_file_resolution_df = (
    remaining_resolution_metric_df
    .groupby(
        "target_source_file"
    )
    .agg(
        presentation_votes=(
            "preferred_mode",
            lambda x:
                (
                    x
                    == "PRESENTATION_SCALED"
                ).sum()
        ),

        full_nominal_votes=(
            "preferred_mode",
            lambda x:
                (
                    x
                    == "FULL_NOMINAL"
                ).sum()
        ),

        no_evidence_votes=(
            "preferred_mode",
            lambda x:
                x.isin(
                    [
                        "NO_EVIDENCE",
                        "TIE"
                    ]
                ).sum()
        ),

        total_metrics=(
            "metric",
            "count"
        )
    )
    .reset_index()
)


def resolve_remaining_file(row):

    if (
        row["presentation_votes"]
        >
        row["full_nominal_votes"]
    ):
        return "PRESENTATION_SCALED"

    if (
        row["full_nominal_votes"]
        >
        row["presentation_votes"]
    ):
        return "FULL_NOMINAL"

    return "UNRESOLVED"


remaining_file_resolution_df[
    "resolved_scale_mode"
] = (
    remaining_file_resolution_df
    .apply(
        resolve_remaining_file,
        axis=1
    )
)


display(
    remaining_file_resolution_df
)


print("\nFINAL COUNTS")

print(
    remaining_file_resolution_df[
        "resolved_scale_mode"
    ].value_counts(
        dropna=False
    )
)

,target_source_file,presentation_votes,full_nominal_votes,no_evidence_votes,total_metrics,resolved_scale_mode
0,CBDK_2024_Q4_FS.xlsx,0,0,6,6,UNRESOLVED
1,HGII_2025_Q1_FS.xlsx,0,0,6,6,UNRESOLVED
2,MDIY_2025_Q1_FS.xlsx,0,0,6,6,UNRESOLVED



FINAL COUNTS
resolved_scale_mode
UNRESOLVED    3
Name: count, dtype: int64
